# RSNA Multiclass Chest X-Ray Reliability and XAI

**Repository snapshot:** `v0.1 — Pre-ISBI-extension reproducible snapshot`

This notebook is a cleaned research record of the completed 2026 multiclass RSNA study. It preserves the data engineering, model-selection, locked-test, subgroup, calibration, and XAI analyses completed before the next ISBI extension phase.

**Primary task:** Normal · No Lung Opacity / Not Normal · Lung Opacity

The study evaluates pneumonia-associated lung opacity classification and reliability. It is not a direct clinical pneumonia-diagnosis system.


In [ ]:
from pathlib import Path
import os, random, json
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

PROJECT_ROOT = Path(os.environ.get("RSNA_PROJECT_ROOT", "/content/RSNA_Pneumonia_Multiclass_XAI_v2"))
DIRS = {
    "raw": PROJECT_ROOT / "data" / "raw",
    "processed": PROJECT_ROOT / "data" / "processed",
    "splits": PROJECT_ROOT / "data" / "splits",
    "figures": PROJECT_ROOT / "figures",
    "results": PROJECT_ROOT / "results",
    "experiments": PROJECT_ROOT / "experiments",
}
for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)


## 1. Environment and reproducibility

In [ ]:
# Environment check

import os
import sys
import platform

print("Python version:", sys.version)
print("Platform:", platform.platform())

# Check available disk space
!df -h /content

# Check GPU
!nvidia-smi


In [ ]:
import tensorflow as tf
import numpy as np
import random
import os

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

if tf.config.list_physical_devices("GPU"):
    print("TensorFlow can access the GPU.")
else:
    print("TensorFlow cannot access the GPU.")


## 2. Dataset acquisition and canonical reconstruction

In [ ]:
import kagglehub

DATASET_SLUG = "iamtapendu/rsna-pneumonia-processed-dataset"

dataset_path = kagglehub.dataset_download(DATASET_SLUG)

print("Dataset downloaded successfully.")
print("Dataset path:", dataset_path)


In [ ]:
from pathlib import Path
import pandas as pd

DATASET_PATH = Path(dataset_path)

# Find all CSV files
csv_files = sorted(DATASET_PATH.rglob("*.csv"))

print("CSV files found:")
for f in csv_files:
    print(" -", f.relative_to(DATASET_PATH))

# Locate the training metadata
train_csvs = [
    f for f in csv_files
    if "train" in f.name.lower() and "metadata" in f.name.lower()
]

print("\nCandidate training metadata files:")
for f in train_csvs:
    print(" -", f)

if train_csvs:
    metadata_path = train_csvs[0]
    df_raw = pd.read_csv(metadata_path)

    print("\nShape:", df_raw.shape)

    print("\nColumns:")
    print(df_raw.columns.tolist())

    print("\nFirst five rows:")
    display(df_raw.head())

    if "class" in df_raw.columns:
        print("\nClass values:")
        print(df_raw["class"].value_counts(dropna=False))

        print("\nTarget × class:")
        print(pd.crosstab(df_raw["class"], df_raw["Target"]))

        print("\nUnique patientIds by class:")
        print(
            df_raw.groupby("class")["patientId"]
                  .nunique()
                  .sort_values(ascending=False)
        )
    else:
        print("\nNo 'class' column found.")


In [ ]:
# ============================================================
# Metadata integrity audit
# ============================================================

import pandas as pd
import numpy as np

# Basic row-level counts
n_rows = len(df_raw)
n_patients = df_raw["patientId"].nunique()
extra_rows = n_rows - n_patients

print("RAW METADATA SUMMARY")
print("-" * 50)
print(f"Metadata rows:       {n_rows:,}")
print(f"Unique patientIds:   {n_patients:,}")
print(f"Extra repeated rows: {extra_rows:,}")


# ------------------------------------------------------------
# 1. How many rows does each patient have?
# ------------------------------------------------------------

rows_per_patient = df_raw.groupby("patientId").size()

print("\nROWS PER PATIENT")
print("-" * 50)
print(rows_per_patient.value_counts().sort_index())

print(f"\nPatients with >1 metadata row: {(rows_per_patient > 1).sum():,}")
print(f"Maximum rows for one patient:  {rows_per_patient.max()}")


# ------------------------------------------------------------
# 2. Which classes contain repeated patientIds?
# ------------------------------------------------------------

repeated_ids = rows_per_patient[rows_per_patient > 1].index

repeated_patient_classes = (
    df_raw[df_raw["patientId"].isin(repeated_ids)]
    .drop_duplicates("patientId")
    ["class"]
    .value_counts()
)

print("\nCLASSES AMONG PATIENTS WITH REPEATED ROWS")
print("-" * 50)
print(repeated_patient_classes)


# ------------------------------------------------------------
# 3. Check whether patient-level metadata is internally
#    consistent across repeated rows
# ------------------------------------------------------------

fields_to_check = [
    "Target",
    "class",
    "age",
    "sex",
    "modality",
    "position"
]

consistency_check = (
    df_raw.groupby("patientId")[fields_to_check]
    .nunique(dropna=False)
)

print("\nPATIENT-LEVEL CONSISTENCY CHECK")
print("-" * 50)

for column in fields_to_check:
    inconsistent = (consistency_check[column] > 1).sum()
    print(f"{column:10s}: {inconsistent:,} patients with conflicting values")


# ------------------------------------------------------------
# 4. Bounding-box completeness
# ------------------------------------------------------------

bbox_cols = ["x", "y", "width", "height"]

bbox_complete = df_raw[bbox_cols].notna().all(axis=1)
bbox_missing = df_raw[bbox_cols].isna().all(axis=1)
bbox_partial = ~(bbox_complete | bbox_missing)

print("\nBOUNDING BOX AUDIT")
print("-" * 50)
print(f"Rows with complete bounding box: {bbox_complete.sum():,}")
print(f"Rows with no bounding box:       {bbox_missing.sum():,}")
print(f"Rows with partial bounding box:  {bbox_partial.sum():,}")


# Bounding boxes by class
bbox_by_class = (
    df_raw.assign(has_bbox=bbox_complete)
    .groupby("class")["has_bbox"]
    .agg(["count", "sum", "mean"])
)

bbox_by_class["mean"] = bbox_by_class["mean"] * 100
bbox_by_class = bbox_by_class.rename(
    columns={
        "count": "metadata_rows",
        "sum": "rows_with_bbox",
        "mean": "percent_with_bbox"
    }
)

print("\nBOUNDING BOXES BY CLASS")
print("-" * 50)
display(bbox_by_class)


### 2.1 Build the Canonical Data Model

The raw metadata is annotation-level: a single X-ray may appear in multiple
rows when radiologists identified multiple opacity regions.

To avoid treating annotation rows as independent samples, the data are
normalized into two linked tables:

1. **Image-level metadata** — one row per unique X-ray/patient.
2. **Bounding-box annotations** — one row per annotated opacity region.

The tables are linked using `patientId`.

This preserves all radiologist annotations while ensuring that each X-ray
contributes only one observation to the classification task.

In [ ]:
# ============================================================
# Build canonical image-level and bounding-box tables
# ============================================================

# Keep the raw dataframe untouched
raw_metadata = df_raw.copy()


# ------------------------------------------------------------
# 1. IMAGE-LEVEL TABLE
#    One row per unique patient/X-ray
# ------------------------------------------------------------

image_columns = [
    "patientId",
    "Target",
    "class",
    "age",
    "sex",
    "modality",
    "position"
]

image_metadata = (
    raw_metadata[image_columns]
    .drop_duplicates(subset="patientId")
    .copy()
)


# ------------------------------------------------------------
# 2. BOUNDING-BOX TABLE
#    One row per radiologist bounding box
# ------------------------------------------------------------

bbox_columns = [
    "patientId",
    "x",
    "y",
    "width",
    "height"
]

bbox_annotations = (
    raw_metadata
    .dropna(subset=["x", "y", "width", "height"])
    [bbox_columns]
    .copy()
)

# Number boxes within each X-ray
bbox_annotations["bbox_id"] = (
    bbox_annotations
    .groupby("patientId")
    .cumcount() + 1
)


# ------------------------------------------------------------
# 3. Add bounding-box count to image-level metadata
# ------------------------------------------------------------

bbox_counts = (
    bbox_annotations
    .groupby("patientId")
    .size()
    .rename("bbox_count")
)

image_metadata = image_metadata.merge(
    bbox_counts,
    on="patientId",
    how="left"
)

image_metadata["bbox_count"] = (
    image_metadata["bbox_count"]
    .fillna(0)
    .astype(int)
)

image_metadata["has_bbox"] = image_metadata["bbox_count"] > 0


# ------------------------------------------------------------
# 4. Integrity assertions
# ------------------------------------------------------------

assert image_metadata["patientId"].is_unique

assert len(image_metadata) == raw_metadata["patientId"].nunique()

assert len(bbox_annotations) == 9555

assert (
    image_metadata.loc[
        image_metadata["class"] == "Lung Opacity",
        "bbox_count"
    ] >= 1
).all()

assert (
    image_metadata.loc[
        image_metadata["class"] != "Lung Opacity",
        "bbox_count"
    ] == 0
).all()


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print("CANONICAL DATA MODEL")
print("-" * 50)

print(f"Raw annotation rows:       {len(raw_metadata):,}")
print(f"Unique X-rays/patients:    {len(image_metadata):,}")
print(f"Bounding-box annotations:  {len(bbox_annotations):,}")

print("\nIMAGE-LEVEL CLASS DISTRIBUTION")
print("-" * 50)
print(image_metadata["class"].value_counts())

print("\nBOUNDING BOX COUNT PER LUNG-OPACITY X-RAY")
print("-" * 50)

print(
    image_metadata.loc[
        image_metadata["class"] == "Lung Opacity",
        "bbox_count"
    ].value_counts().sort_index()
)

print("\nImage metadata preview:")
display(image_metadata.head())

print("\nBounding-box annotation preview:")
display(bbox_annotations.head())


In [ ]:
ACTIVE_DATASET = DATASET_PATH
PROCESSED_DIR = DIRS["processed"]
RAW_METADATA_PATH = DIRS["raw"] / "stage2_train_metadata.csv"
CANONICAL_METADATA_PATH = PROCESSED_DIR / "canonical_image_metadata.csv"
BBOX_PATH = PROCESSED_DIR / "bbox_annotations.csv"

df_raw.to_csv(RAW_METADATA_PATH, index=False)
image_metadata.reset_index(drop=True).to_csv(CANONICAL_METADATA_PATH, index=False)
bbox_annotations.reset_index(drop=True).to_csv(BBOX_PATH, index=False)
print("Active dataset:", ACTIVE_DATASET)
print("Saved canonical metadata:", CANONICAL_METADATA_PATH)
print("Saved bounding boxes:", BBOX_PATH)


## 3. Data-integrity and metadata audits

In [ ]:
# ============================================================
# Verify one-to-one mapping:
# metadata patientId <-> training image <-> training mask
# ============================================================

import pandas as pd
import json
from pathlib import Path


# ------------------------------------------------------------
# Reload checkpoints if Colab was restarted
# ------------------------------------------------------------

if "image_metadata" not in globals():
    image_metadata = pd.read_csv(
        PROCESSED_DIR / "image_metadata.csv"
    )
    print(" Reloaded image_metadata from Drive.")

if "file_inventory" not in globals():
    file_inventory = pd.read_csv(
        PROCESSED_DIR / "physical_file_inventory.csv"
    )
    print(" Reloaded physical file inventory from Drive.")


# ------------------------------------------------------------
# Separate training images and masks
# ------------------------------------------------------------

training_images = (
    file_inventory[
        file_inventory["parent_folder"] == "Training/Images"
    ]
    .copy()
)

training_masks = (
    file_inventory[
        file_inventory["parent_folder"] == "Training/Masks"
    ]
    .copy()
)

print("FILES SELECTED")
print("-" * 55)
print(f"Training images: {len(training_images):,}")
print(f"Training masks:  {len(training_masks):,}")


# ------------------------------------------------------------
# Check for duplicate physical identifiers
# ------------------------------------------------------------

image_duplicate_ids = training_images["stem"].duplicated().sum()
mask_duplicate_ids = training_masks["stem"].duplicated().sum()

print("\nDUPLICATE FILE IDs")
print("-" * 55)
print(f"Duplicate image IDs: {image_duplicate_ids:,}")
print(f"Duplicate mask IDs:  {mask_duplicate_ids:,}")


# ------------------------------------------------------------
# Compare patient-ID sets
# ------------------------------------------------------------

metadata_ids = set(image_metadata["patientId"].astype(str))
image_ids = set(training_images["stem"].astype(str))
mask_ids = set(training_masks["stem"].astype(str))

missing_images = metadata_ids - image_ids
orphan_images = image_ids - metadata_ids

missing_masks = metadata_ids - mask_ids
orphan_masks = mask_ids - metadata_ids

print("\nMETADATA ↔ IMAGE MATCHING")
print("-" * 55)
print(f"Metadata IDs:                  {len(metadata_ids):,}")
print(f"Image IDs:                     {len(image_ids):,}")
print(f"Metadata IDs missing images:   {len(missing_images):,}")
print(f"Images without metadata:       {len(orphan_images):,}")

print("\nMETADATA ↔ MASK MATCHING")
print("-" * 55)
print(f"Mask IDs:                      {len(mask_ids):,}")
print(f"Metadata IDs missing masks:    {len(missing_masks):,}")
print(f"Masks without metadata:        {len(orphan_masks):,}")


# ------------------------------------------------------------
# Hard integrity checks
# ------------------------------------------------------------

assert image_duplicate_ids == 0
assert mask_duplicate_ids == 0

assert len(metadata_ids) == 26684
assert len(image_ids) == 26684
assert len(mask_ids) == 26684

assert len(missing_images) == 0
assert len(orphan_images) == 0

assert len(missing_masks) == 0
assert len(orphan_masks) == 0


# ------------------------------------------------------------
# Attach portable RELATIVE paths to canonical metadata
# ------------------------------------------------------------

image_path_map = (
    training_images
    .set_index("stem")["relative_path"]
    .to_dict()
)

mask_path_map = (
    training_masks
    .set_index("stem")["relative_path"]
    .to_dict()
)

image_metadata["image_relpath"] = (
    image_metadata["patientId"].map(image_path_map)
)

image_metadata["mask_relpath"] = (
    image_metadata["patientId"].map(mask_path_map)
)

assert image_metadata["image_relpath"].notna().all()
assert image_metadata["mask_relpath"].notna().all()


# ------------------------------------------------------------
# Save upgraded canonical table
# ------------------------------------------------------------

canonical_path = (
    PROCESSED_DIR /
    "canonical_image_metadata.csv"
)

image_metadata.to_csv(
    canonical_path,
    index=False
)


# ------------------------------------------------------------
# Save integrity report
# ------------------------------------------------------------

integrity_report = {
    "metadata_images": len(metadata_ids),
    "physical_training_images": len(image_ids),
    "physical_training_masks": len(mask_ids),
    "duplicate_image_ids": int(image_duplicate_ids),
    "duplicate_mask_ids": int(mask_duplicate_ids),
    "metadata_missing_images": len(missing_images),
    "orphan_images": len(orphan_images),
    "metadata_missing_masks": len(missing_masks),
    "orphan_masks": len(orphan_masks),
    "one_to_one_mapping_verified": True
}

integrity_report_path = (
    PROCESSED_DIR /
    "data_integrity_report.json"
)

with open(integrity_report_path, "w") as f:
    json.dump(integrity_report, f, indent=4)


print("\n ONE-TO-ONE DATA INTEGRITY VERIFIED")

print("\nCanonical metadata saved:")
print(canonical_path)

print("\nIntegrity report saved:")
print(integrity_report_path)

print("\nCanonical table preview:")
display(image_metadata.head())


In [ ]:
from pathlib import Path
from PIL import Image
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm

# ============================================================
# Image + mask structural audit
# ============================================================

# Use fast local copy if still available
if Path(dataset_path).exists():
    ACTIVE_DATASET = Path(dataset_path)
else:
    ACTIVE_DATASET = PERSISTENT_DATASET

print("Auditing dataset from:")
print(ACTIVE_DATASET)


# Reload canonical metadata if needed
if "image_metadata" not in globals():
    image_metadata = pd.read_csv(
        PROCESSED_DIR / "canonical_image_metadata.csv"
    )


audit_records = []

for row in tqdm(
    image_metadata.itertuples(index=False),
    total=len(image_metadata),
    desc="Checking image files"
):
    image_path = ACTIVE_DATASET / row.image_relpath
    mask_path = ACTIVE_DATASET / row.mask_relpath

    record = {
        "patientId": row.patientId,
        "image_readable": False,
        "mask_readable": False,
        "image_width": None,
        "image_height": None,
        "image_mode": None,
        "mask_width": None,
        "mask_height": None,
        "mask_mode": None,
        "dimensions_match": False,
        "error": None
    }

    try:
        with Image.open(image_path) as img:
            record["image_width"], record["image_height"] = img.size
            record["image_mode"] = img.mode
            img.verify()

        record["image_readable"] = True

    except Exception as e:
        record["error"] = f"IMAGE: {str(e)}"

    try:
        with Image.open(mask_path) as mask:
            record["mask_width"], record["mask_height"] = mask.size
            record["mask_mode"] = mask.mode
            mask.verify()

        record["mask_readable"] = True

    except Exception as e:
        if record["error"]:
            record["error"] += f" | MASK: {str(e)}"
        else:
            record["error"] = f"MASK: {str(e)}"

    record["dimensions_match"] = (
        record["image_readable"]
        and record["mask_readable"]
        and record["image_width"] == record["mask_width"]
        and record["image_height"] == record["mask_height"]
    )

    audit_records.append(record)


image_format_audit = pd.DataFrame(audit_records)


# ============================================================
# Summary
# ============================================================

print("\nIMAGE / MASK FORMAT AUDIT")
print("-" * 60)

print(
    f"Readable images: "
    f"{image_format_audit['image_readable'].sum():,}"
    f" / {len(image_format_audit):,}"
)

print(
    f"Readable masks:  "
    f"{image_format_audit['mask_readable'].sum():,}"
    f" / {len(image_format_audit):,}"
)

print(
    f"Matching dimensions: "
    f"{image_format_audit['dimensions_match'].sum():,}"
    f" / {len(image_format_audit):,}"
)


print("\nIMAGE DIMENSIONS")
print("-" * 60)

print(
    image_format_audit[
        ["image_width", "image_height"]
    ].value_counts()
)


print("\nIMAGE MODES")
print("-" * 60)

print(
    image_format_audit["image_mode"]
    .value_counts(dropna=False)
)


print("\nMASK DIMENSIONS")
print("-" * 60)

print(
    image_format_audit[
        ["mask_width", "mask_height"]
    ].value_counts()
)


print("\nMASK MODES")
print("-" * 60)

print(
    image_format_audit["mask_mode"]
    .value_counts(dropna=False)
)


print("\nUNREADABLE FILES")
print("-" * 60)

problem_files = image_format_audit[
    (~image_format_audit["image_readable"])
    | (~image_format_audit["mask_readable"])
    | (~image_format_audit["dimensions_match"])
]

print(f"Problem records: {len(problem_files):,}")

if len(problem_files) > 0:
    display(problem_files.head(20))
else:
    print(" No structural file problems detected.")


# ============================================================
# SAVE CHECKPOINT
# ============================================================

audit_path = (
    PROCESSED_DIR /
    "image_format_audit.csv"
)

image_format_audit.to_csv(
    audit_path,
    index=False
)

print("\n Audit checkpoint saved:")
print(audit_path)


In [ ]:
# ============================================================
# Pixel-level audit — corrected version
# ============================================================

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# Reload the SAME saved sample if necessary
sample_path = PROCESSED_DIR / "pixel_audit_sample.csv"

if "pixel_sample" not in globals():
    pixel_sample = pd.read_csv(sample_path)
    print(" Reloaded the original stratified sample from Drive.")
else:
    print(" Using the original stratified sample already in memory.")


pixel_records = []

# iterrows() is clearer here because "class" is a Python keyword
for _, row in tqdm(
    pixel_sample.iterrows(),
    total=len(pixel_sample),
    desc="Auditing pixel values"
):

    image_path = ACTIVE_DATASET / row["image_relpath"]
    mask_path = ACTIVE_DATASET / row["mask_relpath"]

    image = np.asarray(
        Image.open(image_path),
        dtype=np.uint8
    )

    mask = np.asarray(
        Image.open(mask_path),
        dtype=np.uint8
    )

    record = {
        "patientId": row["patientId"],
        "class": row["class"],
        "Target": int(row["Target"]),

        # Image statistics
        "image_min": int(image.min()),
        "image_max": int(image.max()),
        "image_mean": float(image.mean()),
        "image_std": float(image.std()),
        "image_p01": float(np.percentile(image, 1)),
        "image_p50": float(np.percentile(image, 50)),
        "image_p99": float(np.percentile(image, 99)),

        # Mask statistics
        "mask_min": int(mask.min()),
        "mask_max": int(mask.max()),
        "mask_nonzero_pixels": int(np.count_nonzero(mask)),
        "mask_nonzero_fraction": float(
            np.count_nonzero(mask) / mask.size
        ),
        "mask_unique_values": str(
            np.unique(mask).tolist()
        )
    }

    pixel_records.append(record)


pixel_audit = pd.DataFrame(pixel_records)


# ============================================================
# Summary by class
# ============================================================

print("\nIMAGE INTENSITY SUMMARY BY CLASS")
print("-" * 60)

display(
    pixel_audit
    .groupby("class")[
        [
            "image_min",
            "image_max",
            "image_mean",
            "image_std",
            "image_p01",
            "image_p50",
            "image_p99"
        ]
    ]
    .agg(["mean", "min", "max"])
    .round(2)
)


print("\nMASK BEHAVIOR BY CLASS")
print("-" * 60)

mask_summary = (
    pixel_audit
    .groupby("class")
    .agg(
        sampled_images=("patientId", "count"),
        masks_with_signal=(
            "mask_nonzero_pixels",
            lambda x: (x > 0).sum()
        ),
        mean_mask_fraction=(
            "mask_nonzero_fraction",
            "mean"
        ),
        max_mask_fraction=(
            "mask_nonzero_fraction",
            "max"
        )
    )
)

display(mask_summary)


print("\nMASK PIXEL VALUES")
print("-" * 60)

print(
    pixel_audit[
        ["class", "mask_unique_values"]
    ]
    .value_counts()
)


# ============================================================
# Logical consistency checks
# ============================================================

negative_mask_signal = pixel_audit[
    (pixel_audit["class"] != "Lung Opacity")
    &
    (pixel_audit["mask_nonzero_pixels"] > 0)
]

positive_empty_masks = pixel_audit[
    (pixel_audit["class"] == "Lung Opacity")
    &
    (pixel_audit["mask_nonzero_pixels"] == 0)
]

print("\nSAMPLE MASK CONSISTENCY")
print("-" * 60)

print(
    "Negative-class images with non-empty masks:",
    len(negative_mask_signal)
)

print(
    "Lung Opacity images with empty masks:",
    len(positive_empty_masks)
)


# ============================================================
# Save persistent checkpoint
# ============================================================

pixel_audit_path = (
    PROCESSED_DIR /
    "pixel_level_audit.csv"
)

pixel_audit.to_csv(
    pixel_audit_path,
    index=False
)

print("\n Pixel audit completed and saved:")
print(pixel_audit_path)


In [ ]:
# ============================================================
# Verify whether stored masks are generated from bounding boxes
# ============================================================

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# Reload bounding-box checkpoint if needed
if "bbox_annotations" not in globals():
    bbox_annotations = pd.read_csv(
        PROCESSED_DIR / "bbox_annotations.csv"
    )

# Group all boxes belonging to each image
bbox_groups = {
    patient_id: group[["x", "y", "width", "height"]].to_numpy()
    for patient_id, group in bbox_annotations.groupby("patientId")
}

lung_images = image_metadata[
    image_metadata["class"] == "Lung Opacity"
].copy()

mask_comparison_records = []

for _, row in tqdm(
    lung_images.iterrows(),
    total=len(lung_images),
    desc="Comparing masks with bounding boxes"
):
    patient_id = row["patientId"]

    stored_mask = np.asarray(
        Image.open(ACTIVE_DATASET / row["mask_relpath"]),
        dtype=np.uint8
    ) > 0

    # Reconstruct a binary mask from the bounding boxes
    reconstructed_mask = np.zeros(
        stored_mask.shape,
        dtype=bool
    )

    for x, y, width, height in bbox_groups[patient_id]:

        x1 = int(x)
        y1 = int(y)
        x2 = int(x + width)
        y2 = int(y + height)

        # Keep coordinates within image boundaries
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(stored_mask.shape[1], x2)
        y2 = min(stored_mask.shape[0], y2)

        reconstructed_mask[y1:y2, x1:x2] = True

    intersection = np.logical_and(
        stored_mask,
        reconstructed_mask
    ).sum()

    union = np.logical_or(
        stored_mask,
        reconstructed_mask
    ).sum()

    iou = intersection / union if union > 0 else 1.0

    mask_comparison_records.append({
        "patientId": patient_id,
        "bbox_count": int(row["bbox_count"]),
        "stored_mask_pixels": int(stored_mask.sum()),
        "bbox_mask_pixels": int(reconstructed_mask.sum()),
        "iou": float(iou),
        "exact_match": bool(
            np.array_equal(
                stored_mask,
                reconstructed_mask
            )
        )
    })


mask_bbox_audit = pd.DataFrame(mask_comparison_records)


# ============================================================
# Results
# ============================================================

print("\nMASK ↔ BOUNDING-BOX AUDIT")
print("-" * 60)

print(f"Lung Opacity images checked: {len(mask_bbox_audit):,}")

print(
    "Exact mask matches:",
    f"{mask_bbox_audit['exact_match'].sum():,}",
    "/",
    f"{len(mask_bbox_audit):,}"
)

print("\nIoU summary:")
print(
    mask_bbox_audit["iou"]
    .describe()
    .round(6)
)

print("\nLowest-IoU cases:")
display(
    mask_bbox_audit
    .sort_values("iou")
    .head(10)
)


# ============================================================
# Save checkpoint
# ============================================================

mask_bbox_audit_path = (
    PROCESSED_DIR /
    "mask_bbox_consistency_audit.csv"
)

mask_bbox_audit.to_csv(
    mask_bbox_audit_path,
    index=False
)

print("\n Audit saved:")
print(mask_bbox_audit_path)


In [ ]:
# ============================================================
# Visual sanity check: examples from all three classes
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image
import pandas as pd

VIS_SEED = 42
N_EXAMPLES_PER_CLASS = 2

classes = [
    "Normal",
    "No Lung Opacity / Not Normal",
    "Lung Opacity"
]

# Reproducible examples
visual_sample = (
    image_metadata[
        image_metadata["class"].isin(classes)
    ]
    .groupby("class", group_keys=False)
    .sample(
        n=N_EXAMPLES_PER_CLASS,
        random_state=VIS_SEED
    )
)

# Preserve desired class order
visual_sample["class"] = pd.Categorical(
    visual_sample["class"],
    categories=classes,
    ordered=True
)

visual_sample = (
    visual_sample
    .sort_values("class")
    .reset_index(drop=True)
)

# Save selected examples for reproducibility
visual_sample_path = (
    PROCESSED_DIR /
    "visual_sanity_check_sample.csv"
)

visual_sample.to_csv(
    visual_sample_path,
    index=False
)


# ============================================================
# Plot
# ============================================================

fig, axes = plt.subplots(
    len(classes),
    N_EXAMPLES_PER_CLASS,
    figsize=(10, 14)
)

for class_idx, class_name in enumerate(classes):

    class_rows = visual_sample[
        visual_sample["class"] == class_name
    ].reset_index(drop=True)

    for example_idx in range(N_EXAMPLES_PER_CLASS):

        row = class_rows.iloc[example_idx]
        ax = axes[class_idx, example_idx]

        image = np.asarray(
            Image.open(
                ACTIVE_DATASET / row["image_relpath"]
            )
        )

        ax.imshow(image, cmap="gray")

        # Draw radiologist bounding boxes for Lung Opacity
        if class_name == "Lung Opacity":

            patient_boxes = bbox_annotations[
                bbox_annotations["patientId"]
                == row["patientId"]
            ]

            for _, box in patient_boxes.iterrows():

                rectangle = patches.Rectangle(
                    (box["x"], box["y"]),
                    box["width"],
                    box["height"],
                    fill=False,
                    linewidth=2
                )

                ax.add_patch(rectangle)

        ax.set_title(
            f"{class_name}\n"
            f"Age: {row['age']} | "
            f"Sex: {row['sex']} | "
            f"Position: {row['position']}"
        )

        ax.axis("off")


plt.suptitle(
    "Representative RSNA Chest X-rays",
    fontsize=16
)

plt.tight_layout()


# ============================================================
# Save figure
# ============================================================

figure_path = (
    DIRS["figures"] /
    "representative_xrays_three_classes.png"
)

plt.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print(" Figure saved:")
print(figure_path)

print("\n Selected patient IDs saved:")
print(visual_sample_path)


In [ ]:
# ============================================================
# Metadata shortcut / confounding audit
# ============================================================

import pandas as pd
import numpy as np
import json

# Reload canonical metadata if necessary
if "image_metadata" not in globals():
    image_metadata = pd.read_csv(
        PROCESSED_DIR / "canonical_image_metadata.csv"
    )

print("CANONICAL DATASET")
print("-" * 60)
print(f"Images: {len(image_metadata):,}")


# ============================================================
# 1. Missingness
# ============================================================

print("\nMISSING VALUES")
print("-" * 60)

missing_summary = pd.DataFrame({
    "missing_n": image_metadata.isna().sum(),
    "missing_percent": image_metadata.isna().mean() * 100
})

display(
    missing_summary[
        missing_summary["missing_n"] > 0
    ].round(2)
)


# ============================================================
# 2. Acquisition position × class
# ============================================================

print("\nPOSITION × CLASS — COUNTS")
print("-" * 60)

position_counts = pd.crosstab(
    image_metadata["class"],
    image_metadata["position"]
)

display(position_counts)


print("\nPOSITION × CLASS — ROW PERCENTAGES")
print("-" * 60)

position_pct = pd.crosstab(
    image_metadata["class"],
    image_metadata["position"],
    normalize="index"
) * 100

display(position_pct.round(2))


# ============================================================
# 3. Sex × class
# ============================================================

print("\nSEX × CLASS — COUNTS")
print("-" * 60)

sex_counts = pd.crosstab(
    image_metadata["class"],
    image_metadata["sex"]
)

display(sex_counts)


print("\nSEX × CLASS — ROW PERCENTAGES")
print("-" * 60)

sex_pct = pd.crosstab(
    image_metadata["class"],
    image_metadata["sex"],
    normalize="index"
) * 100

display(sex_pct.round(2))


# ============================================================
# 4. Modality × class
# ============================================================

print("\nMODALITY × CLASS")
print("-" * 60)

modality_counts = pd.crosstab(
    image_metadata["class"],
    image_metadata["modality"]
)

display(modality_counts)


# ============================================================
# 5. Age audit
# ============================================================

print("\nAGE SUMMARY BY CLASS")
print("-" * 60)

age_summary = (
    image_metadata
    .groupby("class")["age"]
    .agg([
        "count",
        "mean",
        "std",
        "min",
        "median",
        "max"
    ])
    .round(2)
)

display(age_summary)


# Age ranges
age_bins = [-1, 17, 39, 59, 79, np.inf]
age_labels = [
    "0–17",
    "18–39",
    "40–59",
    "60–79",
    "80+"
]

image_metadata["age_group"] = pd.cut(
    image_metadata["age"],
    bins=age_bins,
    labels=age_labels
)

print("\nAGE GROUP × CLASS — ROW PERCENTAGES")
print("-" * 60)

age_group_pct = pd.crosstab(
    image_metadata["class"],
    image_metadata["age_group"],
    normalize="index"
) * 100

display(age_group_pct.round(2))


# ============================================================
# 6. Combined acquisition setting
# ============================================================

print("\nSEX × POSITION × CLASS")
print("-" * 60)

combined = (
    image_metadata
    .groupby(["class", "position", "sex"])
    .size()
    .rename("n")
    .reset_index()
)

combined["percent_within_class"] = (
    combined["n"]
    / combined.groupby("class")["n"].transform("sum")
    * 100
)

display(
    combined.sort_values(
        ["class", "n"],
        ascending=[True, False]
    ).round(2)
)


# ============================================================
# 7. Save audit tables
# ============================================================

audit_dir = PROCESSED_DIR / "metadata_audit"
audit_dir.mkdir(parents=True, exist_ok=True)

position_counts.to_csv(audit_dir / "position_by_class_counts.csv")
position_pct.to_csv(audit_dir / "position_by_class_percent.csv")

sex_counts.to_csv(audit_dir / "sex_by_class_counts.csv")
sex_pct.to_csv(audit_dir / "sex_by_class_percent.csv")

modality_counts.to_csv(audit_dir / "modality_by_class_counts.csv")

age_summary.to_csv(audit_dir / "age_summary_by_class.csv")
age_group_pct.to_csv(audit_dir / "age_group_by_class_percent.csv")

combined.to_csv(
    audit_dir / "position_sex_by_class.csv",
    index=False
)

print("\n Metadata audit saved:")
print(audit_dir)


In [ ]:
# ============================================================
# Focused audit: age quality + acquisition-position confounding
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import json

# Reload canonical metadata if necessary
if "image_metadata" not in globals():
    image_metadata = pd.read_csv(
        PROCESSED_DIR / "canonical_image_metadata.csv"
    )


# ============================================================
# 1. AGE DATA QUALITY
# ============================================================

print("AGE DATA QUALITY")
print("-" * 60)

print("Lowest ages:")
print(
    image_metadata["age"]
    .value_counts()
    .sort_index()
    .head(15)
)

print("\nHighest ages:")
print(
    image_metadata["age"]
    .value_counts()
    .sort_index(ascending=False)
    .head(20)
)


# Flag implausible / suspicious age values
age_outliers = image_metadata[
    (image_metadata["age"] < 0)
    | (image_metadata["age"] > 100)
].copy()

print("\nAge values >100 or <0:")
print(f"Records: {len(age_outliers):,}")

if len(age_outliers) > 0:
    display(
        age_outliers[
            [
                "patientId",
                "age",
                "class",
                "sex",
                "position"
            ]
        ].sort_values("age", ascending=False)
    )


# ============================================================
# 2. POSITION × CLASS ASSOCIATION
# ============================================================

position_table = pd.crosstab(
    image_metadata["class"],
    image_metadata["position"]
)

chi2, p_value, dof, expected = chi2_contingency(position_table)

n = position_table.to_numpy().sum()
r, k = position_table.shape

cramers_v = np.sqrt(
    chi2 / (n * min(r - 1, k - 1))
)

print("\nPOSITION × CLASS ASSOCIATION")
print("-" * 60)

display(position_table)

print(f"Chi-square:  {chi2:,.2f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.3e}")
print(f"Cramér's V: {cramers_v:.3f}")


# ============================================================
# 3. CLASS DISTRIBUTION WITHIN AP AND PA
# ============================================================

print("\nCLASS DISTRIBUTION WITHIN EACH POSITION")
print("-" * 60)

class_given_position = pd.crosstab(
    image_metadata["position"],
    image_metadata["class"],
    normalize="index"
) * 100

display(class_given_position.round(2))


# ============================================================
# 4. Save audit
# ============================================================

audit_dir = PROCESSED_DIR / "metadata_audit"
audit_dir.mkdir(parents=True, exist_ok=True)

age_outliers.to_csv(
    audit_dir / "age_outliers.csv",
    index=False
)

class_given_position.to_csv(
    audit_dir / "class_distribution_within_position.csv"
)

confounding_summary = {
    "position_class_chi_square": float(chi2),
    "position_class_df": int(dof),
    "position_class_p_value": float(p_value),
    "position_class_cramers_v": float(cramers_v),
    "age_outlier_records_over_100": int(
        (image_metadata["age"] > 100).sum()
    )
}

with open(
    audit_dir / "confounding_audit_summary.json",
    "w"
) as f:
    json.dump(confounding_summary, f, indent=4)

print("\n Focused audit saved:")
print(audit_dir)


## 4. Fixed analytical split

In [ ]:
# ============================================================
# Final analytical metadata + reproducible train/val/test split
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

SPLIT_SEED = 42

# Reload canonical metadata if needed
if "image_metadata" not in globals():
    image_metadata = pd.read_csv(
        PROCESSED_DIR / "canonical_image_metadata.csv"
    )

analysis_metadata = image_metadata.copy()


# ============================================================
# 1. Preserve raw age and create cleaned age
# ============================================================

analysis_metadata["age_raw"] = analysis_metadata["age"]

analysis_metadata["age_clean"] = analysis_metadata["age"].where(
    analysis_metadata["age"].between(0, 100),
    np.nan
)

print("AGE CLEANING")
print("-" * 60)
print(
    "Implausible ages converted to NaN:",
    analysis_metadata["age_clean"].isna().sum()
)


# ============================================================
# 2. Create explicit multiclass label
# ============================================================

CLASS_TO_ID = {
    "Normal": 0,
    "No Lung Opacity / Not Normal": 1,
    "Lung Opacity": 2
}

analysis_metadata["class_id"] = (
    analysis_metadata["class"]
    .map(CLASS_TO_ID)
)

assert analysis_metadata["class_id"].notna().all()


# ============================================================
# 3. Composite stratification variable
# ============================================================

analysis_metadata["stratify_key"] = (
    analysis_metadata["class"].astype(str)
    + "__"
    + analysis_metadata["position"].astype(str)
)

print("\nSTRATIFICATION GROUPS")
print("-" * 60)
print(
    analysis_metadata["stratify_key"]
    .value_counts()
)


# ============================================================
# 4. First split: 70% train / 30% temporary
# ============================================================

train_df, temp_df = train_test_split(
    analysis_metadata,
    test_size=0.30,
    random_state=SPLIT_SEED,
    stratify=analysis_metadata["stratify_key"]
)


# ============================================================
# 5. Split temporary set equally:
#    15% validation / 15% test
# ============================================================

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SPLIT_SEED,
    stratify=temp_df["stratify_key"]
)


# Assign split names
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

split_metadata = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)


# ============================================================
# 6. Integrity checks
# ============================================================

assert len(split_metadata) == len(analysis_metadata)

assert split_metadata["patientId"].is_unique

assert set(train_df["patientId"]).isdisjoint(
    set(val_df["patientId"])
)

assert set(train_df["patientId"]).isdisjoint(
    set(test_df["patientId"])
)

assert set(val_df["patientId"]).isdisjoint(
    set(test_df["patientId"])
)


# ============================================================
# 7. Report split sizes
# ============================================================

print("\nSPLIT SIZES")
print("-" * 60)

print(
    split_metadata["split"]
    .value_counts()
)

print("\nSPLIT PERCENTAGES")
print("-" * 60)

print(
    (
        split_metadata["split"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)


# ============================================================
# 8. Check class balance across splits
# ============================================================

print("\nCLASS DISTRIBUTION (%) BY SPLIT")
print("-" * 60)

class_split_pct = pd.crosstab(
    split_metadata["split"],
    split_metadata["class"],
    normalize="index"
) * 100

display(class_split_pct.round(2))


# ============================================================
# 9. Check AP/PA balance across splits
# ============================================================

print("\nPOSITION DISTRIBUTION (%) BY SPLIT")
print("-" * 60)

position_split_pct = pd.crosstab(
    split_metadata["split"],
    split_metadata["position"],
    normalize="index"
) * 100

display(position_split_pct.round(2))


# ============================================================
# 10. Strongest check:
#     class × position within each split
# ============================================================

print("\nCLASS × POSITION DISTRIBUTION BY SPLIT")
print("-" * 60)

class_position_split = pd.crosstab(
    split_metadata["split"],
    [
        split_metadata["class"],
        split_metadata["position"]
    ],
    normalize="index"
) * 100

display(class_position_split.round(2))


# ============================================================
# 11. Save everything permanently
# ============================================================

SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

analysis_metadata.to_csv(
    PROCESSED_DIR / "analysis_metadata.csv",
    index=False
)

split_metadata.to_csv(
    SPLITS_DIR / "dataset_splits.csv",
    index=False
)

train_df.to_csv(
    SPLITS_DIR / "train.csv",
    index=False
)

val_df.to_csv(
    SPLITS_DIR / "validation.csv",
    index=False
)

test_df.to_csv(
    SPLITS_DIR / "test.csv",
    index=False
)

print("\n Fixed dataset split saved permanently:")
print(SPLITS_DIR)


## 5. Preprocessing and input pipeline

In [ ]:
import json
import tensorflow as tf
import pandas as pd

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
CLASS_TO_ID = {"Normal": 0, "No Lung Opacity / Not Normal": 1, "Lung Opacity": 2}
ID_TO_CLASS = {value: key for key, value in CLASS_TO_ID.items()}
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
if "train_df" not in globals():
    train_df = pd.read_csv(SPLITS_DIR / "train.csv")
    val_df = pd.read_csv(SPLITS_DIR / "validation.csv")
    test_df = pd.read_csv(SPLITS_DIR / "test.csv")
ACTIVE_DATASET = DATASET_PATH
config = {
    "image_height": 224, "image_width": 224, "batch_size": 32, "seed": 42,
    "class_to_id": CLASS_TO_ID, "input_channels": 3, "source_channels": 1,
    "preprocessing": "tf.keras.applications.mobilenet_v2.preprocess_input",
    "horizontal_flip": False,
    "split_strategy": "70/15/15 stratified by class + position"
}
with open(PROJECT_ROOT / "config.json", "w") as f: json.dump(config, f, indent=4)
print("Dataset root:", ACTIVE_DATASET)


In [ ]:
# ============================================================
# tf.data image pipeline
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE


# ------------------------------------------------------------
# Conservative augmentation
# ------------------------------------------------------------

augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(
            factor=0.015,
            seed=SEED
        ),
        tf.keras.layers.RandomTranslation(
            height_factor=0.03,
            width_factor=0.03,
            seed=SEED
        ),
        tf.keras.layers.RandomZoom(
            height_factor=(-0.05, 0.05),
            width_factor=(-0.05, 0.05),
            seed=SEED
        ),
        tf.keras.layers.RandomContrast(
            factor=0.10,
            seed=SEED
        ),
    ],
    name="conservative_medical_augmentation"
)


# ------------------------------------------------------------
# Decode + resize
# ------------------------------------------------------------

def load_xray(path, label):
    image = tf.io.read_file(path)

    # Dataset is confirmed grayscale PNG
    image = tf.image.decode_png(
        image,
        channels=1
    )

    image = tf.image.resize(
        image,
        IMAGE_SIZE,
        method="bilinear"
    )

    image = tf.cast(image, tf.float32)

    # MobileNetV2 expects 3 channels
    image = tf.image.grayscale_to_rgb(image)

    return image, label


# ------------------------------------------------------------
# MobileNetV2 preprocessing
# Input is still in approximately 0–255 range here.
# preprocess_input maps it to approximately [-1, 1].
# ------------------------------------------------------------

def preprocess_mobilenet(image, label):
    image = (
        tf.keras.applications.mobilenet_v2
        .preprocess_input(image)
    )

    return image, label


# ------------------------------------------------------------
# Training-only augmentation
# ------------------------------------------------------------

def augment_image(image, label):
    image = augmentation(
        image,
        training=True
    )

    return image, label


# ------------------------------------------------------------
# Dataset builder
# ------------------------------------------------------------

def build_dataset(
    dataframe,
    training=False
):
    image_paths = [
        str(ACTIVE_DATASET / p)
        for p in dataframe["image_relpath"]
    ]

    labels = dataframe["class_id"].astype("int32").values

    ds = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_xray,
        num_parallel_calls=AUTOTUNE
    )

    if training:
        ds = ds.map(
            augment_image,
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.map(
        preprocess_mobilenet,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE)

    ds = ds.prefetch(AUTOTUNE)

    return ds


train_ds = build_dataset(
    train_df,
    training=True
)

val_ds = build_dataset(
    val_df,
    training=False
)

test_ds = build_dataset(
    test_df,
    training=False
)

print(" tf.data pipelines created.")


In [ ]:
images, labels = next(iter(val_ds))

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)

print("\nImage value range after MobileNetV2 preprocessing:")
print("Min:", float(tf.reduce_min(images)))
print("Max:", float(tf.reduce_max(images)))

print("\nFirst 10 labels:")
print(labels[:10].numpy())


In [ ]:
# ============================================================
# Visual check after resize + augmentation + preprocessing
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Take one augmented training batch
aug_images, aug_labels = next(iter(train_ds))

# Show 12 examples
n_show = 12

fig, axes = plt.subplots(
    3,
    4,
    figsize=(12, 12)
)

axes = axes.flatten()

for i in range(n_show):

    # MobileNetV2 preprocessing mapped [0,255] -> [-1,1]
    # Convert back only for visualization
    img = aug_images[i].numpy()
    img = (img + 1.0) / 2.0
    img = np.clip(img, 0, 1)

    label_id = int(aug_labels[i].numpy())
    class_name = ID_TO_CLASS[label_id]

    axes[i].imshow(img)
    axes[i].set_title(class_name)
    axes[i].axis("off")

plt.suptitle(
    "Training Images After Resize and Conservative Augmentation",
    fontsize=15
)

plt.tight_layout()

# Save checkpoint figure
augmentation_figure_path = (
    DIRS["figures"] /
    "preprocessing_augmentation_sanity_check.png"
)

plt.savefig(
    augmentation_figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print(" Figure saved:")
print(augmentation_figure_path)


## 6. Baseline MobileNetV2

In [ ]:
# ============================================================
# Baseline Model: MobileNetV2 multiclass classifier
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, Model

NUM_CLASSES = 3

# ------------------------------------------------------------
# 1. Pretrained backbone
# ------------------------------------------------------------

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        3
    ),
    include_top=False,
    weights="imagenet"
)

# Stage 1: freeze ImageNet backbone
base_model.trainable = False


# ------------------------------------------------------------
# 2. Classification head
# ------------------------------------------------------------

inputs = layers.Input(
    shape=(
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        3
    ),
    name="xray_input"
)

x = base_model(
    inputs,
    training=False
)

x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = layers.Dropout(
    0.30,
    name="dropout"
)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="class_predictions"
)(x)


baseline_model = Model(
    inputs,
    outputs,
    name="MobileNetV2_3Class_Baseline"
)


# ------------------------------------------------------------
# 3. Compile
# ------------------------------------------------------------

baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)


# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

baseline_model.summary()

print("\nTotal parameters:",
      f"{baseline_model.count_params():,}")

print(
    "Trainable parameters:",
    f"{sum(
        tf.keras.backend.count_params(w)
        for w in baseline_model.trainable_weights
    ):,}"
)

print(
    "Non-trainable parameters:",
    f"{sum(
        tf.keras.backend.count_params(w)
        for w in baseline_model.non_trainable_weights
    ):,}"
)


In [ ]:
# ============================================================
# Baseline v1 — persistent experiment setup
# ============================================================

from pathlib import Path
import json
from datetime import datetime

EXPERIMENT_NAME = "baseline_mobilenetv2_v1"

EXPERIMENT_DIR = (
    PROJECT_ROOT /
    "experiments" /
    EXPERIMENT_NAME
)

CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoints"
BACKUP_DIR = EXPERIMENT_DIR / "training_backup"
LOG_DIR = EXPERIMENT_DIR / "logs"

for directory in [
    EXPERIMENT_DIR,
    CHECKPOINT_DIR,
    BACKUP_DIR,
    LOG_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Save experiment configuration
# ------------------------------------------------------------

experiment_config = {
    "experiment_name": EXPERIMENT_NAME,
    "task": "3-class chest X-ray classification",
    "classes": CLASS_TO_ID,
    "backbone": "MobileNetV2",
    "weights": "ImageNet",
    "input_size": [224, 224, 3],
    "stage_1_backbone_frozen": True,
    "stage_1_learning_rate": 1e-4,
    "batch_size": BATCH_SIZE,
    "loss": "SparseCategoricalCrossentropy",
    "optimizer": "Adam",
    "augmentation": {
        "rotation_factor": 0.015,
        "translation_factor": 0.03,
        "zoom_factor": 0.05,
        "contrast_factor": 0.10,
        "horizontal_flip": False
    },
    "split": "70/15/15 stratified by class + AP/PA position",
    "seed": SEED
}

with open(
    EXPERIMENT_DIR / "experiment_config.json",
    "w"
) as f:
    json.dump(
        experiment_config,
        f,
        indent=4
    )


# ------------------------------------------------------------
# Save the initial untrained baseline model
# ------------------------------------------------------------

initial_model_path = (
    EXPERIMENT_DIR /
    "baseline_initial_frozen.keras"
)

baseline_model.save(initial_model_path)

print(" Baseline experiment created:")
print(EXPERIMENT_DIR)

print("\n Initial model saved:")
print(initial_model_path)


In [ ]:
# ============================================================
# Final pre-training integration test
# ============================================================

sample_images, sample_labels = next(iter(val_ds))

sample_predictions = baseline_model(
    sample_images[:4],
    training=False
)

print("Prediction shape:", sample_predictions.shape)

print("\nPrediction probabilities:")
print(sample_predictions.numpy())

print("\nProbability sums:")
print(
    tf.reduce_sum(
        sample_predictions,
        axis=1
    ).numpy()
)

print("\nTrue labels:")
print(sample_labels[:4].numpy())


In [ ]:
# ============================================================
# Stage 1 — persistent callback setup
# Safe to rerun
# ============================================================

from pathlib import Path
import tensorflow as tf

EXPERIMENT_NAME = "baseline_mobilenetv2_v1"

EXPERIMENT_DIR = (
    PROJECT_ROOT /
    "experiments" /
    EXPERIMENT_NAME
)

CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoints"
LOG_DIR = EXPERIMENT_DIR / "logs"
BACKUP_DIR = EXPERIMENT_DIR / "training_backup"

for directory in [
    EXPERIMENT_DIR,
    CHECKPOINT_DIR,
    LOG_DIR,
    BACKUP_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)


# Best model according to validation loss
BEST_MODEL_PATH = CHECKPOINT_DIR / "stage1_best.keras"

# Latest completed epoch — useful after a disconnect
LATEST_MODEL_PATH = CHECKPOINT_DIR / "stage1_latest.keras"

# Persistent training history
CSV_LOG_PATH = LOG_DIR / "stage1_training_log.csv"


callbacks_stage1 = [

    # Keep best model
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    # Also overwrite with the latest completed epoch
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(LATEST_MODEL_PATH),
        save_best_only=False,
        save_weights_only=False,
        verbose=0
    ),

    # Epoch-by-epoch log saved on Drive
    tf.keras.callbacks.CSVLogger(
        filename=str(CSV_LOG_PATH),
        append=True
    ),

    # Stop if validation loss no longer improves
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    # Reduce learning rate if validation loss plateaus
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print(" Stage 1 callbacks ready")
print("Best model:  ", BEST_MODEL_PATH)
print("Latest model:", LATEST_MODEL_PATH)
print("CSV log:     ", CSV_LOG_PATH)


In [ ]:
# ============================================================
# Stage 1 Training — frozen MobileNetV2
# ============================================================

from datetime import datetime

STAGE1_EPOCHS = 10

print("STAGE 1 TRAINING")
print("-" * 60)
print("Backbone frozen:", not base_model.trainable)
print("Maximum epochs:", STAGE1_EPOCHS)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print(
    "Learning rate:",
    float(tf.keras.backend.get_value(
        baseline_model.optimizer.learning_rate
    ))
)

stage1_start = datetime.now()

history_stage1 = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE1_EPOCHS,
    callbacks=callbacks_stage1,
    verbose=1
)

stage1_end = datetime.now()

print("\n Stage 1 training finished.")
print("Duration:", stage1_end - stage1_start)


In [ ]:
from datetime import datetime
import pandas as pd
import tensorflow as tf

STAGE1_FINAL_PATH = (
    EXPERIMENT_DIR /
    "stage1_final.keras"
)

STAGE1_COMPLETE_PATH = (
    EXPERIMENT_DIR /
    "stage1_complete.txt"
)

# Reload complete training log
training_log = pd.read_csv(CSV_LOG_PATH)

# Load the best model explicitly
best_stage1_model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)

# Save an explicit final Stage 1 copy
best_stage1_model.save(
    STAGE1_FINAL_PATH
)

best_row = training_log.loc[
    training_log["val_loss"].idxmin()
]

best_epoch = int(best_row["epoch"]) + 1

completion_text = f"""
Stage 1: Frozen MobileNetV2 baseline

Completed: {datetime.now().isoformat()}

Total completed epochs: {len(training_log)}
Best epoch: {best_epoch}

Best validation loss: {best_row['val_loss']:.6f}
Validation accuracy at best epoch: {best_row['val_accuracy']:.6f}

Training accuracy at best epoch: {best_row['accuracy']:.6f}
Training loss at best epoch: {best_row['loss']:.6f}

Best checkpoint:
{BEST_MODEL_PATH}

Final Stage 1 model:
{STAGE1_FINAL_PATH}
"""

STAGE1_COMPLETE_PATH.write_text(
    completion_text.strip()
)

print("Stage 1 finalized.")
print("\nCompleted epochs:", len(training_log))
print("Best epoch:", best_epoch)
print(
    "Best validation accuracy:",
    round(float(best_row["val_accuracy"]), 4)
)
print(
    "Best validation loss:",
    round(float(best_row["val_loss"]), 4)
)

print("\nFinal model:")
print(STAGE1_FINAL_PATH)

print("\nCompletion record:")
print(STAGE1_COMPLETE_PATH)


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

CLASS_NAMES = [
    "Normal",
    "No Lung Opacity / Not Normal",
    "Lung Opacity"
]

# Reload the finalized best Stage 1 model
best_stage1_model = tf.keras.models.load_model(
    STAGE1_FINAL_PATH
)

# Generate probabilities on validation data
val_probabilities = best_stage1_model.predict(
    val_ds,
    verbose=1
)

# True and predicted labels
y_val_true = val_df["class_id"].astype(int).to_numpy()
y_val_pred = np.argmax(
    val_probabilities,
    axis=1
)

# Basic integrity checks
assert len(y_val_true) == len(val_probabilities)
assert val_probabilities.shape == (len(val_df), 3)

# Create prediction table
val_predictions = val_df.copy()

val_predictions["predicted_class_id"] = y_val_pred
val_predictions["predicted_class"] = [
    CLASS_NAMES[i]
    for i in y_val_pred
]

for i, class_name in enumerate(CLASS_NAMES):
    safe_name = (
        class_name
        .lower()
        .replace(" / ", "_")
        .replace(" ", "_")
    )

    val_predictions[
        f"prob_{safe_name}"
    ] = val_probabilities[:, i]

# Save predictions
VAL_PREDICTIONS_PATH = (
    EXPERIMENT_DIR /
    "validation_predictions_stage1.csv"
)

val_predictions.to_csv(
    VAL_PREDICTIONS_PATH,
    index=False
)

print("Validation samples:", len(y_val_true))
print("Prediction matrix shape:", val_probabilities.shape)

print("\nTrue class counts:")
print(
    pd.Series(y_val_true)
    .value_counts()
    .sort_index()
)

print("\nPredicted class counts:")
print(
    pd.Series(y_val_pred)
    .value_counts()
    .sort_index()
)

print("\nSaved to:")
print(VAL_PREDICTIONS_PATH)


In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
from sklearn.preprocessing import label_binarize
import pandas as pd
import numpy as np

CLASS_NAMES = [
    "Normal",
    "No Lung Opacity / Not Normal",
    "Lung Opacity"
]

# Overall metrics
accuracy = accuracy_score(
    y_val_true,
    y_val_pred
)

balanced_accuracy = balanced_accuracy_score(
    y_val_true,
    y_val_pred
)

macro_f1 = f1_score(
    y_val_true,
    y_val_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_val_true,
    y_val_pred,
    average="weighted"
)

# One-hot form for multiclass probability metrics
y_val_true_bin = label_binarize(
    y_val_true,
    classes=[0, 1, 2]
)

macro_roc_auc = roc_auc_score(
    y_val_true_bin,
    val_probabilities,
    average="macro",
    multi_class="ovr"
)

weighted_roc_auc = roc_auc_score(
    y_val_true_bin,
    val_probabilities,
    average="weighted",
    multi_class="ovr"
)

macro_pr_auc = average_precision_score(
    y_val_true_bin,
    val_probabilities,
    average="macro"
)

weighted_pr_auc = average_precision_score(
    y_val_true_bin,
    val_probabilities,
    average="weighted"
)

# Per-class report
report = classification_report(
    y_val_true,
    y_val_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    digits=4
)

report_df = pd.DataFrame(report).T

# Confusion matrix
cm = confusion_matrix(
    y_val_true,
    y_val_pred,
    labels=[0, 1, 2]
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True: {x}" for x in CLASS_NAMES],
    columns=[f"Pred: {x}" for x in CLASS_NAMES]
)

print("Stage 1 Validation Metrics")
print("-" * 60)
print(f"Accuracy:              {accuracy:.4f}")
print(f"Balanced accuracy:     {balanced_accuracy:.4f}")
print(f"Macro F1:              {macro_f1:.4f}")
print(f"Weighted F1:           {weighted_f1:.4f}")
print(f"Macro ROC-AUC (OvR):   {macro_roc_auc:.4f}")
print(f"Weighted ROC-AUC:      {weighted_roc_auc:.4f}")
print(f"Macro PR-AUC:          {macro_pr_auc:.4f}")
print(f"Weighted PR-AUC:       {weighted_pr_auc:.4f}")

print("\nPer-class performance:")
display(
    report_df.loc[
        CLASS_NAMES,
        ["precision", "recall", "f1-score", "support"]
    ]
)

print("\nConfusion matrix:")
display(cm_df)


In [ ]:
import json
import pandas as pd
import numpy as np

STAGE1_METRICS_PATH = (
    EXPERIMENT_DIR /
    "stage1_validation_metrics.json"
)

STAGE1_REPORT_PATH = (
    EXPERIMENT_DIR /
    "stage1_validation_classification_report.csv"
)

STAGE1_CM_PATH = (
    EXPERIMENT_DIR /
    "stage1_validation_confusion_matrix.csv"
)

stage1_metrics = {
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_accuracy),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_roc_auc_ovr": float(macro_roc_auc),
    "weighted_roc_auc_ovr": float(weighted_roc_auc),
    "macro_pr_auc": float(macro_pr_auc),
    "weighted_pr_auc": float(weighted_pr_auc)
}

with open(STAGE1_METRICS_PATH, "w") as f:
    json.dump(
        stage1_metrics,
        f,
        indent=4
    )

report_df.to_csv(
    STAGE1_REPORT_PATH
)

cm_df.to_csv(
    STAGE1_CM_PATH
)

print("Stage 1 validation results saved.")
print()
print("Metrics:")
print(STAGE1_METRICS_PATH)
print()
print("Classification report:")
print(STAGE1_REPORT_PATH)
print()
print("Confusion matrix:")
print(STAGE1_CM_PATH)


In [ ]:
print("Top-level model layers:")
print("-" * 70)

for i, layer in enumerate(best_stage1_model.layers):
    print(
        f"{i:2d} | "
        f"{layer.name:35s} | "
        f"{layer.__class__.__name__:25s} | "
        f"trainable={layer.trainable}"
    )


In [ ]:
backbone = best_stage1_model.get_layer(
    "mobilenetv2_1.00_224"
)

print("Total backbone layers:", len(backbone.layers))
print()
print("Last 40 backbone layers:")
print("-" * 90)

start_index = max(0, len(backbone.layers) - 40)

for i in range(start_index, len(backbone.layers)):
    layer = backbone.layers[i]

    print(
        f"{i:3d} | "
        f"{layer.name:38s} | "
        f"{layer.__class__.__name__:25s} | "
        f"trainable={layer.trainable}"
    )


In [ ]:
import tensorflow as tf

backbone = best_stage1_model.get_layer(
    "mobilenetv2_1.00_224"
)

FINE_TUNE_START = "block_13_expand"

start_index = next(
    i for i, layer in enumerate(backbone.layers)
    if layer.name == FINE_TUNE_START
)

# Allow the backbone to contain trainable layers
backbone.trainable = True

# Freeze everything before block 13
for layer in backbone.layers[:start_index]:
    layer.trainable = False

# Fine-tune the upper part, but keep all BatchNorm layers frozen
for layer in backbone.layers[start_index:]:
    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False
    else:
        layer.trainable = True

print("Fine-tuning starts at layer:", start_index)
print("Layer name:", backbone.layers[start_index].name)

print("\nTrainable backbone layers with weights:")
for i, layer in enumerate(backbone.layers):
    if layer.trainable and layer.weights:
        print(
            f"{i:3d} | "
            f"{layer.name:38s} | "
            f"{layer.__class__.__name__}"
        )

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in best_stage1_model.trainable_weights
)

non_trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in best_stage1_model.non_trainable_weights
)

print("\nTrainable parameters:", f"{trainable_params:,}")
print("Non-trainable parameters:", f"{non_trainable_params:,}")


In [ ]:
from pathlib import Path
import tensorflow as tf

STAGE2_DIR = (
    EXPERIMENT_DIR /
    "stage2_finetune_block13"
)

STAGE2_CHECKPOINT_DIR = (
    STAGE2_DIR /
    "checkpoints"
)

STAGE2_LOG_DIR = (
    STAGE2_DIR /
    "logs"
)

STAGE2_DIR.mkdir(
    parents=True,
    exist_ok=True
)

STAGE2_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

STAGE2_LOG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

STAGE2_BEST_PATH = (
    STAGE2_CHECKPOINT_DIR /
    "stage2_best.keras"
)

STAGE2_LATEST_PATH = (
    STAGE2_CHECKPOINT_DIR /
    "stage2_latest.keras"
)

STAGE2_LOG_PATH = (
    STAGE2_LOG_DIR /
    "stage2_training_log.csv"
)

STAGE2_INITIAL_PATH = (
    STAGE2_DIR /
    "stage2_initial.keras"
)

# Fine-tuning learning rate
STAGE2_LR = 1e-5

best_stage1_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=STAGE2_LR
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

# Save the exact model state before Stage 2 training
best_stage1_model.save(
    STAGE2_INITIAL_PATH
)

current_lr = float(
    tf.keras.backend.get_value(
        best_stage1_model.optimizer.learning_rate
    )
)

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in best_stage1_model.trainable_weights
)

print("Stage 2 model compiled.")
print("Learning rate:", current_lr)
print("Trainable parameters:", f"{trainable_params:,}")
print("Stage 2 initial model saved to:")
print(STAGE2_INITIAL_PATH)


In [ ]:
import pandas as pd
import tensorflow as tf

stage1_log = pd.read_csv(CSV_LOG_PATH)
stage1_best_val_loss = float(stage1_log["val_loss"].min())

callbacks_stage2 = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(STAGE2_BEST_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False,
        initial_value_threshold=stage1_best_val_loss,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(STAGE2_LATEST_PATH),
        save_best_only=False,
        save_weights_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(STAGE2_LOG_PATH),
        append=True
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Stage 2 callbacks configured.")
print("Stage 1 reference val_loss:", stage1_best_val_loss)
print("Stage 2 must beat this value to create a new best checkpoint.")


In [ ]:
from datetime import datetime

STAGE2_MAX_EPOCHS = 8

print("Starting Stage 2 fine-tuning")
print("-" * 60)
print("Fine-tuning from: block_13_expand")
print("Learning rate:", STAGE2_LR)
print("Maximum epochs:", STAGE2_MAX_EPOCHS)
print("Stage 1 reference val_loss:", stage1_best_val_loss)

stage2_start = datetime.now()

history_stage2 = best_stage1_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE2_MAX_EPOCHS,
    callbacks=callbacks_stage2,
    verbose=1
)

stage2_end = datetime.now()

print("\nStage 2 training finished.")
print("Duration:", stage2_end - stage2_start)


In [ ]:
from datetime import datetime
import pandas as pd
import tensorflow as tf

stage2_log = pd.read_csv(STAGE2_LOG_PATH)

best_stage2_row = stage2_log.loc[
    stage2_log["val_loss"].idxmin()
]

best_stage2_epoch = int(
    best_stage2_row["epoch"]
) + 1

STAGE2_FINAL_PATH = (
    STAGE2_DIR /
    "stage2_final.keras"
)

STAGE2_COMPLETE_PATH = (
    STAGE2_DIR /
    "stage2_complete.txt"
)

# Explicitly reload the best checkpoint
best_stage2_model = tf.keras.models.load_model(
    STAGE2_BEST_PATH
)

# Save a clearly named final copy
best_stage2_model.save(
    STAGE2_FINAL_PATH
)

completion_text = f"""
Stage 2: MobileNetV2 controlled fine-tuning

Fine-tuning start:
block_13_expand

BatchNormalization layers:
Frozen

Initial learning rate:
1e-5

Completed:
{datetime.now().isoformat()}

Fine-tuning epochs completed:
{len(stage2_log)}

Best Stage 2 epoch:
{best_stage2_epoch}

Best validation loss:
{best_stage2_row['val_loss']:.6f}

Validation accuracy at best epoch:
{best_stage2_row['val_accuracy']:.6f}

Training accuracy at best epoch:
{best_stage2_row['accuracy']:.6f}

Training loss at best epoch:
{best_stage2_row['loss']:.6f}

Best checkpoint:
{STAGE2_BEST_PATH}

Final Stage 2 model:
{STAGE2_FINAL_PATH}
"""

STAGE2_COMPLETE_PATH.write_text(
    completion_text.strip()
)

print("Stage 2 finalized.")
print()
print(
    "Fine-tuning epochs:",
    len(stage2_log)
)
print(
    "Best Stage 2 epoch:",
    best_stage2_epoch
)
print(
    "Best validation accuracy:",
    round(
        float(best_stage2_row["val_accuracy"]),
        4
    )
)
print(
    "Best validation loss:",
    round(
        float(best_stage2_row["val_loss"]),
        4
    )
)

print("\nFinal Stage 2 model:")
print(STAGE2_FINAL_PATH)

print("\nCompletion record:")
print(STAGE2_COMPLETE_PATH)


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

CLASS_NAMES = [
    "Normal",
    "No Lung Opacity / Not Normal",
    "Lung Opacity"
]

# Reload finalized best Stage 2 model
best_stage2_model = tf.keras.models.load_model(
    STAGE2_FINAL_PATH
)

# Generate validation probabilities
stage2_val_probabilities = best_stage2_model.predict(
    val_ds,
    verbose=1
)

# True and predicted labels
y_val_true = val_df["class_id"].astype(int).to_numpy()

stage2_y_val_pred = np.argmax(
    stage2_val_probabilities,
    axis=1
)

# Integrity checks
assert stage2_val_probabilities.shape == (
    len(val_df),
    3
)

assert len(stage2_y_val_pred) == len(y_val_true)

# Save prediction-level results
stage2_val_predictions = val_df.copy()

stage2_val_predictions["predicted_class_id"] = (
    stage2_y_val_pred
)

stage2_val_predictions["predicted_class"] = [
    CLASS_NAMES[i]
    for i in stage2_y_val_pred
]

for i, class_name in enumerate(CLASS_NAMES):

    safe_name = (
        class_name
        .lower()
        .replace(" / ", "_")
        .replace(" ", "_")
    )

    stage2_val_predictions[
        f"prob_{safe_name}"
    ] = stage2_val_probabilities[:, i]


STAGE2_VAL_PREDICTIONS_PATH = (
    STAGE2_DIR /
    "validation_predictions_stage2.csv"
)

stage2_val_predictions.to_csv(
    STAGE2_VAL_PREDICTIONS_PATH,
    index=False
)

print("Validation samples:", len(y_val_true))
print(
    "Prediction matrix shape:",
    stage2_val_probabilities.shape
)

print("\nTrue class counts:")
print(
    pd.Series(y_val_true)
    .value_counts()
    .sort_index()
)

print("\nPredicted class counts:")
print(
    pd.Series(stage2_y_val_pred)
    .value_counts()
    .sort_index()
)

print("\nSaved to:")
print(STAGE2_VAL_PREDICTIONS_PATH)


In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
from sklearn.preprocessing import label_binarize
import pandas as pd

# Overall metrics
stage2_accuracy = accuracy_score(
    y_val_true,
    stage2_y_val_pred
)

stage2_balanced_accuracy = balanced_accuracy_score(
    y_val_true,
    stage2_y_val_pred
)

stage2_macro_f1 = f1_score(
    y_val_true,
    stage2_y_val_pred,
    average="macro"
)

stage2_weighted_f1 = f1_score(
    y_val_true,
    stage2_y_val_pred,
    average="weighted"
)

# One-hot true labels for probability-based metrics
y_val_true_bin = label_binarize(
    y_val_true,
    classes=[0, 1, 2]
)

stage2_macro_roc_auc = roc_auc_score(
    y_val_true_bin,
    stage2_val_probabilities,
    average="macro",
    multi_class="ovr"
)

stage2_weighted_roc_auc = roc_auc_score(
    y_val_true_bin,
    stage2_val_probabilities,
    average="weighted",
    multi_class="ovr"
)

stage2_macro_pr_auc = average_precision_score(
    y_val_true_bin,
    stage2_val_probabilities,
    average="macro"
)

stage2_weighted_pr_auc = average_precision_score(
    y_val_true_bin,
    stage2_val_probabilities,
    average="weighted"
)

# Per-class metrics
stage2_report = classification_report(
    y_val_true,
    stage2_y_val_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    digits=4
)

stage2_report_df = pd.DataFrame(
    stage2_report
).T

# Confusion matrix
stage2_cm = confusion_matrix(
    y_val_true,
    stage2_y_val_pred,
    labels=[0, 1, 2]
)

stage2_cm_df = pd.DataFrame(
    stage2_cm,
    index=[f"True: {x}" for x in CLASS_NAMES],
    columns=[f"Pred: {x}" for x in CLASS_NAMES]
)

print("Stage 2 Validation Metrics")
print("-" * 60)
print(f"Accuracy:              {stage2_accuracy:.4f}")
print(f"Balanced accuracy:     {stage2_balanced_accuracy:.4f}")
print(f"Macro F1:              {stage2_macro_f1:.4f}")
print(f"Weighted F1:           {stage2_weighted_f1:.4f}")
print(f"Macro ROC-AUC (OvR):   {stage2_macro_roc_auc:.4f}")
print(f"Weighted ROC-AUC:      {stage2_weighted_roc_auc:.4f}")
print(f"Macro PR-AUC:          {stage2_macro_pr_auc:.4f}")
print(f"Weighted PR-AUC:       {stage2_weighted_pr_auc:.4f}")

print("\nPer-class performance:")
display(
    stage2_report_df.loc[
        CLASS_NAMES,
        ["precision", "recall", "f1-score", "support"]
    ]
)

print("\nConfusion matrix:")
display(stage2_cm_df)


In [ ]:
import json
import pandas as pd

STAGE2_METRICS_PATH = (
    STAGE2_DIR /
    "stage2_validation_metrics.json"
)

STAGE2_REPORT_PATH = (
    STAGE2_DIR /
    "stage2_validation_classification_report.csv"
)

STAGE2_CM_PATH = (
    STAGE2_DIR /
    "stage2_validation_confusion_matrix.csv"
)

COMPARISON_PATH = (
    STAGE2_DIR /
    "stage1_vs_stage2_validation_comparison.csv"
)

stage2_metrics = {
    "accuracy": float(stage2_accuracy),
    "balanced_accuracy": float(stage2_balanced_accuracy),
    "macro_f1": float(stage2_macro_f1),
    "weighted_f1": float(stage2_weighted_f1),
    "macro_roc_auc_ovr": float(stage2_macro_roc_auc),
    "weighted_roc_auc_ovr": float(stage2_weighted_roc_auc),
    "macro_pr_auc": float(stage2_macro_pr_auc),
    "weighted_pr_auc": float(stage2_weighted_pr_auc)
}

with open(STAGE2_METRICS_PATH, "w") as f:
    json.dump(
        stage2_metrics,
        f,
        indent=4
    )

stage2_report_df.to_csv(
    STAGE2_REPORT_PATH
)

stage2_cm_df.to_csv(
    STAGE2_CM_PATH
)

comparison_df = pd.DataFrame({
    "metric": [
        "accuracy",
        "balanced_accuracy",
        "macro_f1",
        "weighted_f1",
        "macro_roc_auc_ovr",
        "weighted_roc_auc_ovr",
        "macro_pr_auc",
        "weighted_pr_auc",
        "lung_opacity_precision",
        "lung_opacity_recall",
        "lung_opacity_f1"
    ],

    "stage1": [
        accuracy,
        balanced_accuracy,
        macro_f1,
        weighted_f1,
        macro_roc_auc,
        weighted_roc_auc,
        macro_pr_auc,
        weighted_pr_auc,
        report_df.loc[
            "Lung Opacity", "precision"
        ],
        report_df.loc[
            "Lung Opacity", "recall"
        ],
        report_df.loc[
            "Lung Opacity", "f1-score"
        ]
    ],

    "stage2": [
        stage2_accuracy,
        stage2_balanced_accuracy,
        stage2_macro_f1,
        stage2_weighted_f1,
        stage2_macro_roc_auc,
        stage2_weighted_roc_auc,
        stage2_macro_pr_auc,
        stage2_weighted_pr_auc,
        stage2_report_df.loc[
            "Lung Opacity", "precision"
        ],
        stage2_report_df.loc[
            "Lung Opacity", "recall"
        ],
        stage2_report_df.loc[
            "Lung Opacity", "f1-score"
        ]
    ]
})

comparison_df["absolute_change"] = (
    comparison_df["stage2"] -
    comparison_df["stage1"]
)

comparison_df.to_csv(
    COMPARISON_PATH,
    index=False
)

print("Stage 2 validation results saved.")
print()
display(comparison_df.round(4))

print("\nComparison saved to:")
print(COMPARISON_PATH)


## 7. Multi-scale CBAM model

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers


def cbam_block(x, ratio=8, name="cbam"):
    channels = int(x.shape[-1])
    hidden_units = max(channels // ratio, 1)

    # Channel attention
    shared_dense_1 = layers.Dense(
        hidden_units,
        activation="relu",
        use_bias=True,
        name=f"{name}_channel_fc1"
    )

    shared_dense_2 = layers.Dense(
        channels,
        use_bias=True,
        name=f"{name}_channel_fc2"
    )

    avg_pool = layers.GlobalAveragePooling2D(
        keepdims=True,
        name=f"{name}_avg_pool"
    )(x)

    max_pool = layers.GlobalMaxPooling2D(
        keepdims=True,
        name=f"{name}_max_pool"
    )(x)

    avg_attention = shared_dense_2(
        shared_dense_1(avg_pool)
    )

    max_attention = shared_dense_2(
        shared_dense_1(max_pool)
    )

    channel_attention = layers.Activation(
        "sigmoid",
        name=f"{name}_channel_sigmoid"
    )(
        layers.Add(
            name=f"{name}_channel_add"
        )([avg_attention, max_attention])
    )

    x = layers.Multiply(
        name=f"{name}_channel_multiply"
    )([x, channel_attention])

    # Spatial attention
    avg_spatial = keras.ops.mean(
        x,
        axis=-1,
        keepdims=True
    )

    max_spatial = keras.ops.max(
        x,
        axis=-1,
        keepdims=True
    )

    spatial_features = layers.Concatenate(
        axis=-1,
        name=f"{name}_spatial_concat"
    )([avg_spatial, max_spatial])

    spatial_attention = layers.Conv2D(
        filters=1,
        kernel_size=7,
        padding="same",
        activation="sigmoid",
        use_bias=False,
        name=f"{name}_spatial_conv"
    )(spatial_features)

    x = layers.Multiply(
        name=f"{name}_spatial_multiply"
    )([x, spatial_attention])

    return x


def build_enhanced_cbam_model(
    input_shape=(224, 224, 3),
    num_classes=3
):
    inputs = keras.Input(
        shape=input_shape,
        name="xray_input"
    )

    # Independent ImageNet backbone
    base_model = keras.applications.MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape
    )

    base_model.trainable = False

    # Explicit multi-scale feature levels
    high_feature = base_model.get_layer(
        "block_6_expand_relu"
    ).output

    mid_feature = base_model.get_layer(
        "block_13_expand_relu"
    ).output

    low_feature = base_model.get_layer(
        "out_relu"
    ).output

    multiscale_backbone = keras.Model(
        inputs=base_model.input,
        outputs=[
            high_feature,
            mid_feature,
            low_feature
        ],
        name="mobilenetv2_multiscale_backbone"
    )

    multiscale_backbone.trainable = False

    high, mid, low = multiscale_backbone(
        inputs,
        training=False
    )

    # 28x28 -> 14x14
    high = layers.Conv2D(
        128,
        kernel_size=3,
        strides=2,
        padding="same",
        activation="swish",
        name="high_scale_projection"
    )(high)

    # 14x14 -> 14x14
    mid = layers.Conv2D(
        128,
        kernel_size=3,
        strides=1,
        padding="same",
        activation="swish",
        name="mid_scale_projection"
    )(mid)

    # 7x7 -> 14x14
    low = layers.Conv2D(
        128,
        kernel_size=1,
        padding="same",
        activation="swish",
        name="low_scale_projection"
    )(low)

    low = layers.Resizing(
        14,
        14,
        interpolation="bilinear",
        name="low_scale_resize"
    )(low)

    # Multi-scale fusion
    x = layers.Concatenate(
        axis=-1,
        name="multiscale_fusion"
    )([high, mid, low])

    # CBAM
    x = cbam_block(
        x,
        ratio=8,
        name="cbam"
    )

    # Regularization
    x = layers.SpatialDropout2D(
        0.30,
        name="spatial_dropout"
    )(x)

    # Classification head
    x = layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = layers.Dense(
        256,
        activation="swish",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_256"
    )(x)

    x = layers.Dropout(
        0.60,
        name="dropout_06"
    )(x)

    outputs = layers.Dense(
        num_classes,
        activation="softmax",
        name="class_predictions"
    )(x)

    model = keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="enhanced_mobilenetv2_multiscale_cbam"
    )

    return model


enhanced_cbam_model = build_enhanced_cbam_model()

print("Model:", enhanced_cbam_model.name)
print("Output shape:", enhanced_cbam_model.output_shape)

backbone = enhanced_cbam_model.get_layer(
    "mobilenetv2_multiscale_backbone"
)

print("Backbone trainable:", backbone.trainable)

total_params = enhanced_cbam_model.count_params()

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in enhanced_cbam_model.trainable_weights
)

non_trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in enhanced_cbam_model.non_trainable_weights
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Non-trainable parameters:", f"{non_trainable_params:,}")


In [ ]:
from pathlib import Path
import tensorflow as tf

ENHANCED_DIR = (
    PROJECT_ROOT /
    "experiments" /
    "enhanced_mobilenetv2_multiscale_cbam_v1"
)

ENHANCED_CHECKPOINT_DIR = (
    ENHANCED_DIR /
    "checkpoints"
)

ENHANCED_LOG_DIR = (
    ENHANCED_DIR /
    "logs"
)

ENHANCED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ENHANCED_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ENHANCED_LOG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ENHANCED_STAGE1_BEST_PATH = (
    ENHANCED_CHECKPOINT_DIR /
    "stage1_best.keras"
)

ENHANCED_STAGE1_LATEST_PATH = (
    ENHANCED_CHECKPOINT_DIR /
    "stage1_latest.keras"
)

ENHANCED_STAGE1_LOG_PATH = (
    ENHANCED_LOG_DIR /
    "stage1_training_log.csv"
)

ENHANCED_INITIAL_PATH = (
    ENHANCED_DIR /
    "enhanced_initial_frozen.keras"
)

ENHANCED_STAGE1_LR = 1e-4

enhanced_cbam_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=ENHANCED_STAGE1_LR
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

enhanced_cbam_model.save(
    ENHANCED_INITIAL_PATH
)

current_lr = float(
    tf.keras.backend.get_value(
        enhanced_cbam_model.optimizer.learning_rate
    )
)

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in enhanced_cbam_model.trainable_weights
)

print("Enhanced model compiled.")
print("Learning rate:", current_lr)
print("Trainable parameters:", f"{trainable_params:,}")
print("Backbone trainable:",
      enhanced_cbam_model
      .get_layer("mobilenetv2_multiscale_backbone")
      .trainable)

print("\nInitial model saved to:")
print(ENHANCED_INITIAL_PATH)


In [ ]:
import tensorflow as tf

callbacks_enhanced_stage1 = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ENHANCED_STAGE1_BEST_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ENHANCED_STAGE1_LATEST_PATH),
        save_best_only=False,
        save_weights_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(ENHANCED_STAGE1_LOG_PATH),
        append=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Enhanced Stage 1 callbacks configured.")
print("Best checkpoint:")
print(ENHANCED_STAGE1_BEST_PATH)

print("\nLatest checkpoint:")
print(ENHANCED_STAGE1_LATEST_PATH)

print("\nTraining log:")
print(ENHANCED_STAGE1_LOG_PATH)


In [ ]:
import json
from datetime import datetime

ENHANCED_STATE_PATH = (
    ENHANCED_DIR /
    "experiment_state.json"
)

experiment_state = {
    "status": "ready_for_stage1_training",
    "model_name": "enhanced_mobilenetv2_multiscale_cbam_v1",
    "task": "3_class_classification",
    "classes": [
        "Normal",
        "No Lung Opacity / Not Normal",
        "Lung Opacity"
    ],
    "initial_model_path": str(ENHANCED_INITIAL_PATH),
    "stage1_best_path": str(ENHANCED_STAGE1_BEST_PATH),
    "stage1_latest_path": str(ENHANCED_STAGE1_LATEST_PATH),
    "stage1_log_path": str(ENHANCED_STAGE1_LOG_PATH),
    "stage1_learning_rate": 1e-4,
    "stage1_max_epochs": 10,
    "batch_size": 32,
    "backbone": "MobileNetV2 ImageNet",
    "backbone_frozen": True,
    "multiscale_features": [
        "block_6_expand_relu",
        "block_13_expand_relu",
        "out_relu"
    ],
    "attention": "CBAM",
    "class_weights": False,
    "test_set_used": False,
    "last_updated": datetime.now().isoformat()
}

with open(ENHANCED_STATE_PATH, "w") as f:
    json.dump(
        experiment_state,
        f,
        indent=4
    )

print("Experiment state saved.")
print(ENHANCED_STATE_PATH)


In [ ]:
import tensorflow as tf

ENHANCED_STAGE1_LR = 1e-4

enhanced_cbam_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=ENHANCED_STAGE1_LR
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

current_lr = float(
    tf.keras.backend.get_value(
        enhanced_cbam_model.optimizer.learning_rate
    )
)

print("Fresh optimizer created.")
print("Learning rate:", current_lr)
print(
    "Optimizer variables before training:",
    len(enhanced_cbam_model.optimizer.variables)
)


In [ ]:
import pandas as pd
import tensorflow as tf
from pathlib import Path

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

SPLITS_DIR = PROJECT_ROOT / "data" / "splits"

train_df = pd.read_csv(
    SPLITS_DIR / "train.csv"
)

val_df = pd.read_csv(
    SPLITS_DIR / "validation.csv"
)

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(
        factor=0.015,
        seed=SEED
    ),
    tf.keras.layers.RandomTranslation(
        height_factor=0.03,
        width_factor=0.03,
        seed=SEED
    ),
    tf.keras.layers.RandomZoom(
        height_factor=(-0.05, 0.05),
        width_factor=(-0.05, 0.05),
        seed=SEED
    ),
    tf.keras.layers.RandomContrast(
        factor=0.10,
        seed=SEED
    )
])


def load_xray(path, label):
    image = tf.io.read_file(path)

    image = tf.image.decode_png(
        image,
        channels=1
    )

    image = tf.image.resize(
        image,
        IMAGE_SIZE,
        method="bilinear"
    )

    image = tf.cast(
        image,
        tf.float32
    )

    image = tf.image.grayscale_to_rgb(
        image
    )

    return image, label


def augment_image(image, label):
    image = augmentation(
        image,
        training=True
    )

    return image, label


def preprocess_mobilenet(image, label):
    image = tf.keras.applications.mobilenet_v2.preprocess_input(
        image
    )

    return image, label


def build_dataset(dataframe, training=False):
    image_paths = [
        str(ACTIVE_DATASET / p)
        for p in dataframe["image_relpath"]
    ]

    labels = (
        dataframe["class_id"]
        .astype("int32")
        .values
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_xray,
        num_parallel_calls=AUTOTUNE
    )

    if training:
        ds = ds.map(
            augment_image,
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.map(
        preprocess_mobilenet,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        AUTOTUNE
    )

    return ds


train_ds = build_dataset(
    train_df,
    training=True
)

val_ds = build_dataset(
    val_df,
    training=False
)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Pipelines rebuilt.")


In [ ]:
from pathlib import Path
import tensorflow as tf

ENHANCED_STAGE1_BEST_PATH = Path(
    experiment_state["stage1_best_path"]
)

ENHANCED_STAGE1_LATEST_PATH = Path(
    experiment_state["stage1_latest_path"]
)

ENHANCED_STAGE1_LOG_PATH = Path(
    experiment_state["stage1_log_path"]
)

callbacks_enhanced_stage1 = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ENHANCED_STAGE1_BEST_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ENHANCED_STAGE1_LATEST_PATH),
        save_best_only=False,
        save_weights_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(ENHANCED_STAGE1_LOG_PATH),
        append=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Enhanced Stage 1 callbacks restored.")
print("Best checkpoint:", ENHANCED_STAGE1_BEST_PATH)
print("Latest checkpoint:", ENHANCED_STAGE1_LATEST_PATH)
print("Training log:", ENHANCED_STAGE1_LOG_PATH)


In [ ]:
ENHANCED_STAGE1_EPOCHS = int(
    experiment_state["stage1_max_epochs"]
)

history_enhanced_stage1 = enhanced_cbam_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=ENHANCED_STAGE1_EPOCHS,
    callbacks=callbacks_enhanced_stage1,
    verbose=1
)


In [ ]:
import json
from pathlib import Path
from datetime import datetime

ENHANCED_STAGE1_FINAL_PATH = (
    ENHANCED_DIR / "stage1_final.keras"
)

enhanced_cbam_model.save(
    ENHANCED_STAGE1_FINAL_PATH
)

experiment_state["status"] = "stage1_complete"
experiment_state["stage1_best_epoch"] = 8
experiment_state["stage1_best_val_loss"] = 0.67285
experiment_state["stage1_best_val_accuracy"] = 0.7102
experiment_state["stage1_final_model_path"] = str(
    ENHANCED_STAGE1_FINAL_PATH
)
experiment_state["stage1_completed_at"] = (
    datetime.now().isoformat()
)

with open(ENHANCED_STATE_PATH, "w") as f:
    json.dump(
        experiment_state,
        f,
        indent=2
    )

print("Enhanced Stage 1 saved.")
print("Final model:", ENHANCED_STAGE1_FINAL_PATH)
print("Best epoch:", 8)
print("Best val_accuracy:", 0.7102)
print("Best val_loss:", 0.67285)


In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

CLASS_NAMES = [
    "Normal",
    "No Lung Opacity / Not Normal",
    "Lung Opacity"
]

print("Generating validation predictions...")

val_prob = enhanced_cbam_model.predict(
    val_ds,
    verbose=1
)

val_pred = np.argmax(
    val_prob,
    axis=1
)

val_true = val_df["class_id"].astype(int).to_numpy()

val_true_onehot = np.eye(3)[val_true]

accuracy = accuracy_score(
    val_true,
    val_pred
)

balanced_acc = balanced_accuracy_score(
    val_true,
    val_pred
)

macro_f1 = f1_score(
    val_true,
    val_pred,
    average="macro"
)

weighted_f1 = f1_score(
    val_true,
    val_pred,
    average="weighted"
)

macro_roc_auc = roc_auc_score(
    val_true_onehot,
    val_prob,
    average="macro",
    multi_class="ovr"
)

weighted_roc_auc = roc_auc_score(
    val_true_onehot,
    val_prob,
    average="weighted",
    multi_class="ovr"
)

macro_pr_auc = average_precision_score(
    val_true_onehot,
    val_prob,
    average="macro"
)

weighted_pr_auc = average_precision_score(
    val_true_onehot,
    val_prob,
    average="weighted"
)

report = classification_report(
    val_true,
    val_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    val_true,
    val_pred
)

metrics = {
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_roc_auc_ovr": float(macro_roc_auc),
    "weighted_roc_auc_ovr": float(weighted_roc_auc),
    "macro_pr_auc": float(macro_pr_auc),
    "weighted_pr_auc": float(weighted_pr_auc)
}

predictions_df = val_df.copy()

predictions_df["predicted_class_id"] = val_pred
predictions_df["prob_normal"] = val_prob[:, 0]
predictions_df["prob_no_lung_opacity_not_normal"] = val_prob[:, 1]
predictions_df["prob_lung_opacity"] = val_prob[:, 2]

predictions_path = (
    ENHANCED_DIR / "validation_predictions_stage1.csv"
)

metrics_path = (
    ENHANCED_DIR / "stage1_validation_metrics.json"
)

report_path = (
    ENHANCED_DIR / "stage1_validation_classification_report.csv"
)

cm_path = (
    ENHANCED_DIR / "stage1_validation_confusion_matrix.csv"
)

predictions_df.to_csv(
    predictions_path,
    index=False
)

with open(metrics_path, "w") as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

pd.DataFrame(report).T.to_csv(
    report_path
)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    cm_path
)

print("\nOverall validation metrics")
print("--------------------------------")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")
print(f"Macro ROC-AUC:     {macro_roc_auc:.4f}")
print(f"Weighted ROC-AUC:  {weighted_roc_auc:.4f}")
print(f"Macro PR-AUC:      {macro_pr_auc:.4f}")
print(f"Weighted PR-AUC:   {weighted_pr_auc:.4f}")

print("\nPer-class performance")
print("--------------------------------")

for class_name in CLASS_NAMES:
    r = report[class_name]

    print(
        f"{class_name}: "
        f"precision={r['precision']:.4f}, "
        f"recall={r['recall']:.4f}, "
        f"F1={r['f1-score']:.4f}"
    )

print("\nConfusion matrix")
print("--------------------------------")
print(cm)

print("\nPrediction counts")
print("--------------------------------")
print("True:     ", np.bincount(val_true, minlength=3))
print("Predicted:", np.bincount(val_pred, minlength=3))

print("\nValidation results saved.")


In [ ]:
backbone = enhanced_cbam_model.get_layer(
    "mobilenetv2_multiscale_backbone"
)

print("Backbone name:", backbone.name)
print("Number of backbone layers:", len(backbone.layers))

print("\nLayers from index 108 onward:")
for i, layer in enumerate(backbone.layers[108:], start=108):
    print(
        i,
        layer.name,
        type(layer).__name__,
        "trainable=",
        layer.trainable
    )


In [ ]:
import tensorflow as tf

backbone = enhanced_cbam_model.get_layer(
    "mobilenetv2_multiscale_backbone"
)

FINE_TUNE_FROM = 116
STAGE2_LR = 1e-5

backbone.trainable = True

for i, layer in enumerate(backbone.layers):

    if i < FINE_TUNE_FROM:
        layer.trainable = False

    elif isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

    else:
        layer.trainable = True


enhanced_cbam_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=STAGE2_LR
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)


trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in enhanced_cbam_model.trainable_weights
)

nontrainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in enhanced_cbam_model.non_trainable_weights
)

print("Fine-tuning starts from:")
print(
    FINE_TUNE_FROM,
    backbone.layers[FINE_TUNE_FROM].name
)

print("\nBackbone trainable:", backbone.trainable)
print("Learning rate:", float(
    tf.keras.backend.get_value(
        enhanced_cbam_model.optimizer.learning_rate
    )
))

print("\nTrainable parameters:", f"{trainable_params:,}")
print("Non-trainable parameters:", f"{nontrainable_params:,}")

print("\nTrainable backbone layers with weights:")

for i, layer in enumerate(backbone.layers):
    if layer.trainable and layer.weights:
        print(
            i,
            layer.name,
            type(layer).__name__
        )


In [ ]:
from pathlib import Path
import json
from datetime import datetime

ENHANCED_STAGE2_DIR = (
    ENHANCED_DIR / "stage2_finetune_block13"
)


In [ ]:
from pathlib import Path
import json
from datetime import datetime

ENHANCED_STAGE2_DIR = (
    ENHANCED_DIR / "stage2_finetune_block13"
)

ENHANCED_STAGE2_CHECKPOINTS = (
    ENHANCED_STAGE2_DIR / "checkpoints"
)

ENHANCED_STAGE2_LOGS = (
    ENHANCED_STAGE2_DIR / "logs"
)

ENHANCED_STAGE2_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ENHANCED_STAGE2_CHECKPOINTS.mkdir(
    parents=True,
    exist_ok=True
)

ENHANCED_STAGE2_LOGS.mkdir(
    parents=True,
    exist_ok=True
)

ENHANCED_STAGE2_INITIAL_PATH = (
    ENHANCED_STAGE2_DIR / "stage2_initial.keras"
)

ENHANCED_STAGE2_BEST_PATH = (
    ENHANCED_STAGE2_CHECKPOINTS / "stage2_best.keras"
)

ENHANCED_STAGE2_LATEST_PATH = (
    ENHANCED_STAGE2_CHECKPOINTS / "stage2_latest.keras"
)

ENHANCED_STAGE2_LOG_PATH = (
    ENHANCED_STAGE2_LOGS / "stage2_training_log.csv"
)

enhanced_cbam_model.save(
    ENHANCED_STAGE2_INITIAL_PATH
)

experiment_state["status"] = "ready_for_stage2_training"
experiment_state["stage2_fine_tune_from_index"] = 116
experiment_state["stage2_fine_tune_from_layer"] = "block_13_expand"
experiment_state["stage2_learning_rate"] = 1e-5
experiment_state["stage2_trainable_parameters"] = 2849045
experiment_state["stage2_initial_model_path"] = str(
    ENHANCED_STAGE2_INITIAL_PATH
)
experiment_state["stage2_best_path"] = str(
    ENHANCED_STAGE2_BEST_PATH
)
experiment_state["stage2_latest_path"] = str(
    ENHANCED_STAGE2_LATEST_PATH
)
experiment_state["stage2_log_path"] = str(
    ENHANCED_STAGE2_LOG_PATH
)
experiment_state["stage2_prepared_at"] = (
    datetime.now().isoformat()
)

with open(ENHANCED_STATE_PATH, "w") as f:
    json.dump(
        experiment_state,
        f,
        indent=2
    )

print("Enhanced Stage 2 starting point saved.")
print("Initial model:", ENHANCED_STAGE2_INITIAL_PATH)
print("Status:", experiment_state["status"])


In [ ]:
callbacks_enhanced_stage2 = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ENHANCED_STAGE2_BEST_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ENHANCED_STAGE2_LATEST_PATH),
        save_best_only=False,
        save_weights_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(ENHANCED_STAGE2_LOG_PATH),
        append=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Enhanced Stage 2 callbacks ready.")
print("Best checkpoint:", ENHANCED_STAGE2_BEST_PATH)
print("Latest checkpoint:", ENHANCED_STAGE2_LATEST_PATH)
print("Training log:", ENHANCED_STAGE2_LOG_PATH)


In [ ]:
ENHANCED_STAGE2_EPOCHS = 8

history_enhanced_stage2 = enhanced_cbam_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=ENHANCED_STAGE2_EPOCHS,
    callbacks=callbacks_enhanced_stage2,
    verbose=1
)


In [ ]:
import tensorflow as tf
import json
from datetime import datetime

ENHANCED_STAGE2_BEST_PATH = (
    ENHANCED_DIR
    / "stage2_finetune_block13"
    / "checkpoints"
    / "stage2_best.keras"
)

ENHANCED_STAGE2_FINAL_PATH = (
    ENHANCED_DIR
    / "stage2_finetune_block13"
    / "stage2_final.keras"
)

enhanced_stage2_best_model = tf.keras.models.load_model(
    ENHANCED_STAGE2_BEST_PATH
)

enhanced_stage2_best_model.save(
    ENHANCED_STAGE2_FINAL_PATH
)

experiment_state["status"] = "stage2_complete"
experiment_state["stage2_best_epoch"] = 2
experiment_state["stage2_best_val_loss"] = 0.6691204905509949
experiment_state["stage2_best_val_accuracy"] = 0.714464
experiment_state["stage2_final_model_path"] = str(
    ENHANCED_STAGE2_FINAL_PATH
)
experiment_state["stage2_completed_at"] = (
    datetime.now().isoformat()
)

with open(ENHANCED_STATE_PATH, "w") as f:
    json.dump(
        experiment_state,
        f,
        indent=2
    )

print("Enhanced Stage 2 finalized.")
print("Best epoch:", 2)
print("Best val_accuracy:", 0.714464)
print("Best val_loss:", 0.6691204905509949)
print("Final model:", ENHANCED_STAGE2_FINAL_PATH)


In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

CLASS_NAMES = [
    "Normal",
    "No Lung Opacity / Not Normal",
    "Lung Opacity"
]

print("Generating Stage 2 validation predictions...")

val_prob_stage2 = enhanced_stage2_best_model.predict(
    val_ds,
    verbose=1
)

val_pred_stage2 = np.argmax(
    val_prob_stage2,
    axis=1
)

val_true = (
    val_df["class_id"]
    .astype(int)
    .to_numpy()
)

val_true_onehot = np.eye(3)[val_true]

accuracy = accuracy_score(
    val_true,
    val_pred_stage2
)

balanced_acc = balanced_accuracy_score(
    val_true,
    val_pred_stage2
)

macro_f1 = f1_score(
    val_true,
    val_pred_stage2,
    average="macro"
)

weighted_f1 = f1_score(
    val_true,
    val_pred_stage2,
    average="weighted"
)

macro_roc_auc = roc_auc_score(
    val_true_onehot,
    val_prob_stage2,
    average="macro",
    multi_class="ovr"
)

weighted_roc_auc = roc_auc_score(
    val_true_onehot,
    val_prob_stage2,
    average="weighted",
    multi_class="ovr"
)

macro_pr_auc = average_precision_score(
    val_true_onehot,
    val_prob_stage2,
    average="macro"
)

weighted_pr_auc = average_precision_score(
    val_true_onehot,
    val_prob_stage2,
    average="weighted"
)

report = classification_report(
    val_true,
    val_pred_stage2,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    val_true,
    val_pred_stage2
)

metrics = {
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_roc_auc_ovr": float(macro_roc_auc),
    "weighted_roc_auc_ovr": float(weighted_roc_auc),
    "macro_pr_auc": float(macro_pr_auc),
    "weighted_pr_auc": float(weighted_pr_auc)
}

STAGE2_RESULTS_DIR = (
    ENHANCED_DIR
    / "stage2_finetune_block13"
)

predictions_df = val_df.copy()

predictions_df["predicted_class_id"] = (
    val_pred_stage2
)

predictions_df["prob_normal"] = (
    val_prob_stage2[:, 0]
)

predictions_df[
    "prob_no_lung_opacity_not_normal"
] = val_prob_stage2[:, 1]

predictions_df["prob_lung_opacity"] = (
    val_prob_stage2[:, 2]
)

predictions_df.to_csv(
    STAGE2_RESULTS_DIR
    / "validation_predictions_stage2.csv",
    index=False
)

with open(
    STAGE2_RESULTS_DIR
    / "stage2_validation_metrics.json",
    "w"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

pd.DataFrame(report).T.to_csv(
    STAGE2_RESULTS_DIR
    / "stage2_validation_classification_report.csv"
)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    STAGE2_RESULTS_DIR
    / "stage2_validation_confusion_matrix.csv"
)

print("\nOverall validation metrics")
print("--------------------------------")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")
print(f"Macro ROC-AUC:     {macro_roc_auc:.4f}")
print(f"Weighted ROC-AUC:  {weighted_roc_auc:.4f}")
print(f"Macro PR-AUC:      {macro_pr_auc:.4f}")
print(f"Weighted PR-AUC:   {weighted_pr_auc:.4f}")

print("\nPer-class performance")
print("--------------------------------")

for class_name in CLASS_NAMES:
    r = report[class_name]

    print(
        f"{class_name}: "
        f"precision={r['precision']:.4f}, "
        f"recall={r['recall']:.4f}, "
        f"F1={r['f1-score']:.4f}"
    )

print("\nConfusion matrix")
print("--------------------------------")
print(cm)

print("\nPrediction counts")
print("--------------------------------")
print(
    "True:     ",
    np.bincount(
        val_true,
        minlength=3
    )
)
print(
    "Predicted:",
    np.bincount(
        val_pred_stage2,
        minlength=3
    )
)

print("\nStage 2 validation results saved.")


## 8. Multi-scale Transformer + residual model

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from pathlib import Path

NUM_CLASSES = 3
INPUT_SHAPE = (224, 224, 3)

ADVANCED_MODEL_NAME = (
    "enhanced_mobilenetv2_multiscale_transformer_residual"
)

ADVANCED_DIR = (
    PROJECT_ROOT
    / "experiments"
    / "enhanced_mobilenetv2_multiscale_transformer_residual_v1"
)

ADVANCED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def residual_bottleneck(x, block_id):
    shortcut = x

    y = layers.Conv2D(
        192,
        kernel_size=1,
        padding="same",
        activation="swish",
        name=f"res{block_id}_reduce"
    )(x)

    y = layers.SeparableConv2D(
        192,
        kernel_size=3,
        padding="same",
        activation="swish",
        name=f"res{block_id}_separable"
    )(y)

    y = layers.Conv2D(
        384,
        kernel_size=1,
        padding="same",
        name=f"res{block_id}_expand"
    )(y)

    x = layers.Add(
        name=f"res{block_id}_add"
    )([shortcut, y])

    x = layers.Activation(
        "swish",
        name=f"res{block_id}_activation"
    )(x)

    return x


inputs = keras.Input(
    shape=INPUT_SHAPE,
    name="input_image"
)

base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=INPUT_SHAPE
)

high_feature = base_model.get_layer(
    "block_6_expand_relu"
).output

mid_feature = base_model.get_layer(
    "block_13_expand_relu"
).output

low_feature = base_model.get_layer(
    "out_relu"
).output

backbone = keras.Model(
    inputs=base_model.input,
    outputs=[
        high_feature,
        mid_feature,
        low_feature
    ],
    name="mobilenetv2_transformer_backbone"
)

backbone.trainable = False

high, mid, low = backbone(
    inputs,
    training=False
)

high = layers.SeparableConv2D(
    128,
    kernel_size=3,
    strides=2,
    padding="same",
    activation="swish",
    name="high_scale_projection"
)(high)

mid = layers.SeparableConv2D(
    128,
    kernel_size=3,
    strides=1,
    padding="same",
    activation="swish",
    name="mid_scale_projection"
)(mid)

low = layers.SeparableConv2D(
    128,
    kernel_size=3,
    strides=1,
    padding="same",
    activation="swish",
    name="low_scale_projection"
)(low)

low = layers.Resizing(
    14,
    14,
    interpolation="bilinear",
    name="low_scale_resize"
)(low)

x = layers.Concatenate(
    axis=-1,
    name="multiscale_fusion"
)([
    high,
    mid,
    low
])

x = layers.Reshape(
    (196, 384),
    name="spatial_tokens"
)(x)

attention_input = layers.LayerNormalization(
    name="transformer_attention_norm"
)(x)

attention_output = layers.MultiHeadAttention(
    num_heads=4,
    key_dim=64,
    dropout=0.10,
    name="transformer_attention"
)(
    attention_input,
    attention_input
)

x = layers.Add(
    name="transformer_attention_residual"
)([
    x,
    attention_output
])

ffn_input = layers.LayerNormalization(
    name="transformer_ffn_norm"
)(x)

ffn = layers.Dense(
    512,
    activation="swish",
    name="transformer_ffn_expand"
)(ffn_input)

ffn = layers.Dropout(
    0.10,
    name="transformer_ffn_dropout"
)(ffn)

ffn = layers.Dense(
    384,
    name="transformer_ffn_project"
)(ffn)

x = layers.Add(
    name="transformer_ffn_residual"
)([
    x,
    ffn
])

x = layers.Reshape(
    (14, 14, 384),
    name="transformer_feature_map"
)(x)

for block_id in range(1, 4):
    x = residual_bottleneck(
        x,
        block_id
    )

x = layers.SpatialDropout2D(
    0.30,
    name="spatial_dropout"
)(x)

x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = layers.Dense(
    512,
    activation="swish",
    kernel_regularizer=regularizers.l2(1e-4),
    name="dense_512"
)(x)

x = layers.Dropout(
    0.50,
    name="dropout_512"
)(x)

x = layers.Dense(
    256,
    activation="swish",
    kernel_regularizer=regularizers.l2(1e-4),
    name="dense_256"
)(x)

x = layers.Dropout(
    0.70,
    name="dropout_256"
)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="class_predictions"
)(x)

advanced_model = keras.Model(
    inputs=inputs,
    outputs=outputs,
    name=ADVANCED_MODEL_NAME
)

advanced_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in advanced_model.trainable_weights
)

nontrainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in advanced_model.non_trainable_weights
)

print("Model:", advanced_model.name)
print("Output shape:", advanced_model.output_shape)
print("Backbone trainable:", backbone.trainable)
print("Trainable parameters:", f"{trainable_params:,}")
print("Non-trainable parameters:", f"{nontrainable_params:,}")
print("Total parameters:", f"{advanced_model.count_params():,}")


In [ ]:
import json
from datetime import datetime

ADVANCED_INITIAL_PATH = (
    ADVANCED_DIR
    / "advanced_initial_frozen.keras"
)

ADVANCED_STATE_PATH = (
    ADVANCED_DIR
    / "experiment_state.json"
)

ADVANCED_SUMMARY_PATH = (
    ADVANCED_DIR
    / "model_summary.txt"
)

advanced_model.save(
    ADVANCED_INITIAL_PATH
)

with open(ADVANCED_SUMMARY_PATH, "w") as f:
    advanced_model.summary(
        print_fn=lambda line: f.write(line + "\n")
    )

advanced_state = {
    "experiment_name":
        "enhanced_mobilenetv2_multiscale_transformer_residual_v1",

    "model_name":
        advanced_model.name,

    "status":
        "ready_for_stage1_training",

    "task":
        "RSNA multiclass lung opacity classification",

    "classes": [
        "Normal",
        "No Lung Opacity / Not Normal",
        "Lung Opacity"
    ],

    "num_classes": 3,

    "input_shape": [
        224,
        224,
        3
    ],

    "backbone":
        "MobileNetV2 ImageNet",

    "backbone_frozen": True,

    "multiscale_feature_layers": [
        "block_6_expand_relu",
        "block_13_expand_relu",
        "out_relu"
    ],

    "multiscale_projection":
        "SeparableConv2D",

    "fusion_channels": 384,

    "transformer": {
        "num_tokens": 196,
        "embedding_dim": 384,
        "num_heads": 4,
        "key_dim": 64,
        "ffn_dim": 512,
        "dropout": 0.10
    },

    "residual_bottleneck_blocks": 3,

    "stage1_learning_rate": 1e-4,

    "stage1_max_epochs": 15,

    "early_stopping_patience": 3,

    "batch_size": 32,

    "class_weights_used": False,

    "test_set_used": False,

    "trainable_parameters": 1960643,

    "nontrainable_parameters": 2257984,

    "total_parameters": 4218627,

    "initial_model_path":
        str(ADVANCED_INITIAL_PATH),

    "model_summary_path":
        str(ADVANCED_SUMMARY_PATH),

    "created_at":
        datetime.now().isoformat(),

    "historical_note":
        (
            "This is a reproducible 2026 implementation inspired by "
            "the advanced architecture explored in the 2024 project. "
            "Implementation details not explicitly documented in the "
            "2024 work are treated as current reconstruction choices."
        )
}

with open(ADVANCED_STATE_PATH, "w") as f:
    json.dump(
        advanced_state,
        f,
        indent=2
    )

print("Advanced model starting point saved.")
print("Model:", ADVANCED_INITIAL_PATH)
print("State:", ADVANCED_STATE_PATH)
print("Summary:", ADVANCED_SUMMARY_PATH)
print("Status:", advanced_state["status"])
print("Test set used:", advanced_state["test_set_used"])


In [ ]:
ADVANCED_STAGE1_CHECKPOINTS = (
    ADVANCED_DIR / "checkpoints"
)

ADVANCED_STAGE1_LOGS = (
    ADVANCED_DIR / "logs"
)

ADVANCED_STAGE1_CHECKPOINTS.mkdir(
    parents=True,
    exist_ok=True
)

ADVANCED_STAGE1_LOGS.mkdir(
    parents=True,
    exist_ok=True
)

ADVANCED_STAGE1_BEST_PATH = (
    ADVANCED_STAGE1_CHECKPOINTS
    / "stage1_best.keras"
)

ADVANCED_STAGE1_LATEST_PATH = (
    ADVANCED_STAGE1_CHECKPOINTS
    / "stage1_latest.keras"
)

ADVANCED_STAGE1_LOG_PATH = (
    ADVANCED_STAGE1_LOGS
    / "stage1_training_log.csv"
)

callbacks_advanced_stage1 = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ADVANCED_STAGE1_BEST_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ADVANCED_STAGE1_LATEST_PATH),
        save_best_only=False,
        save_weights_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(ADVANCED_STAGE1_LOG_PATH),
        append=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Advanced Stage 1 callbacks ready.")
print("Best checkpoint:", ADVANCED_STAGE1_BEST_PATH)
print("Latest checkpoint:", ADVANCED_STAGE1_LATEST_PATH)
print("Training log:", ADVANCED_STAGE1_LOG_PATH)
print("Maximum epochs:", 15)


In [ ]:
import json

ADVANCED_STATE_PATH = (
    ADVANCED_DIR
    / "experiment_state.json"
)

with open(ADVANCED_STATE_PATH, "r") as f:
    advanced_state = json.load(f)

advanced_state["stage1_max_epochs"] = 10

with open(ADVANCED_STATE_PATH, "w") as f:
    json.dump(
        advanced_state,
        f,
        indent=2
    )

print("Stage 1 maximum epochs updated to:", advanced_state["stage1_max_epochs"])


In [ ]:
ADVANCED_STAGE1_EPOCHS = 10

history_advanced_stage1 = advanced_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=ADVANCED_STAGE1_EPOCHS,
    callbacks=callbacks_advanced_stage1,
    verbose=1
)


In [ ]:
import json
from datetime import datetime

ADVANCED_STAGE1_FINAL_PATH = (
    ADVANCED_DIR / "stage1_final.keras"
)

advanced_model.save(
    ADVANCED_STAGE1_FINAL_PATH
)

advanced_state["status"] = "stage1_complete"
advanced_state["stage1_best_epoch"] = 6
advanced_state["stage1_best_val_loss"] = 0.70379
advanced_state["stage1_best_val_accuracy"] = 0.7150
advanced_state["stage1_final_model_path"] = str(
    ADVANCED_STAGE1_FINAL_PATH
)
advanced_state["stage1_completed_at"] = (
    datetime.now().isoformat()
)

with open(ADVANCED_STATE_PATH, "w") as f:
    json.dump(
        advanced_state,
        f,
        indent=2
    )

print("Advanced Stage 1 saved.")
print("Final model:", ADVANCED_STAGE1_FINAL_PATH)
print("Best epoch:", 6)
print("Best val_accuracy:", 0.7150)
print("Best val_loss:", 0.70379)


In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

CLASS_NAMES = [
    "Normal",
    "No Lung Opacity / Not Normal",
    "Lung Opacity"
]

print("Generating Advanced Stage 1 validation predictions...")

val_prob_advanced_stage1 = advanced_model.predict(
    val_ds,
    verbose=1
)

val_pred_advanced_stage1 = np.argmax(
    val_prob_advanced_stage1,
    axis=1
)

val_true = (
    val_df["class_id"]
    .astype(int)
    .to_numpy()
)

val_true_onehot = np.eye(3)[val_true]

accuracy = accuracy_score(
    val_true,
    val_pred_advanced_stage1
)

balanced_acc = balanced_accuracy_score(
    val_true,
    val_pred_advanced_stage1
)

macro_f1 = f1_score(
    val_true,
    val_pred_advanced_stage1,
    average="macro"
)

weighted_f1 = f1_score(
    val_true,
    val_pred_advanced_stage1,
    average="weighted"
)

macro_roc_auc = roc_auc_score(
    val_true_onehot,
    val_prob_advanced_stage1,
    average="macro",
    multi_class="ovr"
)

weighted_roc_auc = roc_auc_score(
    val_true_onehot,
    val_prob_advanced_stage1,
    average="weighted",
    multi_class="ovr"
)

macro_pr_auc = average_precision_score(
    val_true_onehot,
    val_prob_advanced_stage1,
    average="macro"
)

weighted_pr_auc = average_precision_score(
    val_true_onehot,
    val_prob_advanced_stage1,
    average="weighted"
)

report = classification_report(
    val_true,
    val_pred_advanced_stage1,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    val_true,
    val_pred_advanced_stage1
)

metrics = {
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_roc_auc_ovr": float(macro_roc_auc),
    "weighted_roc_auc_ovr": float(weighted_roc_auc),
    "macro_pr_auc": float(macro_pr_auc),
    "weighted_pr_auc": float(weighted_pr_auc)
}

predictions_df = val_df.copy()

predictions_df["predicted_class_id"] = (
    val_pred_advanced_stage1
)

predictions_df["prob_normal"] = (
    val_prob_advanced_stage1[:, 0]
)

predictions_df[
    "prob_no_lung_opacity_not_normal"
] = val_prob_advanced_stage1[:, 1]

predictions_df["prob_lung_opacity"] = (
    val_prob_advanced_stage1[:, 2]
)

predictions_df.to_csv(
    ADVANCED_DIR
    / "validation_predictions_stage1.csv",
    index=False
)

with open(
    ADVANCED_DIR
    / "stage1_validation_metrics.json",
    "w"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

pd.DataFrame(report).T.to_csv(
    ADVANCED_DIR
    / "stage1_validation_classification_report.csv"
)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    ADVANCED_DIR
    / "stage1_validation_confusion_matrix.csv"
)

print("\nOverall validation metrics")
print("--------------------------------")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")
print(f"Macro ROC-AUC:     {macro_roc_auc:.4f}")
print(f"Weighted ROC-AUC:  {weighted_roc_auc:.4f}")
print(f"Macro PR-AUC:      {macro_pr_auc:.4f}")
print(f"Weighted PR-AUC:   {weighted_pr_auc:.4f}")

print("\nPer-class performance")
print("--------------------------------")

for class_name in CLASS_NAMES:
    r = report[class_name]

    print(
        f"{class_name}: "
        f"precision={r['precision']:.4f}, "
        f"recall={r['recall']:.4f}, "
        f"F1={r['f1-score']:.4f}"
    )

print("\nConfusion matrix")
print("--------------------------------")
print(cm)

print("\nPrediction counts")
print("--------------------------------")
print(
    "True:     ",
    np.bincount(
        val_true,
        minlength=3
    )
)
print(
    "Predicted:",
    np.bincount(
        val_pred_advanced_stage1,
        minlength=3
    )
)

print("\nAdvanced Stage 1 validation results saved.")


In [ ]:
import tensorflow as tf

backbone = advanced_model.get_layer(
    "mobilenetv2_transformer_backbone"
)

FINE_TUNE_FROM = 116
ADVANCED_STAGE2_LR = 1e-5

backbone.trainable = True

for i, layer in enumerate(backbone.layers):

    if i < FINE_TUNE_FROM:
        layer.trainable = False

    elif isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False

    else:
        layer.trainable = True


advanced_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=ADVANCED_STAGE2_LR
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in advanced_model.trainable_weights
)

nontrainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in advanced_model.non_trainable_weights
)

print("Fine-tuning starts from:")
print(
    FINE_TUNE_FROM,
    backbone.layers[FINE_TUNE_FROM].name
)

print("\nBackbone trainable:", backbone.trainable)

print(
    "Learning rate:",
    float(
        advanced_model.optimizer.learning_rate.numpy()
    )
)

print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)

print(
    "Non-trainable parameters:",
    f"{nontrainable_params:,}"
)

print("\nTrainable backbone layers with weights:")

for i, layer in enumerate(backbone.layers):
    if layer.trainable and layer.weights:
        print(
            i,
            layer.name,
            type(layer).__name__
        )


In [ ]:
import json
from datetime import datetime

ADVANCED_STAGE2_DIR = (
    ADVANCED_DIR
    / "stage2_finetune_block13"
)

ADVANCED_STAGE2_CHECKPOINTS = (
    ADVANCED_STAGE2_DIR
    / "checkpoints"
)

ADVANCED_STAGE2_LOGS = (
    ADVANCED_STAGE2_DIR
    / "logs"
)

ADVANCED_STAGE2_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ADVANCED_STAGE2_CHECKPOINTS.mkdir(
    parents=True,
    exist_ok=True
)

ADVANCED_STAGE2_LOGS.mkdir(
    parents=True,
    exist_ok=True
)

ADVANCED_STAGE2_INITIAL_PATH = (
    ADVANCED_STAGE2_DIR
    / "stage2_initial.keras"
)

ADVANCED_STAGE2_BEST_PATH = (
    ADVANCED_STAGE2_CHECKPOINTS
    / "stage2_best.keras"
)

ADVANCED_STAGE2_LATEST_PATH = (
    ADVANCED_STAGE2_CHECKPOINTS
    / "stage2_latest.keras"
)

ADVANCED_STAGE2_LOG_PATH = (
    ADVANCED_STAGE2_LOGS
    / "stage2_training_log.csv"
)

advanced_model.save(
    ADVANCED_STAGE2_INITIAL_PATH
)

advanced_state["status"] = "ready_for_stage2_training"

advanced_state["stage2_fine_tune_from_index"] = 116
advanced_state["stage2_fine_tune_from_layer"] = "block_13_expand"
advanced_state["stage2_learning_rate"] = 1e-5
advanced_state["stage2_max_epochs"] = 8

advanced_state["stage2_trainable_parameters"] = 3624003

advanced_state["stage2_initial_model_path"] = str(
    ADVANCED_STAGE2_INITIAL_PATH
)

advanced_state["stage2_best_path"] = str(
    ADVANCED_STAGE2_BEST_PATH
)

advanced_state["stage2_latest_path"] = str(
    ADVANCED_STAGE2_LATEST_PATH
)

advanced_state["stage2_log_path"] = str(
    ADVANCED_STAGE2_LOG_PATH
)

advanced_state["stage2_prepared_at"] = (
    datetime.now().isoformat()
)

with open(ADVANCED_STATE_PATH, "w") as f:
    json.dump(
        advanced_state,
        f,
        indent=2
    )

print("Advanced Stage 2 starting point saved.")
print("Initial model:", ADVANCED_STAGE2_INITIAL_PATH)
print("Status:", advanced_state["status"])
print("Maximum epochs:", advanced_state["stage2_max_epochs"])


In [ ]:
callbacks_advanced_stage2 = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ADVANCED_STAGE2_BEST_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ADVANCED_STAGE2_LATEST_PATH),
        save_best_only=False,
        save_weights_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(ADVANCED_STAGE2_LOG_PATH),
        append=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Advanced Stage 2 callbacks ready.")
print("Best checkpoint:", ADVANCED_STAGE2_BEST_PATH)
print("Latest checkpoint:", ADVANCED_STAGE2_LATEST_PATH)
print("Training log:", ADVANCED_STAGE2_LOG_PATH)
print("Maximum epochs:", 8)


In [ ]:
ADVANCED_STAGE2_EPOCHS = 8

history_advanced_stage2 = advanced_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=ADVANCED_STAGE2_EPOCHS,
    callbacks=callbacks_advanced_stage2,
    verbose=1
)


In [ ]:
import json
from datetime import datetime

ADVANCED_STAGE2_FINAL_PATH = (
    ADVANCED_STAGE2_DIR
    / "stage2_final.keras"
)

advanced_model.save(
    ADVANCED_STAGE2_FINAL_PATH
)

advanced_state["status"] = "stage2_complete"
advanced_state["stage2_best_epoch"] = 3
advanced_state["stage2_best_val_loss"] = 0.69667
advanced_state["stage2_best_val_accuracy"] = 0.7135
advanced_state["stage2_final_model_path"] = str(
    ADVANCED_STAGE2_FINAL_PATH
)
advanced_state["stage2_completed_at"] = (
    datetime.now().isoformat()
)

with open(ADVANCED_STATE_PATH, "w") as f:
    json.dump(
        advanced_state,
        f,
        indent=2
    )

print("Advanced Stage 2 finalized.")
print("Best epoch:", 3)
print("Best val_accuracy:", 0.7135)
print("Best val_loss:", 0.69667)
print("Final model:", ADVANCED_STAGE2_FINAL_PATH)


In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

CLASS_NAMES = [
    "Normal",
    "No Lung Opacity / Not Normal",
    "Lung Opacity"
]

print("Generating Advanced Stage 2 validation predictions...")

val_prob_advanced_stage2 = advanced_model.predict(
    val_ds,
    verbose=1
)

val_pred_advanced_stage2 = np.argmax(
    val_prob_advanced_stage2,
    axis=1
)

val_true = (
    val_df["class_id"]
    .astype(int)
    .to_numpy()
)

val_true_onehot = np.eye(3)[val_true]

accuracy = accuracy_score(
    val_true,
    val_pred_advanced_stage2
)

balanced_acc = balanced_accuracy_score(
    val_true,
    val_pred_advanced_stage2
)

macro_f1 = f1_score(
    val_true,
    val_pred_advanced_stage2,
    average="macro"
)

weighted_f1 = f1_score(
    val_true,
    val_pred_advanced_stage2,
    average="weighted"
)

macro_roc_auc = roc_auc_score(
    val_true_onehot,
    val_prob_advanced_stage2,
    average="macro",
    multi_class="ovr"
)

weighted_roc_auc = roc_auc_score(
    val_true_onehot,
    val_prob_advanced_stage2,
    average="weighted",
    multi_class="ovr"
)

macro_pr_auc = average_precision_score(
    val_true_onehot,
    val_prob_advanced_stage2,
    average="macro"
)

weighted_pr_auc = average_precision_score(
    val_true_onehot,
    val_prob_advanced_stage2,
    average="weighted"
)

report = classification_report(
    val_true,
    val_pred_advanced_stage2,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    val_true,
    val_pred_advanced_stage2
)

metrics = {
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_roc_auc_ovr": float(macro_roc_auc),
    "weighted_roc_auc_ovr": float(weighted_roc_auc),
    "macro_pr_auc": float(macro_pr_auc),
    "weighted_pr_auc": float(weighted_pr_auc)
}

predictions_df = val_df.copy()

predictions_df["predicted_class_id"] = (
    val_pred_advanced_stage2
)

predictions_df["prob_normal"] = (
    val_prob_advanced_stage2[:, 0]
)

predictions_df[
    "prob_no_lung_opacity_not_normal"
] = val_prob_advanced_stage2[:, 1]

predictions_df["prob_lung_opacity"] = (
    val_prob_advanced_stage2[:, 2]
)

predictions_df.to_csv(
    ADVANCED_STAGE2_DIR
    / "validation_predictions_stage2.csv",
    index=False
)

with open(
    ADVANCED_STAGE2_DIR
    / "stage2_validation_metrics.json",
    "w"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

pd.DataFrame(report).T.to_csv(
    ADVANCED_STAGE2_DIR
    / "stage2_validation_classification_report.csv"
)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    ADVANCED_STAGE2_DIR
    / "stage2_validation_confusion_matrix.csv"
)

print("\nOverall validation metrics")
print("--------------------------------")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")
print(f"Macro ROC-AUC:     {macro_roc_auc:.4f}")
print(f"Weighted ROC-AUC:  {weighted_roc_auc:.4f}")
print(f"Macro PR-AUC:      {macro_pr_auc:.4f}")
print(f"Weighted PR-AUC:   {weighted_pr_auc:.4f}")

print("\nPer-class performance")
print("--------------------------------")

for class_name in CLASS_NAMES:
    r = report[class_name]

    print(
        f"{class_name}: "
        f"precision={r['precision']:.4f}, "
        f"recall={r['recall']:.4f}, "
        f"F1={r['f1-score']:.4f}"
    )

print("\nConfusion matrix")
print("--------------------------------")
print(cm)

print("\nPrediction counts")
print("--------------------------------")
print(
    "True:     ",
    np.bincount(
        val_true,
        minlength=3
    )
)
print(
    "Predicted:",
    np.bincount(
        val_pred_advanced_stage2,
        minlength=3
    )
)

print("\nAdvanced Stage 2 validation results saved.")


In [ ]:
import pandas as pd
import json
from datetime import datetime

model_comparison = pd.DataFrame([
    {
        "model": "Baseline MobileNetV2 Fine-tuned",
        "accuracy": 0.7050,
        "balanced_accuracy": 0.6690,
        "macro_f1": 0.6767,
        "macro_roc_auc": 0.8587,
        "macro_pr_auc": 0.7353,
        "lung_opacity_recall": 0.4180,
        "lung_opacity_f1": 0.5240
    },
    {
        "model": "Multi-scale CBAM Stage 2",
        "accuracy": 0.7152,
        "balanced_accuracy": 0.6887,
        "macro_f1": 0.6942,
        "macro_roc_auc": 0.8644,
        "macro_pr_auc": 0.7446,
        "lung_opacity_recall": 0.4800,
        "lung_opacity_f1": 0.5634
    },
    {
        "model": "Advanced Transformer Stage 1",
        "accuracy": 0.7150,
        "balanced_accuracy": 0.6939,
        "macro_f1": 0.6984,
        "macro_roc_auc": 0.8686,
        "macro_pr_auc": 0.7557,
        "lung_opacity_recall": 0.5177,
        "lung_opacity_f1": 0.5837
    },
    {
        "model": "Advanced Transformer Stage 2",
        "accuracy": 0.7135,
        "balanced_accuracy": 0.6899,
        "macro_f1": 0.6952,
        "macro_roc_auc": 0.8693,
        "macro_pr_auc": 0.7563,
        "lung_opacity_recall": 0.4989,
        "lung_opacity_f1": 0.5725
    }
])

comparison_path = (
    PROJECT_ROOT
    / "results"
    / "validation_model_comparison.csv"
)

comparison_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

model_comparison.to_csv(
    comparison_path,
    index=False
)

advanced_state["model_selection_status"] = {
    "current_best_model":
        "Advanced Transformer Stage 1",

    "selection_basis": (
        "Best overall balance of macro F1, balanced accuracy, "
        "and Lung Opacity recall/F1 on validation."
    ),

    "current_best_model_path":
        str(ADVANCED_STAGE1_FINAL_PATH),

    "test_set_used": False,

    "class_weights_used": False,

    "updated_at":
        datetime.now().isoformat()
}

with open(ADVANCED_STATE_PATH, "w") as f:
    json.dump(
        advanced_state,
        f,
        indent=2
    )

print(model_comparison.to_string(index=False))

print("\nModel comparison saved:")
print(comparison_path)

print("\nCurrent best model:")
print(
    advanced_state[
        "model_selection_status"
    ]["current_best_model"]
)

print(
    "Test set used:",
    advanced_state[
        "model_selection_status"
    ]["test_set_used"]
)


### 8.1 Exploratory fine-tuning variants retained for auditability

In [ ]:
import tensorflow as tf
from pathlib import Path
import json
from datetime import datetime

REFINEMENT_DIR = (
    ADVANCED_DIR
    / "stage1_adamw_refinement_v1"
)

REFINEMENT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REFINEMENT_INITIAL_PATH = (
    REFINEMENT_DIR
    / "refinement_initial.keras"
)

# Load the exact best Advanced Stage 1 model
refinement_model = tf.keras.models.load_model(
    ADVANCED_STAGE1_FINAL_PATH
)

backbone = refinement_model.get_layer(
    "mobilenetv2_transformer_backbone"
)

# Stage 1 refinement:
# keep the entire MobileNetV2 backbone frozen
backbone.trainable = False

for layer in backbone.layers:
    layer.trainable = False

REFINEMENT_LR = 2.5e-5
WEIGHT_DECAY = 1e-4

refinement_model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=REFINEMENT_LR,
        weight_decay=WEIGHT_DECAY
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in refinement_model.trainable_weights
)

nontrainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in refinement_model.non_trainable_weights
)

refinement_model.save(
    REFINEMENT_INITIAL_PATH
)

print("AdamW refinement model prepared.")
print("Backbone trainable:", backbone.trainable)
print("Learning rate:", float(refinement_model.optimizer.learning_rate.numpy()))
print("Weight decay:", refinement_model.optimizer.weight_decay)
print("Trainable parameters:", f"{trainable_params:,}")
print("Non-trainable parameters:", f"{nontrainable_params:,}")
print("Initial model:", REFINEMENT_INITIAL_PATH)


In [ ]:
REFINEMENT_BEST_PATH = (
    REFINEMENT_DIR
    / "refinement_best.keras"
)

REFINEMENT_LATEST_PATH = (
    REFINEMENT_DIR
    / "refinement_latest.keras"
)

REFINEMENT_LOG_PATH = (
    REFINEMENT_DIR
    / "refinement_training_log.csv"
)

callbacks_refinement = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(REFINEMENT_BEST_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(REFINEMENT_LATEST_PATH),
        save_best_only=False,
        save_weights_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(REFINEMENT_LOG_PATH),
        append=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=2,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    )
]

print("AdamW refinement callbacks ready.")
print("Best checkpoint:", REFINEMENT_BEST_PATH)
print("Latest checkpoint:", REFINEMENT_LATEST_PATH)
print("Training log:", REFINEMENT_LOG_PATH)
print("Maximum epochs:", 5)


In [ ]:
REFINEMENT_EPOCHS = 5

history_refinement = refinement_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=REFINEMENT_EPOCHS,
    callbacks=callbacks_refinement,
    verbose=1
)


In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

REFINEMENT_FINAL_PATH = (
    REFINEMENT_DIR
    / "refinement_final.keras"
)

refinement_model.save(
    REFINEMENT_FINAL_PATH
)

print("Generating refinement validation predictions...")

refinement_prob = refinement_model.predict(
    val_ds,
    verbose=1
)

refinement_pred = np.argmax(
    refinement_prob,
    axis=1
)

val_true = (
    val_df["class_id"]
    .astype(int)
    .to_numpy()
)

val_true_onehot = np.eye(3)[val_true]

accuracy = accuracy_score(
    val_true,
    refinement_pred
)

balanced_acc = balanced_accuracy_score(
    val_true,
    refinement_pred
)

macro_f1 = f1_score(
    val_true,
    refinement_pred,
    average="macro"
)

weighted_f1 = f1_score(
    val_true,
    refinement_pred,
    average="weighted"
)

macro_roc_auc = roc_auc_score(
    val_true_onehot,
    refinement_prob,
    average="macro",
    multi_class="ovr"
)

weighted_roc_auc = roc_auc_score(
    val_true_onehot,
    refinement_prob,
    average="weighted",
    multi_class="ovr"
)

macro_pr_auc = average_precision_score(
    val_true_onehot,
    refinement_prob,
    average="macro"
)

weighted_pr_auc = average_precision_score(
    val_true_onehot,
    refinement_prob,
    average="weighted"
)

report = classification_report(
    val_true,
    refinement_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    val_true,
    refinement_pred
)

metrics = {
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_roc_auc_ovr": float(macro_roc_auc),
    "weighted_roc_auc_ovr": float(weighted_roc_auc),
    "macro_pr_auc": float(macro_pr_auc),
    "weighted_pr_auc": float(weighted_pr_auc)
}

with open(
    REFINEMENT_DIR
    / "refinement_validation_metrics.json",
    "w"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

pd.DataFrame(report).T.to_csv(
    REFINEMENT_DIR
    / "refinement_classification_report.csv"
)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    REFINEMENT_DIR
    / "refinement_confusion_matrix.csv"
)

print("\nOverall validation metrics")
print("--------------------------------")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")
print(f"Macro ROC-AUC:     {macro_roc_auc:.4f}")
print(f"Weighted ROC-AUC:  {weighted_roc_auc:.4f}")
print(f"Macro PR-AUC:      {macro_pr_auc:.4f}")
print(f"Weighted PR-AUC:   {weighted_pr_auc:.4f}")

print("\nPer-class performance")
print("--------------------------------")

for class_name in CLASS_NAMES:
    r = report[class_name]

    print(
        f"{class_name}: "
        f"precision={r['precision']:.4f}, "
        f"recall={r['recall']:.4f}, "
        f"F1={r['f1-score']:.4f}"
    )

print("\nConfusion matrix")
print("--------------------------------")
print(cm)

print("\nTrue counts:")
print(np.bincount(val_true, minlength=3))

print("Predicted counts:")
print(np.bincount(refinement_pred, minlength=3))

print("\nFinal refinement model:")
print(REFINEMENT_FINAL_PATH)


In [ ]:
import tensorflow as tf
from pathlib import Path

SHALLOW_DIR = (
    ADVANCED_DIR
    / "stage2_shallow_block16_v1"
)

SHALLOW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SHALLOW_INITIAL_PATH = (
    SHALLOW_DIR
    / "shallow_initial.keras"
)

# Start from the original best Stage 1 model
shallow_model = tf.keras.models.load_model(
    ADVANCED_STAGE1_FINAL_PATH
)

backbone = shallow_model.get_layer(
    "mobilenetv2_transformer_backbone"
)

SHALLOW_FINE_TUNE_FROM = 143
SHALLOW_LR = 2e-6

backbone.trainable = True

for i, layer in enumerate(backbone.layers):

    if i < SHALLOW_FINE_TUNE_FROM:
        layer.trainable = False

    elif isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False

    else:
        layer.trainable = True

shallow_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=SHALLOW_LR
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in shallow_model.trainable_weights
)

nontrainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in shallow_model.non_trainable_weights
)

shallow_model.save(
    SHALLOW_INITIAL_PATH
)

print("Shallow fine-tuning prepared.")
print(
    "Starts from:",
    SHALLOW_FINE_TUNE_FROM,
    backbone.layers[
        SHALLOW_FINE_TUNE_FROM
    ].name
)
print("Learning rate:", float(
    shallow_model.optimizer.learning_rate.numpy()
))
print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)
print(
    "Non-trainable parameters:",
    f"{nontrainable_params:,}"
)

print("\nTrainable backbone layers with weights:")

for i, layer in enumerate(backbone.layers):

    if (
        layer.trainable
        and len(layer.weights) > 0
    ):
        print(
            i,
            layer.name,
            layer.__class__.__name__
        )

print("\nInitial model:")
print(SHALLOW_INITIAL_PATH)


In [ ]:
import json
from datetime import datetime

SHALLOW_BEST_PATH = (
    SHALLOW_DIR
    / "shallow_best.keras"
)

SHALLOW_LATEST_PATH = (
    SHALLOW_DIR
    / "shallow_latest.keras"
)

SHALLOW_LOG_PATH = (
    SHALLOW_DIR
    / "shallow_training_log.csv"
)

SHALLOW_STATE_PATH = (
    SHALLOW_DIR
    / "shallow_experiment_state.json"
)

shallow_state = {
    "experiment": "stage2_shallow_block16_v1",
    "status": "ready_for_training",
    "source_model": str(ADVANCED_STAGE1_FINAL_PATH),
    "fine_tune_from_index": 143,
    "fine_tune_from_layer": "block_16_expand",
    "learning_rate": 2e-6,
    "optimizer": "Adam",
    "batch_norm_frozen": True,
    "trainable_parameters": 2839683,
    "non_trainable_parameters": 1378944,
    "max_epochs": 6,
    "test_set_used": False,
    "created_at": datetime.now().isoformat()
}

with open(SHALLOW_STATE_PATH, "w") as f:
    json.dump(
        shallow_state,
        f,
        indent=2
    )

callbacks_shallow = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(SHALLOW_BEST_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(SHALLOW_LATEST_PATH),
        save_best_only=False,
        save_weights_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(SHALLOW_LOG_PATH),
        append=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Shallow fine-tuning callbacks ready.")
print("Best checkpoint:", SHALLOW_BEST_PATH)
print("Latest checkpoint:", SHALLOW_LATEST_PATH)
print("Training log:", SHALLOW_LOG_PATH)
print("State:", SHALLOW_STATE_PATH)
print("Maximum epochs:", shallow_state["max_epochs"])


In [ ]:
SHALLOW_EPOCHS = 6

history_shallow = shallow_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=SHALLOW_EPOCHS,
    callbacks=callbacks_shallow,
    verbose=1
)


In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

SHALLOW_FINAL_PATH = (
    SHALLOW_DIR
    / "shallow_final.keras"
)

shallow_model.save(
    SHALLOW_FINAL_PATH
)

print("Generating shallow fine-tuning validation predictions...")

shallow_prob = shallow_model.predict(
    val_ds,
    verbose=1
)

shallow_pred = np.argmax(
    shallow_prob,
    axis=1
)

val_true = (
    val_df["class_id"]
    .astype(int)
    .to_numpy()
)

val_true_onehot = np.eye(3)[val_true]

accuracy = accuracy_score(
    val_true,
    shallow_pred
)

balanced_acc = balanced_accuracy_score(
    val_true,
    shallow_pred
)

macro_f1 = f1_score(
    val_true,
    shallow_pred,
    average="macro"
)

weighted_f1 = f1_score(
    val_true,
    shallow_pred,
    average="weighted"
)

macro_roc_auc = roc_auc_score(
    val_true_onehot,
    shallow_prob,
    average="macro",
    multi_class="ovr"
)

weighted_roc_auc = roc_auc_score(
    val_true_onehot,
    shallow_prob,
    average="weighted",
    multi_class="ovr"
)

macro_pr_auc = average_precision_score(
    val_true_onehot,
    shallow_prob,
    average="macro"
)

weighted_pr_auc = average_precision_score(
    val_true_onehot,
    shallow_prob,
    average="weighted"
)

report = classification_report(
    val_true,
    shallow_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    val_true,
    shallow_pred
)

metrics = {
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_roc_auc_ovr": float(macro_roc_auc),
    "weighted_roc_auc_ovr": float(weighted_roc_auc),
    "macro_pr_auc": float(macro_pr_auc),
    "weighted_pr_auc": float(weighted_pr_auc)
}

with open(
    SHALLOW_DIR
    / "shallow_validation_metrics.json",
    "w"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

pd.DataFrame(report).T.to_csv(
    SHALLOW_DIR
    / "shallow_classification_report.csv"
)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    SHALLOW_DIR
    / "shallow_confusion_matrix.csv"
)

print("\nOverall validation metrics")
print("--------------------------------")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")
print(f"Macro ROC-AUC:     {macro_roc_auc:.4f}")
print(f"Weighted ROC-AUC:  {weighted_roc_auc:.4f}")
print(f"Macro PR-AUC:      {macro_pr_auc:.4f}")
print(f"Weighted PR-AUC:   {weighted_pr_auc:.4f}")

print("\nPer-class performance")
print("--------------------------------")

for class_name in CLASS_NAMES:
    r = report[class_name]

    print(
        f"{class_name}: "
        f"precision={r['precision']:.4f}, "
        f"recall={r['recall']:.4f}, "
        f"F1={r['f1-score']:.4f}"
    )

print("\nConfusion matrix")
print("--------------------------------")
print(cm)

print("\nTrue counts:")
print(np.bincount(val_true, minlength=3))

print("Predicted counts:")
print(np.bincount(shallow_pred, minlength=3))

print("\nFinal shallow model:")
print(SHALLOW_FINAL_PATH)


## 9. Controlled ResNet50 benchmark

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

resnet_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(
            0.015,
            seed=SEED
        ),
        tf.keras.layers.RandomTranslation(
            height_factor=0.03,
            width_factor=0.03,
            seed=SEED
        ),
        tf.keras.layers.RandomZoom(
            height_factor=(-0.05, 0.05),
            width_factor=(-0.05, 0.05),
            seed=SEED
        ),
        tf.keras.layers.RandomContrast(
            0.10,
            seed=SEED
        )
    ],
    name="resnet50_train_augmentation"
)


def load_resnet_image(image_relpath, label, training=False):

    image_path = tf.strings.join(
        [
            str(ACTIVE_DATASET),
            "/",
            image_relpath
        ]
    )

    image_bytes = tf.io.read_file(image_path)

    image = tf.io.decode_png(
        image_bytes,
        channels=1
    )

    image = tf.image.resize(
        image,
        IMG_SIZE,
        method="bilinear"
    )

    image = tf.image.grayscale_to_rgb(image)

    image = tf.cast(
        image,
        tf.float32
    )

    if training:
        image = resnet_augmentation(
            image,
            training=True
        )

    image = tf.keras.applications.resnet50.preprocess_input(
        image
    )

    return image, label


def build_resnet_dataset(df, training=False):

    paths = df["image_relpath"].astype(str).to_numpy()

    labels = (
        df["class_id"]
        .astype("int32")
        .to_numpy()
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(df),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        lambda path, label: load_resnet_image(
            path,
            label,
            training=training
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


train_ds_resnet = build_resnet_dataset(
    train_df,
    training=True
)

val_ds_resnet = build_resnet_dataset(
    val_df,
    training=False
)

sample_images, sample_labels = next(
    iter(val_ds_resnet)
)

print("ResNet50 datasets ready.")
print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Batch shape:", sample_images.shape)
print("Labels shape:", sample_labels.shape)
print(
    "Pixel range:",
    float(tf.reduce_min(sample_images)),
    "to",
    float(tf.reduce_max(sample_images))
)
print(
    "First labels:",
    sample_labels[:10].numpy()
)


In [ ]:
import tensorflow as tf
from pathlib import Path
import json
from datetime import datetime

RESNET_DIR = (
    PROJECT_ROOT
    / "experiments"
    / "resnet50_controlled_benchmark_v1"
)

RESNET_STAGE1_DIR = (
    RESNET_DIR
    / "stage1_frozen"
)

RESNET_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESNET_STAGE1_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESNET_INITIAL_PATH = (
    RESNET_STAGE1_DIR
    / "resnet50_stage1_initial.keras"
)

RESNET_STATE_PATH = (
    RESNET_DIR
    / "experiment_state.json"
)

resnet_backbone = tf.keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

resnet_backbone.trainable = False

inputs = tf.keras.Input(
    shape=(224, 224, 3),
    name="image"
)

x = resnet_backbone(
    inputs,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = tf.keras.layers.Dropout(
    0.30,
    name="dropout"
)(x)

outputs = tf.keras.layers.Dense(
    3,
    activation="softmax",
    name="predictions"
)(x)

resnet_model = tf.keras.Model(
    inputs,
    outputs,
    name="resnet50_controlled_benchmark"
)

RESNET_STAGE1_LR = 1e-4

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=RESNET_STAGE1_LR
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in resnet_model.trainable_weights
)

nontrainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in resnet_model.non_trainable_weights
)

total_params = (
    trainable_params
    + nontrainable_params
)

resnet_model.save(
    RESNET_INITIAL_PATH
)

resnet_state = {
    "experiment": "resnet50_controlled_benchmark_v1",
    "architecture": "ResNet50",
    "weights": "ImageNet",
    "input_size": [224, 224, 3],
    "classes": 3,
    "stage1_backbone_frozen": True,
    "stage1_learning_rate": 1e-4,
    "optimizer": "Adam",
    "head": "GlobalAveragePooling2D -> Dropout(0.30) -> Dense(3, softmax)",
    "train_samples": len(train_df),
    "validation_samples": len(val_df),
    "test_set_used": False,
    "trainable_parameters": int(trainable_params),
    "non_trainable_parameters": int(nontrainable_params),
    "total_parameters": int(total_params),
    "status": "stage1_ready",
    "created_at": datetime.now().isoformat()
}

with open(
    RESNET_STATE_PATH,
    "w"
) as f:
    json.dump(
        resnet_state,
        f,
        indent=2
    )

print("ResNet50 controlled benchmark prepared.")
print("Backbone trainable:", resnet_backbone.trainable)
print("Learning rate:", float(
    resnet_model.optimizer.learning_rate.numpy()
))
print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)
print(
    "Non-trainable parameters:",
    f"{nontrainable_params:,}"
)
print(
    "Total parameters:",
    f"{total_params:,}"
)
print("Initial model:", RESNET_INITIAL_PATH)
print("Status:", resnet_state["status"])


In [ ]:
RESNET_STAGE1_BEST_PATH = (
    RESNET_STAGE1_DIR
    / "resnet50_stage1_best.keras"
)

RESNET_STAGE1_LATEST_PATH = (
    RESNET_STAGE1_DIR
    / "resnet50_stage1_latest.keras"
)

RESNET_STAGE1_LOG_PATH = (
    RESNET_STAGE1_DIR
    / "resnet50_stage1_training_log.csv"
)

callbacks_resnet_stage1 = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(RESNET_STAGE1_BEST_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(RESNET_STAGE1_LATEST_PATH),
        save_best_only=False,
        save_weights_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(RESNET_STAGE1_LOG_PATH),
        append=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("ResNet50 Stage 1 callbacks ready.")
print("Best checkpoint:", RESNET_STAGE1_BEST_PATH)
print("Latest checkpoint:", RESNET_STAGE1_LATEST_PATH)
print("Training log:", RESNET_STAGE1_LOG_PATH)
print("Maximum epochs:", 10)


In [ ]:
RESNET_STAGE1_EPOCHS = 10

history_resnet_stage1 = resnet_model.fit(
    train_ds_resnet,
    validation_data=val_ds_resnet,
    epochs=RESNET_STAGE1_EPOCHS,
    callbacks=callbacks_resnet_stage1,
    verbose=1
)


In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

RESNET_STAGE1_FINAL_PATH = (
    RESNET_STAGE1_DIR
    / "resnet50_stage1_final.keras"
)

resnet_model.save(
    RESNET_STAGE1_FINAL_PATH
)

print("Generating ResNet50 Stage 1 validation predictions...")

resnet_stage1_prob = resnet_model.predict(
    val_ds_resnet,
    verbose=1
)

resnet_stage1_pred = np.argmax(
    resnet_stage1_prob,
    axis=1
)

val_true = (
    val_df["class_id"]
    .astype(int)
    .to_numpy()
)

val_true_onehot = np.eye(3)[val_true]

accuracy = accuracy_score(
    val_true,
    resnet_stage1_pred
)

balanced_acc = balanced_accuracy_score(
    val_true,
    resnet_stage1_pred
)

macro_f1 = f1_score(
    val_true,
    resnet_stage1_pred,
    average="macro"
)

weighted_f1 = f1_score(
    val_true,
    resnet_stage1_pred,
    average="weighted"
)

macro_roc_auc = roc_auc_score(
    val_true_onehot,
    resnet_stage1_prob,
    average="macro",
    multi_class="ovr"
)

weighted_roc_auc = roc_auc_score(
    val_true_onehot,
    resnet_stage1_prob,
    average="weighted",
    multi_class="ovr"
)

macro_pr_auc = average_precision_score(
    val_true_onehot,
    resnet_stage1_prob,
    average="macro"
)

weighted_pr_auc = average_precision_score(
    val_true_onehot,
    resnet_stage1_prob,
    average="weighted"
)

report = classification_report(
    val_true,
    resnet_stage1_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    val_true,
    resnet_stage1_pred
)

metrics = {
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_roc_auc_ovr": float(macro_roc_auc),
    "weighted_roc_auc_ovr": float(weighted_roc_auc),
    "macro_pr_auc": float(macro_pr_auc),
    "weighted_pr_auc": float(weighted_pr_auc)
}

with open(
    RESNET_STAGE1_DIR
    / "resnet50_stage1_validation_metrics.json",
    "w"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

pd.DataFrame(report).T.to_csv(
    RESNET_STAGE1_DIR
    / "resnet50_stage1_classification_report.csv"
)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    RESNET_STAGE1_DIR
    / "resnet50_stage1_confusion_matrix.csv"
)

print("\nOverall validation metrics")
print("--------------------------------")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")
print(f"Macro ROC-AUC:     {macro_roc_auc:.4f}")
print(f"Weighted ROC-AUC:  {weighted_roc_auc:.4f}")
print(f"Macro PR-AUC:      {macro_pr_auc:.4f}")
print(f"Weighted PR-AUC:   {weighted_pr_auc:.4f}")

print("\nPer-class performance")
print("--------------------------------")

for class_name in CLASS_NAMES:
    r = report[class_name]

    print(
        f"{class_name}: "
        f"precision={r['precision']:.4f}, "
        f"recall={r['recall']:.4f}, "
        f"F1={r['f1-score']:.4f}"
    )

print("\nConfusion matrix")
print("--------------------------------")
print(cm)

print("\nTrue counts:")
print(np.bincount(val_true, minlength=3))

print("Predicted counts:")
print(np.bincount(resnet_stage1_pred, minlength=3))

print("\nFinal Stage 1 model:")
print(RESNET_STAGE1_FINAL_PATH)


In [ ]:
import tensorflow as tf

RESNET_STAGE2_DIR = (
    RESNET_DIR
    / "stage2_finetune_conv5"
)

RESNET_STAGE2_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESNET_STAGE2_INITIAL_PATH = (
    RESNET_STAGE2_DIR
    / "resnet50_stage2_initial.keras"
)

resnet_stage2_model = tf.keras.models.load_model(
    RESNET_STAGE1_FINAL_PATH
)

resnet_backbone_stage2 = resnet_stage2_model.layers[1]

FINE_TUNE_LAYER_NAME = "conv5_block1_1_conv"

fine_tune_from = None

for i, layer in enumerate(resnet_backbone_stage2.layers):
    if layer.name == FINE_TUNE_LAYER_NAME:
        fine_tune_from = i
        break

if fine_tune_from is None:
    raise ValueError(
        f"{FINE_TUNE_LAYER_NAME} not found."
    )

resnet_backbone_stage2.trainable = True

for i, layer in enumerate(resnet_backbone_stage2.layers):

    if i < fine_tune_from:
        layer.trainable = False

    elif isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False

    else:
        layer.trainable = True

RESNET_STAGE2_LR = 1e-5

resnet_stage2_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=RESNET_STAGE2_LR
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in resnet_stage2_model.trainable_weights
)

nontrainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in resnet_stage2_model.non_trainable_weights
)

resnet_stage2_model.save(
    RESNET_STAGE2_INITIAL_PATH
)

print("ResNet50 conv5_x fine-tuning prepared.")
print(
    "Fine-tuning starts from:",
    fine_tune_from,
    resnet_backbone_stage2.layers[fine_tune_from].name
)

print(
    "Learning rate:",
    float(
        resnet_stage2_model.optimizer.learning_rate.numpy()
    )
)

print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)

print(
    "Non-trainable parameters:",
    f"{nontrainable_params:,}"
)

print("\nTrainable backbone layers with weights:")

for i, layer in enumerate(resnet_backbone_stage2.layers):
    if layer.trainable and len(layer.weights) > 0:
        print(
            i,
            layer.name,
            layer.__class__.__name__
        )

print("\nInitial model:")
print(RESNET_STAGE2_INITIAL_PATH)


In [ ]:
import json
from datetime import datetime

RESNET_STAGE2_BEST_PATH = (
    RESNET_STAGE2_DIR
    / "resnet50_stage2_best.keras"
)

RESNET_STAGE2_LATEST_PATH = (
    RESNET_STAGE2_DIR
    / "resnet50_stage2_latest.keras"
)

RESNET_STAGE2_LOG_PATH = (
    RESNET_STAGE2_DIR
    / "resnet50_stage2_training_log.csv"
)

RESNET_STAGE2_STATE_PATH = (
    RESNET_STAGE2_DIR
    / "stage2_experiment_state.json"
)

resnet_stage2_state = {
    "experiment": "resnet50_controlled_benchmark_v1",
    "stage": "stage2_finetune_conv5",
    "status": "ready_for_training",
    "source_model": str(RESNET_STAGE1_FINAL_PATH),
    "fine_tune_from_index": 143,
    "fine_tune_from_layer": "conv5_block1_1_conv",
    "learning_rate": 1e-5,
    "optimizer": "Adam",
    "batch_norm_frozen": True,
    "trainable_parameters": 14959619,
    "non_trainable_parameters": 8634240,
    "max_epochs": 8,
    "test_set_used": False,
    "created_at": datetime.now().isoformat()
}

with open(
    RESNET_STAGE2_STATE_PATH,
    "w"
) as f:
    json.dump(
        resnet_stage2_state,
        f,
        indent=2
    )

callbacks_resnet_stage2 = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(RESNET_STAGE2_BEST_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(RESNET_STAGE2_LATEST_PATH),
        save_best_only=False,
        save_weights_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(RESNET_STAGE2_LOG_PATH),
        append=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("ResNet50 Stage 2 callbacks ready.")
print("Best checkpoint:", RESNET_STAGE2_BEST_PATH)
print("Latest checkpoint:", RESNET_STAGE2_LATEST_PATH)
print("Training log:", RESNET_STAGE2_LOG_PATH)
print("Maximum epochs:", resnet_stage2_state["max_epochs"])


In [ ]:
RESNET_STAGE2_EPOCHS = 8

history_resnet_stage2 = resnet_stage2_model.fit(
    train_ds_resnet,
    validation_data=val_ds_resnet,
    epochs=RESNET_STAGE2_EPOCHS,
    callbacks=callbacks_resnet_stage2,
    verbose=1
)


In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

RESNET_STAGE2_FINAL_PATH = (
    RESNET_STAGE2_DIR
    / "resnet50_stage2_final.keras"
)

resnet_stage2_model.save(
    RESNET_STAGE2_FINAL_PATH
)

print("Generating ResNet50 Stage 2 validation predictions...")

resnet_stage2_prob = resnet_stage2_model.predict(
    val_ds_resnet,
    verbose=1
)

resnet_stage2_pred = np.argmax(
    resnet_stage2_prob,
    axis=1
)

val_true = (
    val_df["class_id"]
    .astype(int)
    .to_numpy()
)

val_true_onehot = np.eye(3)[val_true]

accuracy = accuracy_score(
    val_true,
    resnet_stage2_pred
)

balanced_acc = balanced_accuracy_score(
    val_true,
    resnet_stage2_pred
)

macro_f1 = f1_score(
    val_true,
    resnet_stage2_pred,
    average="macro"
)

weighted_f1 = f1_score(
    val_true,
    resnet_stage2_pred,
    average="weighted"
)

macro_roc_auc = roc_auc_score(
    val_true_onehot,
    resnet_stage2_prob,
    average="macro",
    multi_class="ovr"
)

weighted_roc_auc = roc_auc_score(
    val_true_onehot,
    resnet_stage2_prob,
    average="weighted",
    multi_class="ovr"
)

macro_pr_auc = average_precision_score(
    val_true_onehot,
    resnet_stage2_prob,
    average="macro"
)

weighted_pr_auc = average_precision_score(
    val_true_onehot,
    resnet_stage2_prob,
    average="weighted"
)

report = classification_report(
    val_true,
    resnet_stage2_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    val_true,
    resnet_stage2_pred
)

metrics = {
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_roc_auc_ovr": float(macro_roc_auc),
    "weighted_roc_auc_ovr": float(weighted_roc_auc),
    "macro_pr_auc": float(macro_pr_auc),
    "weighted_pr_auc": float(weighted_pr_auc)
}

with open(
    RESNET_STAGE2_DIR
    / "resnet50_stage2_validation_metrics.json",
    "w"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

pd.DataFrame(report).T.to_csv(
    RESNET_STAGE2_DIR
    / "resnet50_stage2_classification_report.csv"
)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    RESNET_STAGE2_DIR
    / "resnet50_stage2_confusion_matrix.csv"
)

print("\nOverall validation metrics")
print("--------------------------------")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")
print(f"Macro ROC-AUC:     {macro_roc_auc:.4f}")
print(f"Weighted ROC-AUC:  {weighted_roc_auc:.4f}")
print(f"Macro PR-AUC:      {macro_pr_auc:.4f}")
print(f"Weighted PR-AUC:   {weighted_pr_auc:.4f}")

print("\nPer-class performance")
print("--------------------------------")

for class_name in CLASS_NAMES:
    r = report[class_name]

    print(
        f"{class_name}: "
        f"precision={r['precision']:.4f}, "
        f"recall={r['recall']:.4f}, "
        f"F1={r['f1-score']:.4f}"
    )

print("\nConfusion matrix")
print("--------------------------------")
print(cm)

print("\nTrue counts:")
print(np.bincount(val_true, minlength=3))

print("Predicted counts:")
print(np.bincount(resnet_stage2_pred, minlength=3))

print("\nFinal ResNet50 Stage 2 model:")
print(RESNET_STAGE2_FINAL_PATH)


In [ ]:
resnet_row = {
    "model": "ResNet50 Controlled Fine-tuned",
    "accuracy": 0.6927,
    "balanced_accuracy": 0.6495,
    "macro_f1": 0.6507,
    "macro_roc_auc": 0.8560,
    "macro_pr_auc": 0.7327,
    "lung_opacity_recall": 0.3337,
    "lung_opacity_f1": 0.4574
}

model_comparison = pd.concat(
    [
        model_comparison,
        pd.DataFrame([resnet_row])
    ],
    ignore_index=True
)

model_comparison.to_csv(
    PROJECT_ROOT
    / "results"
    / "validation_model_comparison.csv",
    index=False
)

print(
    model_comparison
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .to_string(index=False)
)


## 10. Validation comparison and model-selection lock

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import json

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

ENSEMBLE_DIR = (
    PROJECT_ROOT
    / "experiments"
    / "advanced_cbam_ensemble_50_50_v1"
)

ENSEMBLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CBAM_STAGE2_FINAL_PATH = (
    PROJECT_ROOT
    / "experiments"
    / "enhanced_mobilenetv2_multiscale_cbam_v1"
    / "stage2_finetune_block13"
    / "stage2_final.keras"
)

print("Loading Advanced Stage 1...")

ensemble_advanced_model = tf.keras.models.load_model(
    ADVANCED_STAGE1_FINAL_PATH
)

print("Loading CBAM Stage 2...")

ensemble_cbam_model = tf.keras.models.load_model(
    CBAM_STAGE2_FINAL_PATH
)

print("Generating Advanced predictions...")

advanced_prob = ensemble_advanced_model.predict(
    val_ds,
    verbose=1
)

print("Generating CBAM predictions...")

cbam_prob = ensemble_cbam_model.predict(
    val_ds,
    verbose=1
)

if advanced_prob.shape != cbam_prob.shape:
    raise ValueError(
        "Prediction shapes do not match."
    )

ensemble_prob = (
    0.5 * advanced_prob
    + 0.5 * cbam_prob
)

ensemble_pred = np.argmax(
    ensemble_prob,
    axis=1
)

val_true = (
    val_df["class_id"]
    .astype(int)
    .to_numpy()
)

val_true_onehot = np.eye(3)[val_true]

CLASS_NAMES = [
    "Normal",
    "No Lung Opacity / Not Normal",
    "Lung Opacity"
]

accuracy = accuracy_score(
    val_true,
    ensemble_pred
)

balanced_acc = balanced_accuracy_score(
    val_true,
    ensemble_pred
)

macro_f1 = f1_score(
    val_true,
    ensemble_pred,
    average="macro"
)

weighted_f1 = f1_score(
    val_true,
    ensemble_pred,
    average="weighted"
)

macro_roc_auc = roc_auc_score(
    val_true_onehot,
    ensemble_prob,
    average="macro",
    multi_class="ovr"
)

weighted_roc_auc = roc_auc_score(
    val_true_onehot,
    ensemble_prob,
    average="weighted",
    multi_class="ovr"
)

macro_pr_auc = average_precision_score(
    val_true_onehot,
    ensemble_prob,
    average="macro"
)

weighted_pr_auc = average_precision_score(
    val_true_onehot,
    ensemble_prob,
    average="weighted"
)

report = classification_report(
    val_true,
    ensemble_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    val_true,
    ensemble_pred
)

metrics = {
    "ensemble": "Advanced Stage 1 + CBAM Stage 2",
    "weights": [0.5, 0.5],
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_roc_auc_ovr": float(macro_roc_auc),
    "weighted_roc_auc_ovr": float(weighted_roc_auc),
    "macro_pr_auc": float(macro_pr_auc),
    "weighted_pr_auc": float(weighted_pr_auc),
    "test_set_used": False
}

with open(
    ENSEMBLE_DIR
    / "validation_metrics.json",
    "w"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

pd.DataFrame(report).T.to_csv(
    ENSEMBLE_DIR
    / "classification_report.csv"
)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    ENSEMBLE_DIR
    / "confusion_matrix.csv"
)

predictions_df = val_df.copy()

predictions_df["ensemble_predicted_class_id"] = (
    ensemble_pred
)

predictions_df["ensemble_prob_normal"] = (
    ensemble_prob[:, 0]
)

predictions_df[
    "ensemble_prob_no_lung_opacity_not_normal"
] = ensemble_prob[:, 1]

predictions_df["ensemble_prob_lung_opacity"] = (
    ensemble_prob[:, 2]
)

predictions_df.to_csv(
    ENSEMBLE_DIR
    / "validation_predictions.csv",
    index=False
)

print("\n50/50 Ensemble validation metrics")
print("--------------------------------")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")
print(f"Macro ROC-AUC:     {macro_roc_auc:.4f}")
print(f"Weighted ROC-AUC:  {weighted_roc_auc:.4f}")
print(f"Macro PR-AUC:      {macro_pr_auc:.4f}")
print(f"Weighted PR-AUC:   {weighted_pr_auc:.4f}")

print("\nPer-class performance")
print("--------------------------------")

for class_name in CLASS_NAMES:
    r = report[class_name]

    print(
        f"{class_name}: "
        f"precision={r['precision']:.4f}, "
        f"recall={r['recall']:.4f}, "
        f"F1={r['f1-score']:.4f}"
    )

print("\nConfusion matrix")
print("--------------------------------")
print(cm)

print("\nTrue counts:")
print(
    np.bincount(
        val_true,
        minlength=3
    )
)

print("Predicted counts:")
print(
    np.bincount(
        ensemble_pred,
        minlength=3
    )
)

print("\nResults saved to:")
print(ENSEMBLE_DIR)


In [ ]:
import json
import pandas as pd
from datetime import datetime

FINAL_SELECTION_PATH = (
    PROJECT_ROOT
    / "results"
    / "final_model_selection.json"
)

ensemble_row = {
    "model": "Advanced + CBAM 50/50 Ensemble",
    "accuracy": 0.7182,
    "balanced_accuracy": 0.6932,
    "macro_f1": 0.6984,
    "macro_roc_auc": 0.8730,
    "macro_pr_auc": 0.7612,
    "lung_opacity_recall": 0.4933,
    "lung_opacity_f1": 0.5716
}

model_comparison = model_comparison[
    model_comparison["model"]
    != "Advanced + CBAM 50/50 Ensemble"
].copy()

model_comparison = pd.concat(
    [
        model_comparison,
        pd.DataFrame([ensemble_row])
    ],
    ignore_index=True
)

comparison_path = (
    PROJECT_ROOT
    / "results"
    / "validation_model_comparison.csv"
)

model_comparison.to_csv(
    comparison_path,
    index=False
)

final_selection = {
    "selection_locked": True,

    "primary_model": {
        "name": "Advanced Transformer Stage 1",
        "path": str(ADVANCED_STAGE1_FINAL_PATH),
        "reason": (
            "Best overall validation balance based on "
            "balanced accuracy, macro F1, and Lung Opacity "
            "recall/F1."
        )
    },

    "secondary_model": {
        "name": "Advanced + CBAM 50/50 Ensemble",
        "advanced_model_path": str(
            ADVANCED_STAGE1_FINAL_PATH
        ),
        "cbam_model_path": str(
            CBAM_STAGE2_FINAL_PATH
        ),
        "weights": [0.5, 0.5],
        "reason": (
            "Pre-specified secondary benchmark with higher "
            "accuracy and ranking metrics but lower Lung "
            "Opacity sensitivity."
        )
    },

    "no_more_validation_tuning": True,
    "test_set_used": False,
    "locked_at": datetime.now().isoformat()
}

with open(
    FINAL_SELECTION_PATH,
    "w"
) as f:
    json.dump(
        final_selection,
        f,
        indent=2
    )

print("Model selection locked.")
print(
    "Primary model:",
    final_selection["primary_model"]["name"]
)
print(
    "Secondary benchmark:",
    final_selection["secondary_model"]["name"]
)
print(
    "No more validation tuning:",
    final_selection[
        "no_more_validation_tuning"
    ]
)
print(
    "Test set used:",
    final_selection["test_set_used"]
)
print(
    "Selection file:",
    FINAL_SELECTION_PATH
)

print("\nFinal validation ranking:")
print(
    model_comparison
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .to_string(index=False)
)


## 11. Locked internal test evaluation

In [ ]:
import pandas as pd
import numpy as np

TEST_SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "test.csv"
)

test_df = pd.read_csv(
    TEST_SPLIT_PATH
)

required_columns = {
    "image_relpath",
    "class_id"
}

missing_columns = (
    required_columns
    - set(test_df.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

if len(test_df) != 4003:
    raise ValueError(
        f"Unexpected test size: {len(test_df)}"
    )

test_ds = build_test_dataset(
    test_df
)

test_images, test_labels = next(
    iter(test_ds)
)

test_true = (
    test_df["class_id"]
    .astype(int)
    .to_numpy()
)

print("Locked internal test split restored.")
print("Test split file:", TEST_SPLIT_PATH)
print("Test samples:", len(test_df))
print("Batch shape:", test_images.shape)
print("Labels shape:", test_labels.shape)

print(
    "Pixel range:",
    float(test_images.numpy().min()),
    "to",
    float(test_images.numpy().max())
)

print(
    "Class counts:",
    np.bincount(
        test_true,
        minlength=3
    )
)

print(
    "First labels:",
    test_labels[:10].numpy()
)


In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import json

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

FINAL_TEST_DIR = (
    PROJECT_ROOT
    / "results"
    / "final_test"
)

FINAL_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PRIMARY_TEST_DIR = (
    FINAL_TEST_DIR
    / "advanced_stage1"
)

PRIMARY_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLASS_NAMES = [
    "Normal",
    "No Lung Opacity / Not Normal",
    "Lung Opacity"
]

print("Loading locked primary model...")

final_advanced_model = tf.keras.models.load_model(
    ADVANCED_STAGE1_FINAL_PATH
)

print("Generating final test predictions...")

test_prob_advanced = final_advanced_model.predict(
    test_ds,
    verbose=1
)

test_pred_advanced = np.argmax(
    test_prob_advanced,
    axis=1
)

test_true_onehot = np.eye(3)[test_true]

accuracy = accuracy_score(
    test_true,
    test_pred_advanced
)

balanced_acc = balanced_accuracy_score(
    test_true,
    test_pred_advanced
)

macro_f1 = f1_score(
    test_true,
    test_pred_advanced,
    average="macro"
)

weighted_f1 = f1_score(
    test_true,
    test_pred_advanced,
    average="weighted"
)

macro_roc_auc = roc_auc_score(
    test_true_onehot,
    test_prob_advanced,
    average="macro",
    multi_class="ovr"
)

weighted_roc_auc = roc_auc_score(
    test_true_onehot,
    test_prob_advanced,
    average="weighted",
    multi_class="ovr"
)

macro_pr_auc = average_precision_score(
    test_true_onehot,
    test_prob_advanced,
    average="macro"
)

weighted_pr_auc = average_precision_score(
    test_true_onehot,
    test_prob_advanced,
    average="weighted"
)

report = classification_report(
    test_true,
    test_pred_advanced,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    test_true,
    test_pred_advanced
)

metrics = {
    "model": "Advanced Transformer Stage 1",
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_roc_auc_ovr": float(macro_roc_auc),
    "weighted_roc_auc_ovr": float(weighted_roc_auc),
    "macro_pr_auc": float(macro_pr_auc),
    "weighted_pr_auc": float(weighted_pr_auc),
    "test_samples": int(len(test_true))
}

with open(
    PRIMARY_TEST_DIR
    / "test_metrics.json",
    "w"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

pd.DataFrame(report).T.to_csv(
    PRIMARY_TEST_DIR
    / "classification_report.csv"
)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    PRIMARY_TEST_DIR
    / "confusion_matrix.csv"
)

test_predictions_df = test_df.copy()

test_predictions_df["predicted_class_id"] = (
    test_pred_advanced
)

test_predictions_df["prob_normal"] = (
    test_prob_advanced[:, 0]
)

test_predictions_df[
    "prob_no_lung_opacity_not_normal"
] = test_prob_advanced[:, 1]

test_predictions_df["prob_lung_opacity"] = (
    test_prob_advanced[:, 2]
)

test_predictions_df.to_csv(
    PRIMARY_TEST_DIR
    / "test_predictions.csv",
    index=False
)

print("\nFinal Test Results: Advanced Stage 1")
print("--------------------------------")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")
print(f"Macro ROC-AUC:     {macro_roc_auc:.4f}")
print(f"Weighted ROC-AUC:  {weighted_roc_auc:.4f}")
print(f"Macro PR-AUC:      {macro_pr_auc:.4f}")
print(f"Weighted PR-AUC:   {weighted_pr_auc:.4f}")

print("\nPer-class performance")
print("--------------------------------")

for class_name in CLASS_NAMES:
    r = report[class_name]

    print(
        f"{class_name}: "
        f"precision={r['precision']:.4f}, "
        f"recall={r['recall']:.4f}, "
        f"F1={r['f1-score']:.4f}"
    )

print("\nConfusion matrix")
print("--------------------------------")
print(cm)

print("\nTrue counts:")
print(
    np.bincount(
        test_true,
        minlength=3
    )
)

print("Predicted counts:")
print(
    np.bincount(
        test_pred_advanced,
        minlength=3
    )
)

print("\nFinal test predictions saved to:")
print(PRIMARY_TEST_DIR)


In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import json

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

ENSEMBLE_TEST_DIR = (
    FINAL_TEST_DIR
    / "advanced_cbam_ensemble_50_50"
)

ENSEMBLE_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Loading locked CBAM Stage 2 model...")

final_cbam_model = tf.keras.models.load_model(
    CBAM_STAGE2_FINAL_PATH
)

print("Generating CBAM test predictions...")

test_prob_cbam = final_cbam_model.predict(
    test_ds,
    verbose=1
)

if test_prob_cbam.shape != test_prob_advanced.shape:
    raise ValueError(
        "Advanced and CBAM prediction shapes do not match."
    )

test_prob_ensemble = (
    0.5 * test_prob_advanced
    + 0.5 * test_prob_cbam
)

test_pred_ensemble = np.argmax(
    test_prob_ensemble,
    axis=1
)

test_true_onehot = np.eye(3)[test_true]

accuracy = accuracy_score(
    test_true,
    test_pred_ensemble
)

balanced_acc = balanced_accuracy_score(
    test_true,
    test_pred_ensemble
)

macro_f1 = f1_score(
    test_true,
    test_pred_ensemble,
    average="macro"
)

weighted_f1 = f1_score(
    test_true,
    test_pred_ensemble,
    average="weighted"
)

macro_roc_auc = roc_auc_score(
    test_true_onehot,
    test_prob_ensemble,
    average="macro",
    multi_class="ovr"
)

weighted_roc_auc = roc_auc_score(
    test_true_onehot,
    test_prob_ensemble,
    average="weighted",
    multi_class="ovr"
)

macro_pr_auc = average_precision_score(
    test_true_onehot,
    test_prob_ensemble,
    average="macro"
)

weighted_pr_auc = average_precision_score(
    test_true_onehot,
    test_prob_ensemble,
    average="weighted"
)

report = classification_report(
    test_true,
    test_pred_ensemble,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    test_true,
    test_pred_ensemble
)

metrics = {
    "model": "Advanced + CBAM 50/50 Ensemble",
    "weights": [0.5, 0.5],
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_roc_auc_ovr": float(macro_roc_auc),
    "weighted_roc_auc_ovr": float(weighted_roc_auc),
    "macro_pr_auc": float(macro_pr_auc),
    "weighted_pr_auc": float(weighted_pr_auc),
    "test_samples": int(len(test_true))
}

with open(
    ENSEMBLE_TEST_DIR
    / "test_metrics.json",
    "w"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

pd.DataFrame(report).T.to_csv(
    ENSEMBLE_TEST_DIR
    / "classification_report.csv"
)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    ENSEMBLE_TEST_DIR
    / "confusion_matrix.csv"
)

ensemble_predictions_df = test_df.copy()

ensemble_predictions_df[
    "predicted_class_id"
] = test_pred_ensemble

ensemble_predictions_df[
    "prob_normal"
] = test_prob_ensemble[:, 0]

ensemble_predictions_df[
    "prob_no_lung_opacity_not_normal"
] = test_prob_ensemble[:, 1]

ensemble_predictions_df[
    "prob_lung_opacity"
] = test_prob_ensemble[:, 2]

ensemble_predictions_df.to_csv(
    ENSEMBLE_TEST_DIR
    / "test_predictions.csv",
    index=False
)

print("\nFinal Test Results: 50/50 Ensemble")
print("--------------------------------")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")
print(f"Macro ROC-AUC:     {macro_roc_auc:.4f}")
print(f"Weighted ROC-AUC:  {weighted_roc_auc:.4f}")
print(f"Macro PR-AUC:      {macro_pr_auc:.4f}")
print(f"Weighted PR-AUC:   {weighted_pr_auc:.4f}")

print("\nPer-class performance")
print("--------------------------------")

for class_name in CLASS_NAMES:
    r = report[class_name]

    print(
        f"{class_name}: "
        f"precision={r['precision']:.4f}, "
        f"recall={r['recall']:.4f}, "
        f"F1={r['f1-score']:.4f}"
    )

print("\nConfusion matrix")
print("--------------------------------")
print(cm)

print("\nTrue counts:")
print(np.bincount(test_true, minlength=3))

print("Predicted counts:")
print(
    np.bincount(
        test_pred_ensemble,
        minlength=3
    )
)

print("\nResults saved to:")
print(ENSEMBLE_TEST_DIR)


In [ ]:
import numpy as np
import pandas as pd
import json

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

BOOTSTRAP_DIR = (
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "bootstrap"
)

BOOTSTRAP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PREDICTIONS_PATH = (
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "test_predictions.csv"
)

bootstrap_df = pd.read_csv(
    PREDICTIONS_PATH
)

y_true = (
    bootstrap_df["class_id"]
    .astype(int)
    .to_numpy()
)

y_pred = (
    bootstrap_df["predicted_class_id"]
    .astype(int)
    .to_numpy()
)

y_prob = bootstrap_df[
    [
        "prob_normal",
        "prob_no_lung_opacity_not_normal",
        "prob_lung_opacity"
    ]
].to_numpy()

y_true_onehot = np.eye(3)[y_true]

N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 42

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

class_indices = {
    c: np.where(y_true == c)[0]
    for c in range(3)
}

bootstrap_results = []

print(
    f"Running {N_BOOTSTRAP} stratified bootstrap resamples..."
)

for b in range(N_BOOTSTRAP):

    sampled_parts = []

    for c in range(3):

        idx = class_indices[c]

        sampled = rng.choice(
            idx,
            size=len(idx),
            replace=True
        )

        sampled_parts.append(
            sampled
        )

    boot_idx = np.concatenate(
        sampled_parts
    )

    rng.shuffle(
        boot_idx
    )

    yt = y_true[boot_idx]
    yp = y_pred[boot_idx]
    pr = y_prob[boot_idx]

    yt_onehot = np.eye(3)[yt]

    report_f1 = f1_score(
        yt,
        yp,
        labels=[0, 1, 2],
        average=None,
        zero_division=0
    )

    opacity_recall = (
        np.sum(
            (yt == 2)
            & (yp == 2)
        )
        /
        np.sum(
            yt == 2
        )
    )

    bootstrap_results.append(
        {
            "accuracy":
                accuracy_score(
                    yt,
                    yp
                ),

            "balanced_accuracy":
                balanced_accuracy_score(
                    yt,
                    yp
                ),

            "macro_f1":
                f1_score(
                    yt,
                    yp,
                    average="macro"
                ),

            "weighted_f1":
                f1_score(
                    yt,
                    yp,
                    average="weighted"
                ),

            "macro_roc_auc":
                roc_auc_score(
                    yt_onehot,
                    pr,
                    average="macro",
                    multi_class="ovr"
                ),

            "macro_pr_auc":
                average_precision_score(
                    yt_onehot,
                    pr,
                    average="macro"
                ),

            "lung_opacity_recall":
                opacity_recall,

            "lung_opacity_f1":
                report_f1[2]
        }
    )

bootstrap_results_df = pd.DataFrame(
    bootstrap_results
)

bootstrap_results_df.to_csv(
    BOOTSTRAP_DIR
    / "bootstrap_resamples.csv",
    index=False
)

point_estimates = {
    "accuracy":
        accuracy_score(
            y_true,
            y_pred
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_true,
            y_pred
        ),

    "macro_f1":
        f1_score(
            y_true,
            y_pred,
            average="macro"
        ),

    "weighted_f1":
        f1_score(
            y_true,
            y_pred,
            average="weighted"
        ),

    "macro_roc_auc":
        roc_auc_score(
            y_true_onehot,
            y_prob,
            average="macro",
            multi_class="ovr"
        ),

    "macro_pr_auc":
        average_precision_score(
            y_true_onehot,
            y_prob,
            average="macro"
        ),

    "lung_opacity_recall":
        np.mean(
            y_pred[
                y_true == 2
            ] == 2
        ),

    "lung_opacity_f1":
        f1_score(
            y_true,
            y_pred,
            labels=[2],
            average="macro"
        )
}

summary = {}

for metric in bootstrap_results_df.columns:

    lower = np.percentile(
        bootstrap_results_df[metric],
        2.5
    )

    upper = np.percentile(
        bootstrap_results_df[metric],
        97.5
    )

    summary[metric] = {
        "estimate":
            float(
                point_estimates[metric]
            ),

        "ci95_lower":
            float(lower),

        "ci95_upper":
            float(upper)
    }

with open(
    BOOTSTRAP_DIR
    / "bootstrap_95ci_summary.json",
    "w"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print("\nAdvanced Stage 1 — Final Test 95% Bootstrap CIs")
print("------------------------------------------------")

for metric, values in summary.items():

    print(
        f"{metric:22s}: "
        f"{values['estimate']:.4f} "
        f"[{values['ci95_lower']:.4f}, "
        f"{values['ci95_upper']:.4f}]"
    )

print("\nBootstrap results saved to:")
print(BOOTSTRAP_DIR)


## 12. Calibration and AP/PA subgroup analysis

In [ ]:
import numpy as np
import pandas as pd
import json

from sklearn.metrics import log_loss

CALIBRATION_DIR = (
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "calibration"
)

CALIBRATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

calibration_df = pd.read_csv(
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "test_predictions.csv"
)

y_true = (
    calibration_df["class_id"]
    .astype(int)
    .to_numpy()
)

y_prob = calibration_df[
    [
        "prob_normal",
        "prob_no_lung_opacity_not_normal",
        "prob_lung_opacity"
    ]
].to_numpy()

y_pred = np.argmax(
    y_prob,
    axis=1
)

confidence = np.max(
    y_prob,
    axis=1
)

correct = (
    y_pred == y_true
).astype(int)

# Negative Log-Likelihood
nll = log_loss(
    y_true,
    y_prob,
    labels=[0, 1, 2]
)

# Multiclass Brier score
y_true_onehot = np.eye(3)[y_true]

brier_score = np.mean(
    np.sum(
        (y_prob - y_true_onehot) ** 2,
        axis=1
    )
)

# Expected Calibration Error
N_BINS = 15

bin_edges = np.linspace(
    0.0,
    1.0,
    N_BINS + 1
)

ece = 0.0
reliability_rows = []

for i in range(N_BINS):

    lower = bin_edges[i]
    upper = bin_edges[i + 1]

    if i == N_BINS - 1:
        mask = (
            (confidence >= lower)
            & (confidence <= upper)
        )
    else:
        mask = (
            (confidence >= lower)
            & (confidence < upper)
        )

    n_bin = np.sum(mask)

    if n_bin == 0:
        continue

    bin_accuracy = np.mean(
        correct[mask]
    )

    bin_confidence = np.mean(
        confidence[mask]
    )

    bin_fraction = (
        n_bin / len(y_true)
    )

    bin_gap = abs(
        bin_accuracy
        - bin_confidence
    )

    ece += (
        bin_fraction
        * bin_gap
    )

    reliability_rows.append(
        {
            "bin_lower": lower,
            "bin_upper": upper,
            "n": int(n_bin),
            "fraction": float(bin_fraction),
            "mean_confidence": float(bin_confidence),
            "accuracy": float(bin_accuracy),
            "calibration_gap": float(
                bin_accuracy
                - bin_confidence
            )
        }
    )

reliability_df = pd.DataFrame(
    reliability_rows
)

mean_confidence = np.mean(
    confidence
)

overall_accuracy = np.mean(
    correct
)

calibration_summary = {
    "n_test_samples": int(len(y_true)),
    "accuracy": float(overall_accuracy),
    "mean_confidence": float(mean_confidence),
    "confidence_minus_accuracy": float(
        mean_confidence
        - overall_accuracy
    ),
    "negative_log_likelihood": float(nll),
    "multiclass_brier_score": float(
        brier_score
    ),
    "ece_15_bins": float(ece)
}

reliability_df.to_csv(
    CALIBRATION_DIR
    / "reliability_bins.csv",
    index=False
)

with open(
    CALIBRATION_DIR
    / "calibration_summary.json",
    "w"
) as f:
    json.dump(
        calibration_summary,
        f,
        indent=2
    )

print("Advanced Stage 1 — Test Calibration")
print("------------------------------------")

print(
    f"Accuracy:                 "
    f"{overall_accuracy:.4f}"
)

print(
    f"Mean confidence:          "
    f"{mean_confidence:.4f}"
)

print(
    f"Confidence - accuracy:    "
    f"{mean_confidence - overall_accuracy:+.4f}"
)

print(
    f"Negative Log-Likelihood:  "
    f"{nll:.4f}"
)

print(
    f"Multiclass Brier score:   "
    f"{brier_score:.4f}"
)

print(
    f"ECE (15 bins):            "
    f"{ece:.4f}"
)

print("\nReliability bins")
print("------------------------------------")

print(
    reliability_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nCalibration results saved to:")
print(CALIBRATION_DIR)


In [ ]:
import numpy as np
from sklearn.metrics import log_loss

prob_sums = y_prob.sum(axis=1)

print("Probability-sum diagnostics")
print("--------------------------------")
print("Minimum sum:", prob_sums.min())
print("Maximum sum:", prob_sums.max())
print("Mean sum:", prob_sums.mean())
print(
    "Maximum |sum - 1|:",
    np.max(np.abs(prob_sums - 1.0))
)

# Numerical normalization only
y_prob_normalized = (
    y_prob
    / prob_sums[:, None]
)

normalized_sums = y_prob_normalized.sum(axis=1)

print("\nAfter normalization")
print("--------------------------------")
print("Minimum sum:", normalized_sums.min())
print("Maximum sum:", normalized_sums.max())

# Recompute NLL
nll_normalized = log_loss(
    y_true,
    y_prob_normalized,
    labels=[0, 1, 2]
)

# Recompute Brier
y_true_onehot = np.eye(3)[y_true]

brier_normalized = np.mean(
    np.sum(
        (
            y_prob_normalized
            - y_true_onehot
        ) ** 2,
        axis=1
    )
)

# Recompute ECE
confidence_norm = np.max(
    y_prob_normalized,
    axis=1
)

pred_norm = np.argmax(
    y_prob_normalized,
    axis=1
)

correct_norm = (
    pred_norm == y_true
).astype(int)

N_BINS = 15

bin_edges = np.linspace(
    0.0,
    1.0,
    N_BINS + 1
)

ece_normalized = 0.0

for i in range(N_BINS):

    lower = bin_edges[i]
    upper = bin_edges[i + 1]

    if i == N_BINS - 1:
        mask = (
            (confidence_norm >= lower)
            & (confidence_norm <= upper)
        )
    else:
        mask = (
            (confidence_norm >= lower)
            & (confidence_norm < upper)
        )

    if np.sum(mask) == 0:
        continue

    bin_accuracy = np.mean(
        correct_norm[mask]
    )

    bin_confidence = np.mean(
        confidence_norm[mask]
    )

    ece_normalized += (
        np.mean(mask)
        * abs(
            bin_accuracy
            - bin_confidence
        )
    )

print("\nRecomputed calibration metrics")
print("--------------------------------")
print(
    f"NLL:    {nll_normalized:.6f}"
)
print(
    f"Brier:  {brier_normalized:.6f}"
)
print(
    f"ECE:    {ece_normalized:.6f}"
)

print(
    "\nPredictions changed:",
    np.sum(pred_norm != y_pred)
)


In [ ]:
import numpy as np
import pandas as pd
import json

from scipy.optimize import minimize_scalar
from sklearn.metrics import log_loss

TEMPERATURE_DIR = (
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "calibration"
    / "temperature_scaling"
)

TEMPERATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Generating validation probabilities for temperature fitting...")

val_prob_temp = final_advanced_model.predict(
    val_ds,
    verbose=1
)

val_true_temp = (
    val_df["class_id"]
    .astype(int)
    .to_numpy()
)

# Normalize for numerical safety
val_prob_temp = (
    val_prob_temp
    / val_prob_temp.sum(
        axis=1,
        keepdims=True
    )
)

test_prob_temp = (
    y_prob_normalized.copy()
)

EPS = 1e-12


def apply_temperature(probabilities, temperature):

    log_prob = np.log(
        np.clip(
            probabilities,
            EPS,
            1.0
        )
    )

    scaled_logits = (
        log_prob
        / temperature
    )

    scaled_logits = (
        scaled_logits
        - np.max(
            scaled_logits,
            axis=1,
            keepdims=True
        )
    )

    exp_logits = np.exp(
        scaled_logits
    )

    return (
        exp_logits
        / exp_logits.sum(
            axis=1,
            keepdims=True
        )
    )


def validation_nll(temperature):

    calibrated_prob = apply_temperature(
        val_prob_temp,
        temperature
    )

    return log_loss(
        val_true_temp,
        calibrated_prob,
        labels=[0, 1, 2]
    )


optimization = minimize_scalar(
    validation_nll,
    bounds=(0.25, 4.0),
    method="bounded",
    options={
        "xatol": 1e-5
    }
)

best_temperature = float(
    optimization.x
)

print("\nTemperature fitted on VALIDATION only")
print("--------------------------------------")
print(
    f"Optimal T: {best_temperature:.6f}"
)
print(
    f"Validation NLL before: "
    f"{validation_nll(1.0):.6f}"
)
print(
    f"Validation NLL after:  "
    f"{validation_nll(best_temperature):.6f}"
)

# Apply locked temperature to test
test_prob_calibrated = apply_temperature(
    test_prob_temp,
    best_temperature
)

test_pred_before = np.argmax(
    test_prob_temp,
    axis=1
)

test_pred_after = np.argmax(
    test_prob_calibrated,
    axis=1
)

test_onehot = np.eye(3)[y_true]


def compute_calibration_metrics(
    y_true,
    probabilities,
    n_bins=15
):

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    confidence = np.max(
        probabilities,
        axis=1
    )

    correct = (
        predictions == y_true
    ).astype(int)

    nll = log_loss(
        y_true,
        probabilities,
        labels=[0, 1, 2]
    )

    onehot = np.eye(3)[y_true]

    brier = np.mean(
        np.sum(
            (
                probabilities
                - onehot
            ) ** 2,
            axis=1
        )
    )

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    ece = 0.0

    for i in range(n_bins):

        lower = edges[i]
        upper = edges[i + 1]

        if i == n_bins - 1:
            mask = (
                (confidence >= lower)
                & (confidence <= upper)
            )
        else:
            mask = (
                (confidence >= lower)
                & (confidence < upper)
            )

        if np.sum(mask) == 0:
            continue

        bin_acc = np.mean(
            correct[mask]
        )

        bin_conf = np.mean(
            confidence[mask]
        )

        ece += (
            np.mean(mask)
            * abs(
                bin_acc
                - bin_conf
            )
        )

    return {
        "nll": float(nll),
        "brier": float(brier),
        "ece": float(ece),
        "mean_confidence": float(
            np.mean(confidence)
        )
    }


before = compute_calibration_metrics(
    y_true,
    test_prob_temp
)

after = compute_calibration_metrics(
    y_true,
    test_prob_calibrated
)

temperature_summary = {
    "temperature": best_temperature,
    "temperature_fitted_on": "validation_only",
    "validation_nll_before": float(
        validation_nll(1.0)
    ),
    "validation_nll_after": float(
        validation_nll(
            best_temperature
        )
    ),
    "test_before": before,
    "test_after": after,
    "test_predictions_changed": int(
        np.sum(
            test_pred_before
            != test_pred_after
        )
    )
}

with open(
    TEMPERATURE_DIR
    / "temperature_scaling_summary.json",
    "w"
) as f:
    json.dump(
        temperature_summary,
        f,
        indent=2
    )

print("\nLocked TEST calibration")
print("--------------------------------------")
print(
    f"NLL:   {before['nll']:.6f}"
    f" -> {after['nll']:.6f}"
)
print(
    f"Brier: {before['brier']:.6f}"
    f" -> {after['brier']:.6f}"
)
print(
    f"ECE:   {before['ece']:.6f}"
    f" -> {after['ece']:.6f}"
)
print(
    f"Mean confidence: "
    f"{before['mean_confidence']:.6f}"
    f" -> {after['mean_confidence']:.6f}"
)

print(
    "Predictions changed:",
    temperature_summary[
        "test_predictions_changed"
    ]
)

print("\nResults saved to:")
print(TEMPERATURE_DIR)


In [ ]:
import numpy as np
import pandas as pd
import json

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

SUBGROUP_DIR = (
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "subgroup_analysis"
    / "position"
)

SUBGROUP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

position_df = pd.read_csv(
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "test_predictions.csv"
)

if "position" not in position_df.columns:
    raise ValueError(
        "Column 'position' was not found in test_predictions.csv."
    )

position_df["position"] = (
    position_df["position"]
    .astype(str)
    .str.upper()
    .str.strip()
)

prob_cols = [
    "prob_normal",
    "prob_no_lung_opacity_not_normal",
    "prob_lung_opacity"
]

CLASS_NAMES = [
    "Normal",
    "No Lung Opacity / Not Normal",
    "Lung Opacity"
]


def compute_ece(y_true, probabilities, n_bins=15):

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    confidence = np.max(
        probabilities,
        axis=1
    )

    correct = (
        predictions == y_true
    ).astype(int)

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    ece = 0.0

    for i in range(n_bins):

        lower = edges[i]
        upper = edges[i + 1]

        if i == n_bins - 1:
            mask = (
                (confidence >= lower)
                & (confidence <= upper)
            )
        else:
            mask = (
                (confidence >= lower)
                & (confidence < upper)
            )

        if np.sum(mask) == 0:
            continue

        bin_acc = np.mean(
            correct[mask]
        )

        bin_conf = np.mean(
            confidence[mask]
        )

        ece += (
            np.mean(mask)
            * abs(
                bin_acc
                - bin_conf
            )
        )

    return float(ece)


summary_rows = []

print("AP/PA test distribution")
print("--------------------------------")

distribution = pd.crosstab(
    position_df["position"],
    position_df["class_id"]
)

distribution.columns = CLASS_NAMES

print(distribution)

print("\nAP/PA performance")
print("--------------------------------")

for position in ["AP", "PA"]:

    group = position_df[
        position_df["position"] == position
    ].copy()

    if len(group) == 0:
        print(f"\nNo samples found for {position}")
        continue

    y_true_group = (
        group["class_id"]
        .astype(int)
        .to_numpy()
    )

    y_pred_group = (
        group["predicted_class_id"]
        .astype(int)
        .to_numpy()
    )

    y_prob_group = (
        group[prob_cols]
        .to_numpy()
    )

    y_prob_group = (
        y_prob_group
        / y_prob_group.sum(
            axis=1,
            keepdims=True
        )
    )

    y_true_onehot = np.eye(3)[
        y_true_group
    ]

    accuracy = accuracy_score(
        y_true_group,
        y_pred_group
    )

    balanced_acc = balanced_accuracy_score(
        y_true_group,
        y_pred_group
    )

    macro_f1 = f1_score(
        y_true_group,
        y_pred_group,
        average="macro"
    )

    weighted_f1 = f1_score(
        y_true_group,
        y_pred_group,
        average="weighted"
    )

    macro_roc_auc = roc_auc_score(
        y_true_onehot,
        y_prob_group,
        average="macro",
        multi_class="ovr"
    )

    macro_pr_auc = average_precision_score(
        y_true_onehot,
        y_prob_group,
        average="macro"
    )

    mean_confidence = float(
        np.mean(
            np.max(
                y_prob_group,
                axis=1
            )
        )
    )

    ece = compute_ece(
        y_true_group,
        y_prob_group,
        n_bins=15
    )

    report = classification_report(
        y_true_group,
        y_pred_group,
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true_group,
        y_pred_group,
        labels=[0, 1, 2]
    )

    summary_rows.append(
        {
            "position": position,
            "n": len(group),
            "accuracy": accuracy,
            "balanced_accuracy": balanced_acc,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
            "macro_roc_auc": macro_roc_auc,
            "macro_pr_auc": macro_pr_auc,
            "mean_confidence": mean_confidence,
            "ece_15_bins": ece,
            "normal_recall":
                report["Normal"]["recall"],
            "no_opacity_not_normal_recall":
                report[
                    "No Lung Opacity / Not Normal"
                ]["recall"],
            "lung_opacity_recall":
                report[
                    "Lung Opacity"
                ]["recall"],
            "lung_opacity_precision":
                report[
                    "Lung Opacity"
                ]["precision"],
            "lung_opacity_f1":
                report[
                    "Lung Opacity"
                ]["f1-score"]
        }
    )

    pd.DataFrame(report).T.to_csv(
        SUBGROUP_DIR
        / f"{position.lower()}_classification_report.csv"
    )

    pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    ).to_csv(
        SUBGROUP_DIR
        / f"{position.lower()}_confusion_matrix.csv"
    )

    print(f"\n{position}")
    print("--------------------------------")
    print(f"Samples:            {len(group)}")
    print(f"Accuracy:           {accuracy:.4f}")
    print(f"Balanced accuracy:  {balanced_acc:.4f}")
    print(f"Macro F1:           {macro_f1:.4f}")
    print(f"Macro ROC-AUC:      {macro_roc_auc:.4f}")
    print(f"Macro PR-AUC:       {macro_pr_auc:.4f}")
    print(f"Mean confidence:    {mean_confidence:.4f}")
    print(f"ECE:                {ece:.4f}")

    print(
        "Opacity precision: "
        f"{report['Lung Opacity']['precision']:.4f}"
    )

    print(
        "Opacity recall:    "
        f"{report['Lung Opacity']['recall']:.4f}"
    )

    print(
        "Opacity F1:        "
        f"{report['Lung Opacity']['f1-score']:.4f}"
    )

    print("Confusion matrix:")
    print(cm)


position_summary = pd.DataFrame(
    summary_rows
)

position_summary.to_csv(
    SUBGROUP_DIR
    / "ap_pa_summary.csv",
    index=False
)

distribution.to_csv(
    SUBGROUP_DIR
    / "ap_pa_class_distribution.csv"
)

print("\nAP vs PA summary")
print("--------------------------------")

print(
    position_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nResults saved to:")
print(SUBGROUP_DIR)


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

POSITION_BASELINE_DIR = (
    FINAL_TEST_DIR
    / "position_only_baseline"
)

POSITION_BASELINE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

train_position = train_df.copy()
test_position = test_df.copy()

train_position["position"] = (
    train_position["position"]
    .astype(str)
    .str.upper()
    .str.strip()
)

test_position["position"] = (
    test_position["position"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Estimate P(class | position) from TRAIN only
position_class_probs = (
    train_position
    .groupby("position")["class_id"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(columns=[0, 1, 2], fill_value=0)
)

print("P(class | position) estimated from TRAIN only")
print("-----------------------------------------------")
print(position_class_probs)

y_true_pos = (
    test_position["class_id"]
    .astype(int)
    .to_numpy()
)

position_prob = np.vstack([
    position_class_probs.loc[pos].to_numpy()
    for pos in test_position["position"]
])

position_pred = np.argmax(
    position_prob,
    axis=1
)

y_true_onehot = np.eye(3)[
    y_true_pos
]

accuracy = accuracy_score(
    y_true_pos,
    position_pred
)

balanced_acc = balanced_accuracy_score(
    y_true_pos,
    position_pred
)

macro_f1 = f1_score(
    y_true_pos,
    position_pred,
    average="macro"
)

macro_roc_auc = roc_auc_score(
    y_true_onehot,
    position_prob,
    average="macro",
    multi_class="ovr"
)

macro_pr_auc = average_precision_score(
    y_true_onehot,
    position_prob,
    average="macro"
)

cm = confusion_matrix(
    y_true_pos,
    position_pred,
    labels=[0, 1, 2]
)

results = pd.DataFrame({
    "true_class_id": y_true_pos,
    "position": test_position["position"].to_numpy(),
    "predicted_class_id": position_pred,
    "prob_normal": position_prob[:, 0],
    "prob_no_opacity_not_normal": position_prob[:, 1],
    "prob_lung_opacity": position_prob[:, 2]
})

results.to_csv(
    POSITION_BASELINE_DIR
    / "position_only_test_predictions.csv",
    index=False
)

print("\nPosition-only TEST performance")
print("-----------------------------------------------")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Macro ROC-AUC:     {macro_roc_auc:.4f}")
print(f"Macro PR-AUC:      {macro_pr_auc:.4f}")

print("\nConfusion matrix")
print("-----------------------------------------------")
print(cm)

print("\nPredicted counts")
print(
    np.bincount(
        position_pred,
        minlength=3
    )
)

print("\nResults saved to:")
print(POSITION_BASELINE_DIR)


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

POSITION_PROB_DIR = (
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "subgroup_analysis"
    / "position_probability_analysis"
)

POSITION_PROB_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.read_csv(
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "test_predictions.csv"
)

df["position"] = (
    df["position"]
    .astype(str)
    .str.upper()
    .str.strip()
)

prob_cols = [
    "prob_normal",
    "prob_no_lung_opacity_not_normal",
    "prob_lung_opacity"
]

class_names = {
    0: "Normal",
    1: "No Lung Opacity / Not Normal",
    2: "Lung Opacity"
}

rows = []

print("Class-conditional probabilities by AP/PA")
print("------------------------------------------------")

for class_id, class_name in class_names.items():

    class_df = df[
        df["class_id"] == class_id
    ]

    print(f"\nTrue class: {class_name}")

    for position in ["AP", "PA"]:

        group = class_df[
            class_df["position"] == position
        ]

        if len(group) == 0:
            continue

        mean_probs = (
            group[prob_cols]
            .mean()
        )

        median_opacity_prob = (
            group["prob_lung_opacity"]
            .median()
        )

        predicted_opacity_rate = np.mean(
            group["predicted_class_id"] == 2
        )

        correct_rate = np.mean(
            group["predicted_class_id"] == class_id
        )

        rows.append(
            {
                "true_class": class_name,
                "true_class_id": class_id,
                "position": position,
                "n": len(group),
                "mean_prob_normal":
                    mean_probs["prob_normal"],
                "mean_prob_no_opacity_not_normal":
                    mean_probs[
                        "prob_no_lung_opacity_not_normal"
                    ],
                "mean_prob_lung_opacity":
                    mean_probs[
                        "prob_lung_opacity"
                    ],
                "median_prob_lung_opacity":
                    median_opacity_prob,
                "predicted_opacity_rate":
                    predicted_opacity_rate,
                "correct_rate":
                    correct_rate
            }
        )

        print(
            f"{position}: "
            f"n={len(group)}, "
            f"mean P(opacity)="
            f"{mean_probs['prob_lung_opacity']:.4f}, "
            f"median P(opacity)="
            f"{median_opacity_prob:.4f}, "
            f"predicted opacity rate="
            f"{predicted_opacity_rate:.4f}, "
            f"correct rate="
            f"{correct_rate:.4f}"
        )


summary_df = pd.DataFrame(
    rows
)

summary_df.to_csv(
    POSITION_PROB_DIR
    / "class_conditional_probability_summary.csv",
    index=False
)


# -------------------------------------------------
# Focus specifically on TRUE Lung Opacity cases
# -------------------------------------------------

opacity_true = df[
    df["class_id"] == 2
].copy()

ap_opacity_prob = (
    opacity_true.loc[
        opacity_true["position"] == "AP",
        "prob_lung_opacity"
    ]
    .to_numpy()
)

pa_opacity_prob = (
    opacity_true.loc[
        opacity_true["position"] == "PA",
        "prob_lung_opacity"
    ]
    .to_numpy()
)

observed_difference = (
    np.mean(ap_opacity_prob)
    - np.mean(pa_opacity_prob)
)

# Bootstrap CI for AP - PA difference
rng = np.random.default_rng(42)

N_BOOTSTRAP = 5000

bootstrap_differences = np.empty(
    N_BOOTSTRAP
)

for i in range(N_BOOTSTRAP):

    ap_sample = rng.choice(
        ap_opacity_prob,
        size=len(ap_opacity_prob),
        replace=True
    )

    pa_sample = rng.choice(
        pa_opacity_prob,
        size=len(pa_opacity_prob),
        replace=True
    )

    bootstrap_differences[i] = (
        np.mean(ap_sample)
        - np.mean(pa_sample)
    )

ci_lower = np.percentile(
    bootstrap_differences,
    2.5
)

ci_upper = np.percentile(
    bootstrap_differences,
    97.5
)

print("\nTrue Lung Opacity: AP vs PA")
print("------------------------------------------------")

print(
    f"AP n:                    "
    f"{len(ap_opacity_prob)}"
)

print(
    f"PA n:                    "
    f"{len(pa_opacity_prob)}"
)

print(
    f"Mean P(opacity), AP:     "
    f"{np.mean(ap_opacity_prob):.4f}"
)

print(
    f"Mean P(opacity), PA:     "
    f"{np.mean(pa_opacity_prob):.4f}"
)

print(
    f"AP - PA difference:      "
    f"{observed_difference:+.4f}"
)

print(
    f"95% bootstrap CI:        "
    f"[{ci_lower:+.4f}, {ci_upper:+.4f}]"
)


# -------------------------------------------------
# Opacity-vs-rest discrimination within position
# -------------------------------------------------

position_auc_rows = []

print("\nOpacity-vs-rest discrimination")
print("------------------------------------------------")

for position in ["AP", "PA"]:

    group = df[
        df["position"] == position
    ]

    y_binary = (
        group["class_id"].to_numpy()
        == 2
    ).astype(int)

    opacity_prob = (
        group["prob_lung_opacity"]
        .to_numpy()
    )

    auc = roc_auc_score(
        y_binary,
        opacity_prob
    )

    ap_score = average_precision_score(
        y_binary,
        opacity_prob
    )

    position_auc_rows.append(
        {
            "position": position,
            "n": len(group),
            "opacity_prevalence":
                np.mean(y_binary),
            "opacity_roc_auc": auc,
            "opacity_pr_auc": ap_score
        }
    )

    print(
        f"{position}: "
        f"ROC-AUC={auc:.4f}, "
        f"PR-AUC={ap_score:.4f}, "
        f"prevalence={np.mean(y_binary):.4f}"
    )

pd.DataFrame(
    position_auc_rows
).to_csv(
    POSITION_PROB_DIR
    / "opacity_discrimination_by_position.csv",
    index=False
)

pd.DataFrame(
    {
        "bootstrap_AP_minus_PA_mean_opacity_probability":
            bootstrap_differences
    }
).to_csv(
    POSITION_PROB_DIR
    / "opacity_probability_difference_bootstrap.csv",
    index=False
)

print("\nResults saved to:")
print(POSITION_PROB_DIR)


## 13. Radiologist-box area and adjusted AP/PA analysis

In [ ]:
import pandas as pd

BBOX_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bbox_annotations.csv"
)

bbox_df = pd.read_csv(
    BBOX_PATH
)

print("bbox_annotations shape:")
print(bbox_df.shape)

print("\nColumns:")
print(bbox_df.columns.tolist())

print("\nFirst rows:")
display(
    bbox_df.head()
)

print("\nMissing values:")
print(
    bbox_df.isna().sum()
)

print("\nUnique patients with boxes:")
print(
    bbox_df["patientId"].nunique()
    if "patientId" in bbox_df.columns
    else "patientId column not found"
)


In [ ]:
import numpy as np
import pandas as pd

LESION_BURDEN_DIR = (
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "subgroup_analysis"
    / "position_lesion_burden"
)

LESION_BURDEN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

IMAGE_WIDTH = 1024
IMAGE_HEIGHT = 1024
IMAGE_AREA = IMAGE_WIDTH * IMAGE_HEIGHT


def rectangle_union_area(group):
    """
    Exact union area of axis-aligned bounding boxes.
    Handles overlap without double counting.
    """

    rectangles = []

    for _, row in group.iterrows():

        x1 = max(
            0.0,
            float(row["x"])
        )

        y1 = max(
            0.0,
            float(row["y"])
        )

        x2 = min(
            float(IMAGE_WIDTH),
            x1 + float(row["width"])
        )

        y2 = min(
            float(IMAGE_HEIGHT),
            y1 + float(row["height"])
        )

        if x2 > x1 and y2 > y1:
            rectangles.append(
                (x1, y1, x2, y2)
            )

    if not rectangles:
        return 0.0

    x_edges = sorted(
        set(
            [r[0] for r in rectangles]
            + [r[2] for r in rectangles]
        )
    )

    total_area = 0.0

    for i in range(
        len(x_edges) - 1
    ):

        xa = x_edges[i]
        xb = x_edges[i + 1]

        if xb <= xa:
            continue

        active_y_intervals = []

        for x1, y1, x2, y2 in rectangles:

            if (
                x1 < xb
                and x2 > xa
            ):
                active_y_intervals.append(
                    (y1, y2)
                )

        if not active_y_intervals:
            continue

        active_y_intervals.sort()

        union_y = 0.0

        current_start, current_end = (
            active_y_intervals[0]
        )

        for start, end in (
            active_y_intervals[1:]
        ):

            if start <= current_end:

                current_end = max(
                    current_end,
                    end
                )

            else:

                union_y += (
                    current_end
                    - current_start
                )

                current_start = start
                current_end = end

        union_y += (
            current_end
            - current_start
        )

        total_area += (
            (xb - xa)
            * union_y
        )

    return total_area


# ------------------------------------------------
# Patient-level lesion burden from radiologist boxes
# ------------------------------------------------

bbox_work = bbox_df.copy()

bbox_work["bbox_area"] = (
    bbox_work["width"]
    * bbox_work["height"]
)

patient_rows = []

for patient_id, group in bbox_work.groupby(
    "patientId"
):

    summed_area = float(
        group["bbox_area"].sum()
    )

    union_area = float(
        rectangle_union_area(
            group
        )
    )

    patient_rows.append(
        {
            "patientId":
                patient_id,

            "bbox_count_from_annotations":
                int(len(group)),

            "summed_bbox_area":
                summed_area,

            "union_bbox_area":
                union_area,

            "summed_bbox_area_fraction":
                summed_area
                / IMAGE_AREA,

            "union_bbox_area_fraction":
                union_area
                / IMAGE_AREA
        }
    )

lesion_burden = pd.DataFrame(
    patient_rows
)

# ------------------------------------------------
# Merge with locked TEST predictions
# ------------------------------------------------

prediction_df = pd.read_csv(
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "test_predictions.csv"
)

prediction_df["position"] = (
    prediction_df["position"]
    .astype(str)
    .str.upper()
    .str.strip()
)

opacity_test = prediction_df[
    prediction_df["class_id"] == 2
].copy()

opacity_test = opacity_test.merge(
    lesion_burden,
    on="patientId",
    how="left",
    validate="one_to_one"
)

print("Integrity checks")
print("--------------------------------")

print(
    "Test Lung Opacity cases:",
    len(opacity_test)
)

print(
    "Missing lesion burden:",
    opacity_test[
        "union_bbox_area_fraction"
    ].isna().sum()
)

print(
    "bbox_count agreement:",
    np.mean(
        opacity_test["bbox_count"]
        ==
        opacity_test[
            "bbox_count_from_annotations"
        ]
    )
)

# ------------------------------------------------
# AP vs PA descriptive comparison
# ------------------------------------------------

summary_rows = []

print("\nLesion burden by position")
print("--------------------------------")

for position in ["AP", "PA"]:

    group = opacity_test[
        opacity_test["position"]
        == position
    ]

    burden = group[
        "union_bbox_area_fraction"
    ].to_numpy()

    summary_rows.append(
        {
            "position": position,
            "n": len(group),

            "mean_union_fraction":
                np.mean(burden),

            "median_union_fraction":
                np.median(burden),

            "q25_union_fraction":
                np.percentile(
                    burden,
                    25
                ),

            "q75_union_fraction":
                np.percentile(
                    burden,
                    75
                ),

            "mean_bbox_count":
                group["bbox_count"].mean(),

            "mean_model_opacity_probability":
                group[
                    "prob_lung_opacity"
                ].mean()
        }
    )

    print(f"\n{position}")
    print(
        f"n:                       "
        f"{len(group)}"
    )

    print(
        f"Mean union fraction:      "
        f"{np.mean(burden):.4f}"
    )

    print(
        f"Median union fraction:    "
        f"{np.median(burden):.4f}"
    )

    print(
        f"IQR:                      "
        f"[{np.percentile(burden, 25):.4f}, "
        f"{np.percentile(burden, 75):.4f}]"
    )

    print(
        f"Mean bbox count:           "
        f"{group['bbox_count'].mean():.3f}"
    )

    print(
        f"Mean P(opacity):           "
        f"{group['prob_lung_opacity'].mean():.4f}"
    )


# ------------------------------------------------
# Bootstrap AP - PA difference in lesion burden
# ------------------------------------------------

ap_burden = opacity_test.loc[
    opacity_test["position"] == "AP",
    "union_bbox_area_fraction"
].to_numpy()

pa_burden = opacity_test.loc[
    opacity_test["position"] == "PA",
    "union_bbox_area_fraction"
].to_numpy()

observed_mean_difference = (
    np.mean(ap_burden)
    - np.mean(pa_burden)
)

rng = np.random.default_rng(42)

N_BOOTSTRAP = 5000

bootstrap_differences = np.empty(
    N_BOOTSTRAP
)

for i in range(N_BOOTSTRAP):

    ap_sample = rng.choice(
        ap_burden,
        size=len(ap_burden),
        replace=True
    )

    pa_sample = rng.choice(
        pa_burden,
        size=len(pa_burden),
        replace=True
    )

    bootstrap_differences[i] = (
        np.mean(ap_sample)
        - np.mean(pa_sample)
    )

ci_lower = np.percentile(
    bootstrap_differences,
    2.5
)

ci_upper = np.percentile(
    bootstrap_differences,
    97.5
)

print("\nAP - PA lesion burden difference")
print("--------------------------------")

print(
    f"Observed mean difference: "
    f"{observed_mean_difference:+.4f}"
)

print(
    f"95% bootstrap CI:         "
    f"[{ci_lower:+.4f}, "
    f"{ci_upper:+.4f}]"
)

print(
    "\nInterpretation scale: "
    "fraction of total 1024x1024 image area"
)

# ------------------------------------------------
# Save
# ------------------------------------------------

lesion_summary = pd.DataFrame(
    summary_rows
)

lesion_summary.to_csv(
    LESION_BURDEN_DIR
    / "ap_pa_lesion_burden_summary.csv",
    index=False
)

opacity_test.to_csv(
    LESION_BURDEN_DIR
    / "test_opacity_cases_with_lesion_burden.csv",
    index=False
)

pd.DataFrame(
    {
        "bootstrap_AP_minus_PA_union_fraction":
            bootstrap_differences
    }
).to_csv(
    LESION_BURDEN_DIR
    / "lesion_burden_difference_bootstrap.csv",
    index=False
)

print("\nResults saved to:")
print(LESION_BURDEN_DIR)


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import json

ADJUSTED_POSITION_DIR = (
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "subgroup_analysis"
    / "position_adjusted_analysis"
)

ADJUSTED_POSITION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

analysis_df = pd.read_csv(
    LESION_BURDEN_DIR
    / "test_opacity_cases_with_lesion_burden.csv"
)

analysis_df["position"] = (
    analysis_df["position"]
    .astype(str)
    .str.upper()
    .str.strip()
)

analysis_df["PA"] = (
    analysis_df["position"] == "PA"
).astype(int)

# Standardize lesion burden for easier interpretation
burden_mean = (
    analysis_df["union_bbox_area_fraction"].mean()
)

burden_sd = (
    analysis_df["union_bbox_area_fraction"].std()
)

analysis_df["lesion_burden_z"] = (
    analysis_df["union_bbox_area_fraction"]
    - burden_mean
) / burden_sd

# Predictor matrix
X = analysis_df[
    [
        "PA",
        "lesion_burden_z",
        "bbox_count"
    ]
].copy()

X = sm.add_constant(
    X
)

y = analysis_df[
    "prob_lung_opacity"
].astype(float)

# OLS with heteroskedasticity-robust HC3 standard errors
model = sm.OLS(
    y,
    X
).fit(
    cov_type="HC3"
)

print("Adjusted model")
print("---------------------------------------------")
print(
    "Outcome: model-predicted P(Lung Opacity)"
)
print(
    "Reference position: AP"
)
print(
    "Predictors: PA + lesion burden + bbox count"
)

print("\n")
print(model.summary())

# Extract interpretable PA effect
pa_beta = float(
    model.params["PA"]
)

pa_se = float(
    model.bse["PA"]
)

pa_p = float(
    model.pvalues["PA"]
)

pa_ci_low = float(
    model.conf_int().loc["PA", 0]
)

pa_ci_high = float(
    model.conf_int().loc["PA", 1]
)

burden_beta = float(
    model.params["lesion_burden_z"]
)

burden_p = float(
    model.pvalues["lesion_burden_z"]
)

bbox_beta = float(
    model.params["bbox_count"]
)

bbox_p = float(
    model.pvalues["bbox_count"]
)

print("\nKey adjusted effects")
print("---------------------------------------------")

print(
    f"PA coefficient:              "
    f"{pa_beta:+.4f}"
)

print(
    f"95% CI for PA:               "
    f"[{pa_ci_low:+.4f}, "
    f"{pa_ci_high:+.4f}]"
)

print(
    f"PA p-value:                  "
    f"{pa_p:.6g}"
)

print(
    f"Lesion burden coefficient:   "
    f"{burden_beta:+.4f}"
)

print(
    f"Lesion burden p-value:       "
    f"{burden_p:.6g}"
)

print(
    f"bbox count coefficient:      "
    f"{bbox_beta:+.4f}"
)

print(
    f"bbox count p-value:          "
    f"{bbox_p:.6g}"
)

print(
    f"R-squared:                   "
    f"{model.rsquared:.4f}"
)

# Predicted opacity probability for a typical case,
# varying position only
typical_bbox_count = float(
    analysis_df["bbox_count"].median()
)

X_ap = pd.DataFrame(
    {
        "const": [1.0],
        "PA": [0],
        "lesion_burden_z": [0.0],
        "bbox_count": [typical_bbox_count]
    }
)

X_pa = pd.DataFrame(
    {
        "const": [1.0],
        "PA": [1],
        "lesion_burden_z": [0.0],
        "bbox_count": [typical_bbox_count]
    }
)

adjusted_ap = float(
    model.predict(X_ap)[0]
)

adjusted_pa = float(
    model.predict(X_pa)[0]
)

print("\nAdjusted comparison at mean lesion burden")
print("---------------------------------------------")

print(
    f"Adjusted AP P(opacity):      "
    f"{adjusted_ap:.4f}"
)

print(
    f"Adjusted PA P(opacity):      "
    f"{adjusted_pa:.4f}"
)

print(
    f"Adjusted PA - AP difference: "
    f"{adjusted_pa - adjusted_ap:+.4f}"
)

summary = {
    "n": int(len(analysis_df)),
    "outcome": "prob_lung_opacity",
    "reference_position": "AP",
    "pa_coefficient": pa_beta,
    "pa_standard_error_hc3": pa_se,
    "pa_ci95_lower": pa_ci_low,
    "pa_ci95_upper": pa_ci_high,
    "pa_p_value": pa_p,
    "lesion_burden_z_coefficient": burden_beta,
    "lesion_burden_p_value": burden_p,
    "bbox_count_coefficient": bbox_beta,
    "bbox_count_p_value": bbox_p,
    "r_squared": float(model.rsquared),
    "adjusted_ap_probability": adjusted_ap,
    "adjusted_pa_probability": adjusted_pa,
    "adjusted_pa_minus_ap": (
        adjusted_pa - adjusted_ap
    )
}

with open(
    ADJUSTED_POSITION_DIR
    / "adjusted_position_model.json",
    "w"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

with open(
    ADJUSTED_POSITION_DIR
    / "adjusted_position_model_summary.txt",
    "w"
) as f:
    f.write(
        model.summary().as_text()
    )

print("\nResults saved to:")
print(ADJUSTED_POSITION_DIR)


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from patsy import bs
import json

ROBUSTNESS_DIR = (
    ADJUSTED_POSITION_DIR
    / "robustness"
)

ROBUSTNESS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

robust_df = analysis_df.copy()

robust_df["age_clean"] = pd.to_numeric(
    robust_df["age_clean"],
    errors="coerce"
)

robust_df["sex"] = (
    robust_df["sex"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Keep complete cases for adjusted robustness model
robust_complete = robust_df.dropna(
    subset=[
        "prob_lung_opacity",
        "PA",
        "union_bbox_area_fraction",
        "bbox_count",
        "age_clean",
        "sex"
    ]
).copy()

print("Robustness analysis sample")
print("--------------------------------------------")
print("Original opacity cases:", len(robust_df))
print("Complete cases:", len(robust_complete))

print("\nPosition distribution:")
print(
    robust_complete["position"]
    .value_counts()
)

print("\nSex distribution:")
print(
    pd.crosstab(
        robust_complete["position"],
        robust_complete["sex"]
    )
)

print("\nMean age by position:")
print(
    robust_complete
    .groupby("position")["age_clean"]
    .agg(["count", "mean", "std", "median"])
)

# Flexible nonlinear adjustment for lesion burden
formula = (
    "prob_lung_opacity ~ "
    "PA + "
    "bs(union_bbox_area_fraction, df=4, degree=3) + "
    "bbox_count + "
    "age_clean + "
    "C(sex)"
)

robust_model = smf.ols(
    formula=formula,
    data=robust_complete
).fit(
    cov_type="HC3"
)

print("\nRobust adjusted model")
print("--------------------------------------------")
print(robust_model.summary())

pa_beta = float(
    robust_model.params["PA"]
)

pa_se = float(
    robust_model.bse["PA"]
)

pa_p = float(
    robust_model.pvalues["PA"]
)

pa_ci = (
    robust_model
    .conf_int()
    .loc["PA"]
)

print("\nKey robustness result")
print("--------------------------------------------")

print(
    f"PA coefficient:        {pa_beta:+.4f}"
)

print(
    f"95% CI:                "
    f"[{pa_ci.iloc[0]:+.4f}, "
    f"{pa_ci.iloc[1]:+.4f}]"
)

print(
    f"PA p-value:            {pa_p:.6g}"
)

print(
    f"R-squared:             "
    f"{robust_model.rsquared:.4f}"
)

print(
    "\nOriginal adjusted PA coefficient: "
    "-0.2088"
)

print(
    "Change from original coefficient: "
    f"{pa_beta - (-0.2088):+.4f}"
)

robustness_summary = {
    "n_complete": int(
        len(robust_complete)
    ),
    "model": (
        "OLS HC3 with nonlinear spline adjustment "
        "for lesion burden plus bbox count, age, and sex"
    ),
    "pa_coefficient": pa_beta,
    "pa_standard_error_hc3": pa_se,
    "pa_ci95_lower": float(
        pa_ci.iloc[0]
    ),
    "pa_ci95_upper": float(
        pa_ci.iloc[1]
    ),
    "pa_p_value": pa_p,
    "r_squared": float(
        robust_model.rsquared
    ),
    "original_pa_coefficient": -0.2088
}

with open(
    ROBUSTNESS_DIR
    / "position_effect_robustness.json",
    "w"
) as f:
    json.dump(
        robustness_summary,
        f,
        indent=2
    )

with open(
    ROBUSTNESS_DIR
    / "position_effect_robustness_summary.txt",
    "w"
) as f:
    f.write(
        robust_model.summary().as_text()
    )

print("\nResults saved to:")
print(ROBUSTNESS_DIR)


## 14. Validation-locked XAI protocol

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

XAI_LAYER_SELECTION_DIR = (
    PROJECT_ROOT
    / "results"
    / "xai_protocol"
    / "validation_gradcam_layer_comparison"
)

XAI_LAYER_SELECTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CANDIDATE_LAYERS = [
    "multiscale_fusion",
    "transformer_feature_map",
    "res3_activation"
]

TARGET_CLASS = 2
XAI_SEED = 42
MAX_PER_POSITION = 100

classifier_layer = final_advanced_model.get_layer(
    "class_predictions"
)

classifier_kernel = classifier_layer.kernel
classifier_bias = classifier_layer.bias

head_layer = final_advanced_model.get_layer(
    "dropout_256"
)

grad_models = {}

for layer_name in CANDIDATE_LAYERS:

    grad_models[layer_name] = tf.keras.Model(
        inputs=final_advanced_model.inputs,
        outputs=[
            final_advanced_model
            .get_layer(layer_name)
            .output,

            head_layer.output
        ]
    )


def make_bbox_union_mask(
    patient_id,
    size=224
):

    boxes = bbox_df[
        bbox_df["patientId"] == patient_id
    ]

    mask = np.zeros(
        (size, size),
        dtype=bool
    )

    scale_x = size / 1024.0
    scale_y = size / 1024.0

    for _, box in boxes.iterrows():

        x1 = int(
            np.floor(
                box["x"]
                * scale_x
            )
        )

        y1 = int(
            np.floor(
                box["y"]
                * scale_y
            )
        )

        x2 = int(
            np.ceil(
                (box["x"] + box["width"])
                * scale_x
            )
        )

        y2 = int(
            np.ceil(
                (box["y"] + box["height"])
                * scale_y
            )
        )

        x1 = np.clip(
            x1,
            0,
            size
        )

        x2 = np.clip(
            x2,
            0,
            size
        )

        y1 = np.clip(
            y1,
            0,
            size
        )

        y2 = np.clip(
            y2,
            0,
            size
        )

        mask[
            y1:y2,
            x1:x2
        ] = True

    return mask


def compute_gradcam(
    image_batch,
    layer_name
):

    grad_model = grad_models[
        layer_name
    ]

    with tf.GradientTape() as tape:

        feature_maps, head_features = (
            grad_model(
                image_batch,
                training=False
            )
        )

        logits = (
            tf.linalg.matmul(
                head_features,
                classifier_kernel
            )
            + classifier_bias
        )

        target_logit = logits[
            :,
            TARGET_CLASS
        ]

    gradients = tape.gradient(
        target_logit,
        feature_maps
    )

    weights = tf.reduce_mean(
        gradients,
        axis=(1, 2)
    )

    cam = tf.reduce_sum(
        feature_maps
        * weights[
            :,
            tf.newaxis,
            tf.newaxis,
            :
        ],
        axis=-1
    )

    cam = tf.nn.relu(
        cam
    )

    cam = cam[0]

    cam = tf.image.resize(
        cam[
            ...,
            tf.newaxis
        ],
        (224, 224),
        method="bilinear"
    )[
        ...,
        0
    ]

    cam = cam.numpy()

    if np.max(cam) > 0:

        cam = (
            cam
            / np.max(cam)
        )

    return cam


def heatmap_metrics(
    cam,
    bbox_mask
):

    total_energy = float(
        cam.sum()
    )

    bbox_area_fraction = float(
        bbox_mask.mean()
    )

    if total_energy > 0:

        bbox_energy_fraction = float(
            cam[
                bbox_mask
            ].sum()
            / total_energy
        )

    else:

        bbox_energy_fraction = np.nan

    if (
        bbox_area_fraction > 0
        and np.isfinite(
            bbox_energy_fraction
        )
    ):

        energy_enrichment = (
            bbox_energy_fraction
            / bbox_area_fraction
        )

    else:

        energy_enrichment = np.nan

    max_location = np.unravel_index(
        np.argmax(cam),
        cam.shape
    )

    pointing_game = int(
        bbox_mask[
            max_location
        ]
    )

    flat = cam.ravel().astype(
        np.float64
    )

    flat_sum = flat.sum()

    if flat_sum > 0:

        probability_mass = (
            flat
            / flat_sum
        )

        positive = (
            probability_mass > 0
        )

        entropy = -np.sum(
            probability_mass[
                positive
            ]
            * np.log(
                probability_mass[
                    positive
                ]
            )
        )

        normalized_entropy = (
            entropy
            / np.log(
                len(flat)
            )
        )

    else:

        normalized_entropy = np.nan

    return {
        "bbox_area_fraction":
            bbox_area_fraction,

        "bbox_energy_fraction":
            bbox_energy_fraction,

        "energy_enrichment":
            energy_enrichment,

        "pointing_game":
            pointing_game,

        "normalized_entropy":
            normalized_entropy
    }


val_opacity = val_df[
    val_df["class_id"] == 2
].copy()

val_opacity["position"] = (
    val_opacity["position"]
    .astype(str)
    .str.upper()
    .str.strip()
)

sample_parts = []

for position in ["AP", "PA"]:

    group = val_opacity[
        val_opacity["position"]
        == position
    ]

    n_take = min(
        MAX_PER_POSITION,
        len(group)
    )

    sampled = group.sample(
        n=n_take,
        random_state=XAI_SEED
    )

    sample_parts.append(
        sampled
    )

xai_validation_sample = pd.concat(
    sample_parts,
    ignore_index=True
)

print("Validation XAI sample")
print("--------------------------------------------")
print(
    xai_validation_sample[
        "position"
    ].value_counts()
)

print(
    "\nTotal:",
    len(xai_validation_sample)
)

results = []

for row_index, row in (
    xai_validation_sample
    .iterrows()
):

    image_tensor, _ = load_test_image(
        tf.constant(
            row["image_relpath"]
        ),
        tf.constant(
            TARGET_CLASS
        )
    )

    image_batch_local = tf.expand_dims(
        image_tensor,
        axis=0
    )

    bbox_mask = make_bbox_union_mask(
        row["patientId"],
        size=224
    )

    for layer_name in CANDIDATE_LAYERS:

        cam = compute_gradcam(
            image_batch_local,
            layer_name
        )

        metrics = heatmap_metrics(
            cam,
            bbox_mask
        )

        results.append(
            {
                "patientId":
                    row["patientId"],

                "position":
                    row["position"],

                "layer":
                    layer_name,

                **metrics
            }
        )

    if (
        (row_index + 1) % 25
        == 0
    ):

        print(
            "Processed",
            row_index + 1,
            "of",
            len(
                xai_validation_sample
            )
        )

layer_results = pd.DataFrame(
    results
)

layer_results.to_csv(
    XAI_LAYER_SELECTION_DIR
    / "validation_gradcam_layer_metrics.csv",
    index=False
)

overall_summary = (
    layer_results
    .groupby("layer")
    .agg(
        n=(
            "patientId",
            "count"
        ),
        mean_bbox_area_fraction=(
            "bbox_area_fraction",
            "mean"
        ),
        mean_bbox_energy_fraction=(
            "bbox_energy_fraction",
            "mean"
        ),
        mean_energy_enrichment=(
            "energy_enrichment",
            "mean"
        ),
        median_energy_enrichment=(
            "energy_enrichment",
            "median"
        ),
        pointing_game_accuracy=(
            "pointing_game",
            "mean"
        ),
        mean_normalized_entropy=(
            "normalized_entropy",
            "mean"
        )
    )
    .reset_index()
)

position_summary = (
    layer_results
    .groupby(
        [
            "layer",
            "position"
        ]
    )
    .agg(
        n=(
            "patientId",
            "count"
        ),
        mean_energy_enrichment=(
            "energy_enrichment",
            "mean"
        ),
        pointing_game_accuracy=(
            "pointing_game",
            "mean"
        ),
        mean_normalized_entropy=(
            "normalized_entropy",
            "mean"
        )
    )
    .reset_index()
)

overall_summary.to_csv(
    XAI_LAYER_SELECTION_DIR
    / "validation_gradcam_layer_summary.csv",
    index=False
)

position_summary.to_csv(
    XAI_LAYER_SELECTION_DIR
    / "validation_gradcam_layer_by_position.csv",
    index=False
)

print("\nOverall layer comparison")
print("--------------------------------------------")

print(
    overall_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nLayer comparison by position")
print("--------------------------------------------")

print(
    position_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nInterpretation")
print("--------------------------------------------")
print(
    "Energy enrichment > 1 means attribution is "
    "more concentrated inside radiologist boxes "
    "than expected from box area alone."
)

print(
    "Pointing game = fraction of cases whose "
    "maximum Grad-CAM location lies inside a box."
)

print(
    "Lower normalized entropy means a more "
    "spatially concentrated heatmap."
)

print("\nResults saved to:")
print(XAI_LAYER_SELECTION_DIR)


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tf_keras_vis.gradcam_plus_plus import GradcamPlusPlus
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
from tf_keras_vis.utils.scores import CategoricalScore

GRADCAMPP_DIR = (
    PROJECT_ROOT
    / "results"
    / "xai_protocol"
    / "validation_gradcamplusplus"
)

GRADCAMPP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CAM_LAYER = "multiscale_fusion"
TARGET_CLASS = 2

gradcam_pp = GradcamPlusPlus(
    final_advanced_model,
    model_modifier=ReplaceToLinear(),
    clone=True
)

score = CategoricalScore(
    TARGET_CLASS
)

gradcampp_results = []

print("Grad-CAM++ validation analysis")
print("--------------------------------------------")
print("Layer:", CAM_LAYER)
print(
    "Cases:",
    len(xai_validation_sample)
)

for row_index, row in (
    xai_validation_sample
    .reset_index(drop=True)
    .iterrows()
):

    image_tensor, _ = load_test_image(
        tf.constant(
            row["image_relpath"]
        ),
        tf.constant(
            TARGET_CLASS
        )
    )

    seed_input = np.expand_dims(
        image_tensor.numpy(),
        axis=0
    )

    cam = gradcam_pp(
        score,
        seed_input,
        penultimate_layer=CAM_LAYER,
        seek_penultimate_conv_layer=False,
        expand_cam=True,
        normalize_cam=True
    )

    cam = np.asarray(
        cam[0],
        dtype=np.float32
    )

    bbox_mask = make_bbox_union_mask(
        row["patientId"],
        size=224
    )

    metrics = heatmap_metrics(
        cam,
        bbox_mask
    )

    gradcampp_results.append(
        {
            "patientId":
                row["patientId"],

            "position":
                row["position"],

            "method":
                "Grad-CAM++",

            "layer":
                CAM_LAYER,

            **metrics
        }
    )

    if (
        (row_index + 1) % 25
        == 0
    ):
        print(
            "Processed",
            row_index + 1,
            "of",
            len(
                xai_validation_sample
            )
        )

gradcampp_df = pd.DataFrame(
    gradcampp_results
)

gradcampp_df.to_csv(
    GRADCAMPP_DIR
    / "validation_gradcamplusplus_metrics.csv",
    index=False
)

overall_gradcampp = (
    gradcampp_df
    .agg(
        {
            "bbox_area_fraction":
                "mean",

            "bbox_energy_fraction":
                "mean",

            "energy_enrichment":
                ["mean", "median"],

            "pointing_game":
                "mean",

            "normalized_entropy":
                "mean"
        }
    )
)

position_gradcampp = (
    gradcampp_df
    .groupby("position")
    .agg(
        n=(
            "patientId",
            "count"
        ),
        mean_bbox_energy_fraction=(
            "bbox_energy_fraction",
            "mean"
        ),
        mean_energy_enrichment=(
            "energy_enrichment",
            "mean"
        ),
        median_energy_enrichment=(
            "energy_enrichment",
            "median"
        ),
        pointing_game_accuracy=(
            "pointing_game",
            "mean"
        ),
        mean_normalized_entropy=(
            "normalized_entropy",
            "mean"
        )
    )
    .reset_index()
)

position_gradcampp.to_csv(
    GRADCAMPP_DIR
    / "validation_gradcamplusplus_by_position.csv",
    index=False
)

print("\nOverall Grad-CAM++")
print("--------------------------------------------")

print(
    "Mean bbox area fraction:",
    f"{gradcampp_df['bbox_area_fraction'].mean():.4f}"
)

print(
    "Mean bbox energy fraction:",
    f"{gradcampp_df['bbox_energy_fraction'].mean():.4f}"
)

print(
    "Mean energy enrichment:",
    f"{gradcampp_df['energy_enrichment'].mean():.4f}"
)

print(
    "Median energy enrichment:",
    f"{gradcampp_df['energy_enrichment'].median():.4f}"
)

print(
    "Pointing game accuracy:",
    f"{gradcampp_df['pointing_game'].mean():.4f}"
)

print(
    "Mean normalized entropy:",
    f"{gradcampp_df['normalized_entropy'].mean():.4f}"
)

print("\nGrad-CAM++ by position")
print("--------------------------------------------")

print(
    position_gradcampp.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# Compare directly with existing Grad-CAM
gradcam_reference = (
    layer_results[
        layer_results["layer"]
        == CAM_LAYER
    ]
    .copy()
)

comparison = pd.DataFrame(
    {
        "method": [
            "Grad-CAM",
            "Grad-CAM++"
        ],

        "mean_energy_enrichment": [
            gradcam_reference[
                "energy_enrichment"
            ].mean(),

            gradcampp_df[
                "energy_enrichment"
            ].mean()
        ],

        "median_energy_enrichment": [
            gradcam_reference[
                "energy_enrichment"
            ].median(),

            gradcampp_df[
                "energy_enrichment"
            ].median()
        ],

        "pointing_game_accuracy": [
            gradcam_reference[
                "pointing_game"
            ].mean(),

            gradcampp_df[
                "pointing_game"
            ].mean()
        ],

        "mean_normalized_entropy": [
            gradcam_reference[
                "normalized_entropy"
            ].mean(),

            gradcampp_df[
                "normalized_entropy"
            ].mean()
        ]
    }
)

comparison.to_csv(
    GRADCAMPP_DIR
    / "gradcam_vs_gradcamplusplus.csv",
    index=False
)

print("\nGrad-CAM vs Grad-CAM++")
print("--------------------------------------------")

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nResults saved to:")
print(GRADCAMPP_DIR)


In [ ]:
import numpy as np
import pandas as pd

from tf_keras_vis.layercam import Layercam
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
from tf_keras_vis.utils.scores import CategoricalScore


LAYERCAM_DIR = (
    PROJECT_ROOT
    / "results"
    / "xai_protocol"
    / "validation_layercam"
)

LAYERCAM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CAM_LAYER = "multiscale_fusion"
TARGET_CLASS = 2

layercam = Layercam(
    final_advanced_model,
    model_modifier=ReplaceToLinear(),
    clone=True
)

score = CategoricalScore(
    TARGET_CLASS
)

layercam_results = []

print("LayerCAM validation analysis")
print("--------------------------------------------")
print("Layer:", CAM_LAYER)
print(
    "Cases:",
    len(xai_validation_sample)
)

for row_index, row in (
    xai_validation_sample
    .reset_index(drop=True)
    .iterrows()
):

    image_tensor, _ = load_test_image(
        tf.constant(
            row["image_relpath"]
        ),
        tf.constant(
            TARGET_CLASS
        )
    )

    seed_input = np.expand_dims(
        image_tensor.numpy(),
        axis=0
    )

    cam = layercam(
        score,
        seed_input,
        penultimate_layer=CAM_LAYER,
        seek_penultimate_conv_layer=False,
        expand_cam=True,
        normalize_cam=True
    )

    cam = np.asarray(
        cam[0],
        dtype=np.float32
    )

    bbox_mask = make_bbox_union_mask(
        row["patientId"],
        size=224
    )

    metrics = heatmap_metrics(
        cam,
        bbox_mask
    )

    layercam_results.append(
        {
            "patientId":
                row["patientId"],

            "position":
                row["position"],

            "method":
                "LayerCAM",

            "layer":
                CAM_LAYER,

            **metrics
        }
    )

    if (
        (row_index + 1) % 25
        == 0
    ):
        print(
            "Processed",
            row_index + 1,
            "of",
            len(
                xai_validation_sample
            )
        )

layercam_df = pd.DataFrame(
    layercam_results
)

layercam_df.to_csv(
    LAYERCAM_DIR
    / "validation_layercam_metrics.csv",
    index=False
)

position_layercam = (
    layercam_df
    .groupby("position")
    .agg(
        n=(
            "patientId",
            "count"
        ),
        mean_bbox_energy_fraction=(
            "bbox_energy_fraction",
            "mean"
        ),
        mean_energy_enrichment=(
            "energy_enrichment",
            "mean"
        ),
        median_energy_enrichment=(
            "energy_enrichment",
            "median"
        ),
        pointing_game_accuracy=(
            "pointing_game",
            "mean"
        ),
        mean_normalized_entropy=(
            "normalized_entropy",
            "mean"
        )
    )
    .reset_index()
)

position_layercam.to_csv(
    LAYERCAM_DIR
    / "validation_layercam_by_position.csv",
    index=False
)

print("\nOverall LayerCAM")
print("--------------------------------------------")

print(
    "Mean bbox area fraction:",
    f"{layercam_df['bbox_area_fraction'].mean():.4f}"
)

print(
    "Mean bbox energy fraction:",
    f"{layercam_df['bbox_energy_fraction'].mean():.4f}"
)

print(
    "Mean energy enrichment:",
    f"{layercam_df['energy_enrichment'].mean():.4f}"
)

print(
    "Median energy enrichment:",
    f"{layercam_df['energy_enrichment'].median():.4f}"
)

print(
    "Pointing game accuracy:",
    f"{layercam_df['pointing_game'].mean():.4f}"
)

print(
    "Mean normalized entropy:",
    f"{layercam_df['normalized_entropy'].mean():.4f}"
)

print("\nLayerCAM by position")
print("--------------------------------------------")

print(
    position_layercam.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

gradcam_reference = (
    layer_results[
        layer_results["layer"]
        == CAM_LAYER
    ]
    .copy()
)

comparison_three = pd.DataFrame(
    {
        "method": [
            "Grad-CAM",
            "Grad-CAM++",
            "LayerCAM"
        ],

        "mean_energy_enrichment": [
            gradcam_reference[
                "energy_enrichment"
            ].mean(),

            gradcampp_df[
                "energy_enrichment"
            ].mean(),

            layercam_df[
                "energy_enrichment"
            ].mean()
        ],

        "median_energy_enrichment": [
            gradcam_reference[
                "energy_enrichment"
            ].median(),

            gradcampp_df[
                "energy_enrichment"
            ].median(),

            layercam_df[
                "energy_enrichment"
            ].median()
        ],

        "pointing_game_accuracy": [
            gradcam_reference[
                "pointing_game"
            ].mean(),

            gradcampp_df[
                "pointing_game"
            ].mean(),

            layercam_df[
                "pointing_game"
            ].mean()
        ],

        "mean_normalized_entropy": [
            gradcam_reference[
                "normalized_entropy"
            ].mean(),

            gradcampp_df[
                "normalized_entropy"
            ].mean(),

            layercam_df[
                "normalized_entropy"
            ].mean()
        ]
    }
)

comparison_three.to_csv(
    LAYERCAM_DIR
    / "gradcam_gradcampp_layercam_comparison.csv",
    index=False
)

print("\nThree-method comparison")
print("--------------------------------------------")

print(
    comparison_three.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nResults saved to:")
print(LAYERCAM_DIR)


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

IG_DIR = (
    PROJECT_ROOT
    / "results"
    / "xai_protocol"
    / "validation_integrated_gradients"
)

IG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TARGET_CLASS = 2
IG_STEPS = 32

classifier_layer = final_advanced_model.get_layer(
    "class_predictions"
)

classifier_kernel = classifier_layer.kernel
classifier_bias = classifier_layer.bias

head_model = tf.keras.Model(
    inputs=final_advanced_model.inputs,
    outputs=final_advanced_model
    .get_layer("dropout_256")
    .output
)


def get_target_logit(
    inputs
):

    head_features = head_model(
        inputs,
        training=False
    )

    logits = (
        tf.linalg.matmul(
            head_features,
            classifier_kernel
        )
        + classifier_bias
    )

    return logits[
        :,
        TARGET_CLASS
    ]


def integrated_gradients_map(
    image_tensor,
    steps=32
):

    image_tensor = tf.cast(
        image_tensor,
        tf.float32
    )

    # Zero in preprocessed [-1, 1] space
    # corresponds approximately to neutral mid-gray
    baseline = tf.zeros_like(
        image_tensor
    )

    alphas = tf.linspace(
        0.0,
        1.0,
        steps + 1
    )

    interpolated = (
        baseline[
            tf.newaxis,
            ...
        ]
        +
        alphas[
            :,
            tf.newaxis,
            tf.newaxis,
            tf.newaxis
        ]
        *
        (
            image_tensor
            - baseline
        )[
            tf.newaxis,
            ...
        ]
    )

    with tf.GradientTape() as tape:

        tape.watch(
            interpolated
        )

        target_scores = get_target_logit(
            interpolated
        )

    gradients = tape.gradient(
        target_scores,
        interpolated
    )

    # Trapezoidal approximation
    trapezoid_gradients = (
        gradients[:-1]
        + gradients[1:]
    ) / 2.0

    average_gradients = tf.reduce_mean(
        trapezoid_gradients,
        axis=0
    )

    integrated_gradients = (
        image_tensor
        - baseline
    ) * average_gradients

    # Signed attribution summed across duplicated RGB channels
    signed_map = tf.reduce_sum(
        integrated_gradients,
        axis=-1
    )

    # Positive evidence for Lung Opacity only
    positive_map = tf.nn.relu(
        signed_map
    )

    positive_map_np = (
        positive_map.numpy()
        .astype(np.float32)
    )

    maximum = float(
        positive_map_np.max()
    )

    if maximum > 0:

        normalized_map = (
            positive_map_np
            / maximum
        )

    else:

        normalized_map = (
            positive_map_np
        )

    # Integrated Gradients completeness diagnostic
    baseline_score = float(
        get_target_logit(
            baseline[
                tf.newaxis,
                ...
            ]
        )[0]
    )

    input_score = float(
        get_target_logit(
            image_tensor[
                tf.newaxis,
                ...
            ]
        )[0]
    )

    actual_logit_change = (
        input_score
        - baseline_score
    )

    attributed_change = float(
        tf.reduce_sum(
            integrated_gradients
        )
    )

    completeness_error = abs(
        actual_logit_change
        - attributed_change
    )

    return (
        normalized_map,
        completeness_error,
        actual_logit_change,
        attributed_change
    )


ig_results = []

print("Integrated Gradients validation analysis")
print("--------------------------------------------")
print(
    "Cases:",
    len(xai_validation_sample)
)
print(
    "Integration steps:",
    IG_STEPS
)
print(
    "Target:",
    "positive attribution to Lung Opacity logit"
)

for row_index, row in (
    xai_validation_sample
    .reset_index(drop=True)
    .iterrows()
):

    image_tensor, _ = load_test_image(
        tf.constant(
            row["image_relpath"]
        ),
        tf.constant(
            TARGET_CLASS
        )
    )

    (
        ig_map,
        completeness_error,
        actual_logit_change,
        attributed_change
    ) = integrated_gradients_map(
        image_tensor,
        steps=IG_STEPS
    )

    bbox_mask = make_bbox_union_mask(
        row["patientId"],
        size=224
    )

    metrics = heatmap_metrics(
        ig_map,
        bbox_mask
    )

    ig_results.append(
        {
            "patientId":
                row["patientId"],

            "position":
                row["position"],

            "method":
                "Integrated Gradients",

            "baseline":
                "zero_preprocessed_space",

            "steps":
                IG_STEPS,

            "completeness_error":
                completeness_error,

            "actual_logit_change":
                actual_logit_change,

            "attributed_logit_change":
                attributed_change,

            **metrics
        }
    )

    if (
        (row_index + 1) % 25
        == 0
    ):

        print(
            "Processed",
            row_index + 1,
            "of",
            len(
                xai_validation_sample
            )
        )

ig_df = pd.DataFrame(
    ig_results
)

ig_df.to_csv(
    IG_DIR
    / "validation_integrated_gradients_metrics.csv",
    index=False
)

position_ig = (
    ig_df
    .groupby("position")
    .agg(
        n=(
            "patientId",
            "count"
        ),

        mean_bbox_energy_fraction=(
            "bbox_energy_fraction",
            "mean"
        ),

        mean_energy_enrichment=(
            "energy_enrichment",
            "mean"
        ),

        median_energy_enrichment=(
            "energy_enrichment",
            "median"
        ),

        pointing_game_accuracy=(
            "pointing_game",
            "mean"
        ),

        mean_normalized_entropy=(
            "normalized_entropy",
            "mean"
        ),

        mean_completeness_error=(
            "completeness_error",
            "mean"
        )
    )
    .reset_index()
)

position_ig.to_csv(
    IG_DIR
    / "validation_integrated_gradients_by_position.csv",
    index=False
)

print("\nOverall Integrated Gradients")
print("--------------------------------------------")

print(
    "Mean bbox area fraction:",
    f"{ig_df['bbox_area_fraction'].mean():.4f}"
)

print(
    "Mean bbox energy fraction:",
    f"{ig_df['bbox_energy_fraction'].mean():.4f}"
)

print(
    "Mean energy enrichment:",
    f"{ig_df['energy_enrichment'].mean():.4f}"
)

print(
    "Median energy enrichment:",
    f"{ig_df['energy_enrichment'].median():.4f}"
)

print(
    "Pointing game accuracy:",
    f"{ig_df['pointing_game'].mean():.4f}"
)

print(
    "Mean normalized entropy:",
    f"{ig_df['normalized_entropy'].mean():.4f}"
)

print("\nCompleteness diagnostics")
print("--------------------------------------------")

print(
    "Mean absolute completeness error:",
    f"{ig_df['completeness_error'].mean():.6f}"
)

print(
    "Median absolute completeness error:",
    f"{ig_df['completeness_error'].median():.6f}"
)

print(
    "95th percentile completeness error:",
    f"{np.percentile(ig_df['completeness_error'], 95):.6f}"
)

print("\nIntegrated Gradients by position")
print("--------------------------------------------")

print(
    position_ig.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

comparison_four = pd.DataFrame(
    {
        "method": [
            "Grad-CAM",
            "Grad-CAM++",
            "LayerCAM",
            "Integrated Gradients"
        ],

        "mean_energy_enrichment": [
            gradcam_reference[
                "energy_enrichment"
            ].mean(),

            gradcampp_df[
                "energy_enrichment"
            ].mean(),

            layercam_df[
                "energy_enrichment"
            ].mean(),

            ig_df[
                "energy_enrichment"
            ].mean()
        ],

        "median_energy_enrichment": [
            gradcam_reference[
                "energy_enrichment"
            ].median(),

            gradcampp_df[
                "energy_enrichment"
            ].median(),

            layercam_df[
                "energy_enrichment"
            ].median(),

            ig_df[
                "energy_enrichment"
            ].median()
        ],

        "pointing_game_accuracy": [
            gradcam_reference[
                "pointing_game"
            ].mean(),

            gradcampp_df[
                "pointing_game"
            ].mean(),

            layercam_df[
                "pointing_game"
            ].mean(),

            ig_df[
                "pointing_game"
            ].mean()
        ],

        "mean_normalized_entropy": [
            gradcam_reference[
                "normalized_entropy"
            ].mean(),

            gradcampp_df[
                "normalized_entropy"
            ].mean(),

            layercam_df[
                "normalized_entropy"
            ].mean(),

            ig_df[
                "normalized_entropy"
            ].mean()
        ]
    }
)

comparison_four.to_csv(
    IG_DIR
    / "four_method_validation_comparison.csv",
    index=False
)

print("\nFour-method validation comparison")
print("--------------------------------------------")

print(
    comparison_four.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nResults saved to:")
print(IG_DIR)


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

IG_CONVERGENCE_DIR = (
    PROJECT_ROOT
    / "results"
    / "xai_protocol"
    / "integrated_gradients_convergence"
)

IG_CONVERGENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

IG_STEP_VALUES = [
    32,
    64,
    128,
    256
]

AUDIT_PER_POSITION = 20
TARGET_CLASS = 2

audit_parts = []

for position in ["AP", "PA"]:

    group = (
        xai_validation_sample[
            xai_validation_sample["position"] == position
        ]
        .sample(
            n=AUDIT_PER_POSITION,
            random_state=42
        )
    )

    audit_parts.append(
        group
    )

ig_audit_sample = pd.concat(
    audit_parts,
    ignore_index=True
)

print("IG convergence audit")
print("--------------------------------------------")
print(
    ig_audit_sample["position"]
    .value_counts()
)
print(
    "Total cases:",
    len(ig_audit_sample)
)


def integrated_gradients_audit(
    image_tensor,
    steps
):

    image_tensor = tf.cast(
        image_tensor,
        tf.float32
    )

    baseline = tf.zeros_like(
        image_tensor
    )

    alphas = tf.linspace(
        0.0,
        1.0,
        steps + 1
    )

    interpolated = (
        baseline[tf.newaxis, ...]
        +
        alphas[
            :,
            tf.newaxis,
            tf.newaxis,
            tf.newaxis
        ]
        *
        (
            image_tensor
            - baseline
        )[tf.newaxis, ...]
    )

    with tf.GradientTape() as tape:

        tape.watch(
            interpolated
        )

        scores = get_target_logit(
            interpolated
        )

    gradients = tape.gradient(
        scores,
        interpolated
    )

    trapezoid_gradients = (
        gradients[:-1]
        + gradients[1:]
    ) / 2.0

    average_gradients = tf.reduce_mean(
        trapezoid_gradients,
        axis=0
    )

    ig = (
        image_tensor
        - baseline
    ) * average_gradients

    baseline_score = float(
        get_target_logit(
            baseline[tf.newaxis, ...]
        )[0]
    )

    input_score = float(
        get_target_logit(
            image_tensor[tf.newaxis, ...]
        )[0]
    )

    true_delta = (
        input_score
        - baseline_score
    )

    ig_sum = float(
        tf.reduce_sum(
            ig
        )
    )

    absolute_error = abs(
        true_delta
        - ig_sum
    )

    relative_error = (
        absolute_error
        / (
            abs(true_delta)
            + 1e-8
        )
    )

    signed_map = tf.reduce_sum(
        ig,
        axis=-1
    )

    positive_map = tf.nn.relu(
        signed_map
    ).numpy()

    if positive_map.max() > 0:

        positive_map = (
            positive_map
            / positive_map.max()
        )

    return (
        positive_map,
        true_delta,
        ig_sum,
        absolute_error,
        relative_error
    )


audit_rows = []

for case_index, row in (
    ig_audit_sample
    .iterrows()
):

    image_tensor, _ = load_test_image(
        tf.constant(
            row["image_relpath"]
        ),
        tf.constant(
            TARGET_CLASS
        )
    )

    bbox_mask = make_bbox_union_mask(
        row["patientId"],
        size=224
    )

    for steps in IG_STEP_VALUES:

        (
            ig_map,
            true_delta,
            ig_sum,
            abs_error,
            rel_error
        ) = integrated_gradients_audit(
            image_tensor,
            steps=steps
        )

        localization = heatmap_metrics(
            ig_map,
            bbox_mask
        )

        audit_rows.append(
            {
                "patientId":
                    row["patientId"],

                "position":
                    row["position"],

                "steps":
                    steps,

                "true_logit_delta":
                    true_delta,

                "integrated_attribution_sum":
                    ig_sum,

                "absolute_completeness_error":
                    abs_error,

                "relative_completeness_error":
                    rel_error,

                "energy_enrichment":
                    localization[
                        "energy_enrichment"
                    ],

                "pointing_game":
                    localization[
                        "pointing_game"
                    ],

                "normalized_entropy":
                    localization[
                        "normalized_entropy"
                    ]
            }
        )

    if (
        (case_index + 1) % 10
        == 0
    ):

        print(
            "Processed",
            case_index + 1,
            "of",
            len(ig_audit_sample)
        )


ig_convergence_df = pd.DataFrame(
    audit_rows
)

ig_convergence_df.to_csv(
    IG_CONVERGENCE_DIR
    / "ig_convergence_case_level.csv",
    index=False
)

convergence_summary = (
    ig_convergence_df
    .groupby("steps")
    .agg(
        n=(
            "patientId",
            "count"
        ),

        mean_abs_error=(
            "absolute_completeness_error",
            "mean"
        ),

        median_abs_error=(
            "absolute_completeness_error",
            "median"
        ),

        p95_abs_error=(
            "absolute_completeness_error",
            lambda x: np.percentile(
                x,
                95
            )
        ),

        mean_relative_error=(
            "relative_completeness_error",
            "mean"
        ),

        median_relative_error=(
            "relative_completeness_error",
            "median"
        ),

        mean_energy_enrichment=(
            "energy_enrichment",
            "mean"
        ),

        pointing_game_accuracy=(
            "pointing_game",
            "mean"
        ),

        mean_normalized_entropy=(
            "normalized_entropy",
            "mean"
        )
    )
    .reset_index()
)

convergence_summary.to_csv(
    IG_CONVERGENCE_DIR
    / "ig_convergence_summary.csv",
    index=False
)

print("\nIG convergence summary")
print("--------------------------------------------")

print(
    convergence_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nResults saved to:")
print(IG_CONVERGENCE_DIR)


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

IG_CONVERGENCE_DIR = (
    PROJECT_ROOT
    / "results"
    / "xai_protocol"
    / "integrated_gradients_convergence"
)

IG_CONVERGENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

IG_STEP_VALUES = [
    32,
    64,
    128,
    256
]

AUDIT_PER_POSITION = 20
TARGET_CLASS = 2

audit_parts = []

for position in ["AP", "PA"]:

    group = (
        xai_validation_sample[
            xai_validation_sample["position"] == position
        ]
        .sample(
            n=AUDIT_PER_POSITION,
            random_state=42
        )
    )

    audit_parts.append(group)

ig_audit_sample = pd.concat(
    audit_parts,
    ignore_index=True
)

print("IG convergence audit")
print("--------------------------------------------")
print(
    ig_audit_sample["position"]
    .value_counts()
)
print(
    "Total cases:",
    len(ig_audit_sample)
)


def integrated_gradients_audit(
    image_tensor,
    steps
):

    image_tensor = tf.cast(
        image_tensor,
        tf.float32
    )

    baseline = tf.zeros_like(
        image_tensor
    )

    alphas = tf.linspace(
        0.0,
        1.0,
        steps + 1
    )

    interpolated = (
        baseline[tf.newaxis, ...]
        +
        alphas[
            :,
            tf.newaxis,
            tf.newaxis,
            tf.newaxis
        ]
        *
        (
            image_tensor
            - baseline
        )[tf.newaxis, ...]
    )

    with tf.GradientTape() as tape:

        tape.watch(
            interpolated
        )

        scores = get_target_logit(
            interpolated
        )

    gradients = tape.gradient(
        scores,
        interpolated
    )

    trapezoid_gradients = (
        gradients[:-1]
        + gradients[1:]
    ) / 2.0

    average_gradients = tf.reduce_mean(
        trapezoid_gradients,
        axis=0
    )

    ig = (
        image_tensor
        - baseline
    ) * average_gradients

    baseline_score = float(
        get_target_logit(
            baseline[tf.newaxis, ...]
        )[0]
    )

    input_score = float(
        get_target_logit(
            image_tensor[tf.newaxis, ...]
        )[0]
    )

    true_delta = (
        input_score
        - baseline_score
    )

    ig_sum = float(
        tf.reduce_sum(ig)
    )

    absolute_error = abs(
        true_delta
        - ig_sum
    )

    relative_error = (
        absolute_error
        / (
            abs(true_delta)
            + 1e-8
        )
    )

    signed_map = tf.reduce_sum(
        ig,
        axis=-1
    )

    positive_map = tf.nn.relu(
        signed_map
    ).numpy()

    if positive_map.max() > 0:

        positive_map = (
            positive_map
            / positive_map.max()
        )

    return (
        positive_map,
        true_delta,
        ig_sum,
        absolute_error,
        relative_error
    )


audit_rows = []

for case_index, row in (
    ig_audit_sample
    .iterrows()
):

    image_tensor, _ = load_test_image(
        tf.constant(
            row["image_relpath"]
        ),
        tf.constant(
            TARGET_CLASS
        )
    )

    bbox_mask = make_bbox_union_mask(
        row["patientId"],
        size=224
    )

    for steps in IG_STEP_VALUES:

        (
            ig_map,
            true_delta,
            ig_sum,
            abs_error,
            rel_error
        ) = integrated_gradients_audit(
            image_tensor,
            steps=steps
        )

        localization = heatmap_metrics(
            ig_map,
            bbox_mask
        )

        audit_rows.append(
            {
                "patientId":
                    row["patientId"],

                "position":
                    row["position"],

                "steps":
                    steps,

                "true_logit_delta":
                    true_delta,

                "integrated_attribution_sum":
                    ig_sum,

                "absolute_completeness_error":
                    abs_error,

                "relative_completeness_error":
                    rel_error,

                "energy_enrichment":
                    localization[
                        "energy_enrichment"
                    ],

                "pointing_game":
                    localization[
                        "pointing_game"
                    ],

                "normalized_entropy":
                    localization[
                        "normalized_entropy"
                    ]
            }
        )

    if (
        (case_index + 1) % 10
        == 0
    ):

        print(
            "Processed",
            case_index + 1,
            "of",
            len(ig_audit_sample)
        )


ig_convergence_df = pd.DataFrame(
    audit_rows
)

ig_convergence_df.to_csv(
    IG_CONVERGENCE_DIR
    / "ig_convergence_case_level.csv",
    index=False
)

convergence_summary = (
    ig_convergence_df
    .groupby("steps")
    .agg(
        n=(
            "patientId",
            "count"
        ),

        mean_abs_error=(
            "absolute_completeness_error",
            "mean"
        ),

        median_abs_error=(
            "absolute_completeness_error",
            "median"
        ),

        p95_abs_error=(
            "absolute_completeness_error",
            lambda x: np.percentile(
                x,
                95
            )
        ),

        mean_relative_error=(
            "relative_completeness_error",
            "mean"
        ),

        median_relative_error=(
            "relative_completeness_error",
            "median"
        ),

        mean_energy_enrichment=(
            "energy_enrichment",
            "mean"
        ),

        pointing_game_accuracy=(
            "pointing_game",
            "mean"
        ),

        mean_normalized_entropy=(
            "normalized_entropy",
            "mean"
        )
    )
    .reset_index()
)

convergence_summary.to_csv(
    IG_CONVERGENCE_DIR
    / "ig_convergence_summary.csv",
    index=False
)

print("\nIG convergence summary")
print("--------------------------------------------")

print(
    convergence_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nResults saved to:")
print(IG_CONVERGENCE_DIR)


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

IG_FINAL_STEPS = 256

IG_FINAL_DIR = (
    PROJECT_ROOT
    / "results"
    / "xai_protocol"
    / "validation_integrated_gradients_256"
)

IG_FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

final_ig_results = []

print("Final Integrated Gradients validation analysis")
print("--------------------------------------------")
print("Cases:", len(xai_validation_sample))
print("Integration steps:", IG_FINAL_STEPS)
print("Target: positive attribution to Lung Opacity logit")

for row_index, row in (
    xai_validation_sample
    .reset_index(drop=True)
    .iterrows()
):

    image_tensor, _ = load_test_image(
        tf.constant(
            row["image_relpath"]
        ),
        tf.constant(
            TARGET_CLASS
        )
    )

    (
        ig_map,
        true_delta,
        ig_sum,
        abs_error,
        rel_error
    ) = integrated_gradients_audit(
        image_tensor,
        steps=IG_FINAL_STEPS
    )

    bbox_mask = make_bbox_union_mask(
        row["patientId"],
        size=224
    )

    metrics = heatmap_metrics(
        ig_map,
        bbox_mask
    )

    final_ig_results.append(
        {
            "patientId": row["patientId"],
            "position": row["position"],
            "method": "Integrated Gradients",
            "steps": IG_FINAL_STEPS,
            "true_logit_delta": true_delta,
            "integrated_attribution_sum": ig_sum,
            "absolute_completeness_error": abs_error,
            "relative_completeness_error": rel_error,
            **metrics
        }
    )

    if (row_index + 1) % 25 == 0:

        print(
            "Processed",
            row_index + 1,
            "of",
            len(xai_validation_sample)
        )


ig256_df = pd.DataFrame(
    final_ig_results
)

ig256_df.to_csv(
    IG_FINAL_DIR
    / "validation_integrated_gradients_256_metrics.csv",
    index=False
)

ig256_position = (
    ig256_df
    .groupby("position")
    .agg(
        n=(
            "patientId",
            "count"
        ),
        mean_bbox_energy_fraction=(
            "bbox_energy_fraction",
            "mean"
        ),
        mean_energy_enrichment=(
            "energy_enrichment",
            "mean"
        ),
        median_energy_enrichment=(
            "energy_enrichment",
            "median"
        ),
        pointing_game_accuracy=(
            "pointing_game",
            "mean"
        ),
        mean_normalized_entropy=(
            "normalized_entropy",
            "mean"
        ),
        mean_abs_completeness_error=(
            "absolute_completeness_error",
            "mean"
        ),
        median_abs_completeness_error=(
            "absolute_completeness_error",
            "median"
        ),
        median_relative_completeness_error=(
            "relative_completeness_error",
            "median"
        )
    )
    .reset_index()
)

ig256_position.to_csv(
    IG_FINAL_DIR
    / "validation_integrated_gradients_256_by_position.csv",
    index=False
)

print("\nOverall Integrated Gradients, 256 steps")
print("--------------------------------------------")

print(
    "Mean bbox area fraction:",
    f"{ig256_df['bbox_area_fraction'].mean():.4f}"
)

print(
    "Mean bbox energy fraction:",
    f"{ig256_df['bbox_energy_fraction'].mean():.4f}"
)

print(
    "Mean energy enrichment:",
    f"{ig256_df['energy_enrichment'].mean():.4f}"
)

print(
    "Median energy enrichment:",
    f"{ig256_df['energy_enrichment'].median():.4f}"
)

print(
    "Pointing game accuracy:",
    f"{ig256_df['pointing_game'].mean():.4f}"
)

print(
    "Mean normalized entropy:",
    f"{ig256_df['normalized_entropy'].mean():.4f}"
)

print("\nCompleteness")
print("--------------------------------------------")

print(
    "Mean absolute error:",
    f"{ig256_df['absolute_completeness_error'].mean():.6f}"
)

print(
    "Median absolute error:",
    f"{ig256_df['absolute_completeness_error'].median():.6f}"
)

print(
    "95th percentile absolute error:",
    f"{np.percentile(ig256_df['absolute_completeness_error'], 95):.6f}"
)

print(
    "Median relative error:",
    f"{ig256_df['relative_completeness_error'].median():.6f}"
)

print("\nIG 256 by position")
print("--------------------------------------------")

print(
    ig256_position.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

final_method_comparison = pd.DataFrame(
    {
        "method": [
            "Grad-CAM",
            "Grad-CAM++",
            "LayerCAM",
            "Integrated Gradients 256"
        ],

        "mean_energy_enrichment": [
            gradcam_reference[
                "energy_enrichment"
            ].mean(),

            gradcampp_df[
                "energy_enrichment"
            ].mean(),

            layercam_df[
                "energy_enrichment"
            ].mean(),

            ig256_df[
                "energy_enrichment"
            ].mean()
        ],

        "median_energy_enrichment": [
            gradcam_reference[
                "energy_enrichment"
            ].median(),

            gradcampp_df[
                "energy_enrichment"
            ].median(),

            layercam_df[
                "energy_enrichment"
            ].median(),

            ig256_df[
                "energy_enrichment"
            ].median()
        ],

        "pointing_game_accuracy": [
            gradcam_reference[
                "pointing_game"
            ].mean(),

            gradcampp_df[
                "pointing_game"
            ].mean(),

            layercam_df[
                "pointing_game"
            ].mean(),

            ig256_df[
                "pointing_game"
            ].mean()
        ],

        "mean_normalized_entropy": [
            gradcam_reference[
                "normalized_entropy"
            ].mean(),

            gradcampp_df[
                "normalized_entropy"
            ].mean(),

            layercam_df[
                "normalized_entropy"
            ].mean(),

            ig256_df[
                "normalized_entropy"
            ].mean()
        ]
    }
)

final_method_comparison.to_csv(
    IG_FINAL_DIR
    / "final_validation_xai_method_comparison.csv",
    index=False
)

print("\nFinal validation XAI comparison")
print("--------------------------------------------")

print(
    final_method_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nResults saved to:")
print(IG_FINAL_DIR)


In [ ]:
import numpy as np
import pandas as pd

from tf_keras_vis.scorecam import Scorecam
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
from tf_keras_vis.utils.scores import CategoricalScore


SCORECAM_DIR = (
    PROJECT_ROOT
    / "results"
    / "xai_protocol"
    / "validation_scorecam"
)

SCORECAM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CAM_LAYER = "multiscale_fusion"
TARGET_CLASS = 2
SCORECAM_MAX_N = 64
SAMPLE_PER_POSITION = 50

scorecam_parts = []

for position in ["AP", "PA"]:

    subset = (
        xai_validation_sample[
            xai_validation_sample["position"] == position
        ]
        .sample(
            n=SAMPLE_PER_POSITION,
            random_state=42
        )
    )

    scorecam_parts.append(
        subset
    )

scorecam_sample = pd.concat(
    scorecam_parts,
    ignore_index=True
)

scorecam = Scorecam(
    final_advanced_model,
    model_modifier=ReplaceToLinear(),
    clone=True
)

score = CategoricalScore(
    TARGET_CLASS
)

scorecam_results = []

print("Score-CAM validation sensitivity analysis")
print("--------------------------------------------")
print("Layer:", CAM_LAYER)
print("Cases:", len(scorecam_sample))
print("AP:", sum(scorecam_sample["position"] == "AP"))
print("PA:", sum(scorecam_sample["position"] == "PA"))
print("max_N:", SCORECAM_MAX_N)

for row_index, row in (
    scorecam_sample
    .reset_index(drop=True)
    .iterrows()
):

    image_tensor, _ = load_test_image(
        tf.constant(
            row["image_relpath"]
        ),
        tf.constant(
            TARGET_CLASS
        )
    )

    seed_input = np.expand_dims(
        image_tensor.numpy(),
        axis=0
    )

    cam = scorecam(
        score,
        seed_input,
        penultimate_layer=CAM_LAYER,
        seek_penultimate_conv_layer=False,
        max_N=SCORECAM_MAX_N,
        expand_cam=True,
        normalize_cam=True
    )

    cam = np.asarray(
        cam[0],
        dtype=np.float32
    )

    bbox_mask = make_bbox_union_mask(
        row["patientId"],
        size=224
    )

    metrics = heatmap_metrics(
        cam,
        bbox_mask
    )

    scorecam_results.append(
        {
            "patientId":
                row["patientId"],

            "position":
                row["position"],

            "method":
                "Score-CAM top-64",

            "layer":
                CAM_LAYER,

            "max_N":
                SCORECAM_MAX_N,

            **metrics
        }
    )

    if (
        (row_index + 1) % 10
        == 0
    ):

        print(
            "Processed",
            row_index + 1,
            "of",
            len(scorecam_sample)
        )


scorecam_df = pd.DataFrame(
    scorecam_results
)

scorecam_df.to_csv(
    SCORECAM_DIR
    / "validation_scorecam_top64_metrics.csv",
    index=False
)


print("\nOverall Score-CAM")
print("--------------------------------------------")

print(
    "Mean bbox area fraction:",
    f"{scorecam_df['bbox_area_fraction'].mean():.4f}"
)

print(
    "Mean bbox energy fraction:",
    f"{scorecam_df['bbox_energy_fraction'].mean():.4f}"
)

print(
    "Mean energy enrichment:",
    f"{scorecam_df['energy_enrichment'].mean():.4f}"
)

print(
    "Median energy enrichment:",
    f"{scorecam_df['energy_enrichment'].median():.4f}"
)

print(
    "Pointing game accuracy:",
    f"{scorecam_df['pointing_game'].mean():.4f}"
)

print(
    "Mean normalized entropy:",
    f"{scorecam_df['normalized_entropy'].mean():.4f}"
)


position_scorecam = (
    scorecam_df
    .groupby("position")
    .agg(
        n=(
            "patientId",
            "count"
        ),
        mean_bbox_energy_fraction=(
            "bbox_energy_fraction",
            "mean"
        ),
        mean_energy_enrichment=(
            "energy_enrichment",
            "mean"
        ),
        median_energy_enrichment=(
            "energy_enrichment",
            "median"
        ),
        pointing_game_accuracy=(
            "pointing_game",
            "mean"
        ),
        mean_normalized_entropy=(
            "normalized_entropy",
            "mean"
        )
    )
    .reset_index()
)

print("\nScore-CAM by position")
print("--------------------------------------------")

print(
    position_scorecam.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ----------------------------------------------
# Fair comparison on the exact same 100 cases
# ----------------------------------------------

sample_ids = set(
    scorecam_sample["patientId"]
)

gradcam_same = gradcam_reference[
    gradcam_reference[
        "patientId"
    ].isin(sample_ids)
]

gradcampp_same = gradcampp_df[
    gradcampp_df[
        "patientId"
    ].isin(sample_ids)
]

layercam_same = layercam_df[
    layercam_df[
        "patientId"
    ].isin(sample_ids)
]

ig_same = ig256_df[
    ig256_df[
        "patientId"
    ].isin(sample_ids)
]


same_sample_comparison = pd.DataFrame(
    {
        "method": [
            "Grad-CAM",
            "Grad-CAM++",
            "LayerCAM",
            "Integrated Gradients 256",
            "Score-CAM top-64"
        ],

        "mean_energy_enrichment": [
            gradcam_same[
                "energy_enrichment"
            ].mean(),

            gradcampp_same[
                "energy_enrichment"
            ].mean(),

            layercam_same[
                "energy_enrichment"
            ].mean(),

            ig_same[
                "energy_enrichment"
            ].mean(),

            scorecam_df[
                "energy_enrichment"
            ].mean()
        ],

        "median_energy_enrichment": [
            gradcam_same[
                "energy_enrichment"
            ].median(),

            gradcampp_same[
                "energy_enrichment"
            ].median(),

            layercam_same[
                "energy_enrichment"
            ].median(),

            ig_same[
                "energy_enrichment"
            ].median(),

            scorecam_df[
                "energy_enrichment"
            ].median()
        ],

        "pointing_game_accuracy": [
            gradcam_same[
                "pointing_game"
            ].mean(),

            gradcampp_same[
                "pointing_game"
            ].mean(),

            layercam_same[
                "pointing_game"
            ].mean(),

            ig_same[
                "pointing_game"
            ].mean(),

            scorecam_df[
                "pointing_game"
            ].mean()
        ],

        "mean_normalized_entropy": [
            gradcam_same[
                "normalized_entropy"
            ].mean(),

            gradcampp_same[
                "normalized_entropy"
            ].mean(),

            layercam_same[
                "normalized_entropy"
            ].mean(),

            ig_same[
                "normalized_entropy"
            ].mean(),

            scorecam_df[
                "normalized_entropy"
            ].mean()
        ]
    }
)

same_sample_comparison.to_csv(
    SCORECAM_DIR
    / "five_method_same_100_case_comparison.csv",
    index=False
)

position_scorecam.to_csv(
    SCORECAM_DIR
    / "validation_scorecam_top64_by_position.csv",
    index=False
)

print("\nFive-method comparison on SAME 100 cases")
print("--------------------------------------------")

print(
    same_sample_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nResults saved to:")
print(SCORECAM_DIR)


## 15. Locked-test XAI analysis

In [ ]:
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from tf_keras_vis.layercam import Layercam
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
from tf_keras_vis.utils.scores import CategoricalScore


# ---------------------------------------------------------
# 1. Lock XAI protocol before test localization analysis
# ---------------------------------------------------------

XAI_PROTOCOL_DIR = (
    PROJECT_ROOT
    / "results"
    / "xai_protocol"
)

XAI_PROTOCOL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

XAI_PROTOCOL_PATH = (
    XAI_PROTOCOL_DIR
    / "xai_protocol_lock.json"
)

xai_protocol = {
    "selection_split": "validation",
    "no_more_xai_method_or_layer_tuning": True,

    "target_class": {
        "class_id": 2,
        "name": "Lung Opacity"
    },

    "primary_method": {
        "method": "LayerCAM",
        "layer": "multiscale_fusion",
        "score": "pre-softmax Lung Opacity logit"
    },

    "complementary_method": {
        "method": "Integrated Gradients",
        "integration_steps": 256,
        "baseline": "zero in preprocessed [-1, 1] space",
        "attribution": "positive attribution to Lung Opacity logit"
    },

    "baseline_method": {
        "method": "Grad-CAM",
        "layer": "multiscale_fusion"
    },

    "sensitivity_methods": [
        {
            "method": "Grad-CAM++",
            "layer": "multiscale_fusion"
        },
        {
            "method": "Score-CAM top-64 approximation",
            "layer": "multiscale_fusion",
            "max_N": 64
        }
    ],

    "primary_localization_metrics": [
        "bbox_energy_fraction",
        "energy_enrichment",
        "pointing_game",
        "normalized_entropy"
    ],

    "planned_test_subgroups": [
        "AP",
        "PA",
        "correct Lung Opacity",
        "missed Lung Opacity",
        "AP correct",
        "AP missed",
        "PA correct",
        "PA missed"
    ]
}

with open(
    XAI_PROTOCOL_PATH,
    "w"
) as f:

    json.dump(
        xai_protocol,
        f,
        indent=2
    )


# ---------------------------------------------------------
# 2. Prepare locked test Lung Opacity cases
# ---------------------------------------------------------

TEST_LAYERCAM_DIR = (
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "xai"
    / "layercam"
)

TEST_LAYERCAM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_PATH = (
    TEST_LAYERCAM_DIR
    / "test_layercam_case_metrics_checkpoint.csv"
)

FINAL_METRICS_PATH = (
    TEST_LAYERCAM_DIR
    / "test_layercam_case_metrics.csv"
)

test_xai_df = pd.read_csv(
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "test_predictions.csv"
)

test_xai_df["position"] = (
    test_xai_df["position"]
    .astype(str)
    .str.upper()
    .str.strip()
)

test_opacity_xai = (
    test_xai_df[
        test_xai_df["class_id"] == 2
    ]
    .copy()
    .reset_index(drop=True)
)

test_opacity_xai["correct_opacity"] = (
    test_opacity_xai[
        "predicted_class_id"
    ] == 2
)

print("Locked TEST Lung Opacity XAI set")
print("--------------------------------------------")
print("Total cases:", len(test_opacity_xai))

print("\nPosition:")
print(
    test_opacity_xai[
        "position"
    ].value_counts()
)

print("\nCorrect vs missed:")
print(
    test_opacity_xai[
        "correct_opacity"
    ].value_counts()
)

print("\nPosition x correctness:")
print(
    pd.crosstab(
        test_opacity_xai["position"],
        test_opacity_xai["correct_opacity"]
    )
)


# ---------------------------------------------------------
# 3. Create locked LayerCAM
# ---------------------------------------------------------

TEST_CAM_LAYER = "multiscale_fusion"
TARGET_CLASS = 2

test_layercam = Layercam(
    final_advanced_model,
    model_modifier=ReplaceToLinear(),
    clone=True
)

test_layercam_score = CategoricalScore(
    TARGET_CLASS
)


# ---------------------------------------------------------
# 4. Resume support
# ---------------------------------------------------------

if CHECKPOINT_PATH.exists():

    completed_df = pd.read_csv(
        CHECKPOINT_PATH
    )

    completed_ids = set(
        completed_df["patientId"]
    )

    layercam_test_results = (
        completed_df
        .to_dict("records")
    )

    print(
        "\nResuming from checkpoint:"
    )

    print(
        len(completed_ids),
        "cases already completed"
    )

else:

    completed_ids = set()
    layercam_test_results = []

    print(
        "\nStarting LayerCAM test analysis from beginning."
    )


remaining_df = (
    test_opacity_xai[
        ~test_opacity_xai[
            "patientId"
        ].isin(
            completed_ids
        )
    ]
    .reset_index(drop=True)
)

print(
    "Remaining cases:",
    len(remaining_df)
)


# ---------------------------------------------------------
# 5. Full locked-test LayerCAM analysis
# ---------------------------------------------------------

for row_index, row in (
    remaining_df.iterrows()
):

    image_tensor, _ = load_test_image(
        tf.constant(
            row["image_relpath"]
        ),
        tf.constant(
            TARGET_CLASS
        )
    )

    seed_input = np.expand_dims(
        image_tensor.numpy(),
        axis=0
    )

    cam = test_layercam(
        test_layercam_score,
        seed_input,
        penultimate_layer=TEST_CAM_LAYER,
        seek_penultimate_conv_layer=False,
        expand_cam=True,
        normalize_cam=True
    )

    cam = np.asarray(
        cam[0],
        dtype=np.float32
    )

    bbox_mask = make_bbox_union_mask(
        row["patientId"],
        size=224
    )

    metrics = heatmap_metrics(
        cam,
        bbox_mask
    )

    layercam_test_results.append(
        {
            "patientId":
                row["patientId"],

            "position":
                row["position"],

            "predicted_class_id":
                int(
                    row[
                        "predicted_class_id"
                    ]
                ),

            "correct_opacity":
                bool(
                    row[
                        "correct_opacity"
                    ]
                ),

            "prob_lung_opacity":
                float(
                    row[
                        "prob_lung_opacity"
                    ]
                ),

            "method":
                "LayerCAM",

            "layer":
                TEST_CAM_LAYER,

            **metrics
        }
    )

    total_completed = (
        len(layercam_test_results)
    )

    if (
        total_completed % 25
        == 0
    ):

        pd.DataFrame(
            layercam_test_results
        ).to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        print(
            "Completed",
            total_completed,
            "of",
            len(test_opacity_xai)
        )


# ---------------------------------------------------------
# 6. Final save
# ---------------------------------------------------------

test_layercam_df = pd.DataFrame(
    layercam_test_results
)

test_layercam_df = (
    test_layercam_df
    .drop_duplicates(
        subset="patientId",
        keep="last"
    )
)

test_layercam_df.to_csv(
    FINAL_METRICS_PATH,
    index=False
)

test_layercam_df.to_csv(
    CHECKPOINT_PATH,
    index=False
)


# ---------------------------------------------------------
# 7. Initial summaries only
# ---------------------------------------------------------

print("\nLayerCAM locked-test analysis complete")
print("--------------------------------------------")

print(
    "Cases:",
    len(test_layercam_df)
)

print(
    "Mean bbox energy fraction:",
    f"{test_layercam_df['bbox_energy_fraction'].mean():.4f}"
)

print(
    "Mean energy enrichment:",
    f"{test_layercam_df['energy_enrichment'].mean():.4f}"
)

print(
    "Median energy enrichment:",
    f"{test_layercam_df['energy_enrichment'].median():.4f}"
)

print(
    "Pointing game accuracy:",
    f"{test_layercam_df['pointing_game'].mean():.4f}"
)

print(
    "Mean normalized entropy:",
    f"{test_layercam_df['normalized_entropy'].mean():.4f}"
)


group_summary = (
    test_layercam_df
    .groupby(
        [
            "position",
            "correct_opacity"
        ]
    )
    .agg(
        n=(
            "patientId",
            "count"
        ),

        mean_bbox_energy_fraction=(
            "bbox_energy_fraction",
            "mean"
        ),

        mean_energy_enrichment=(
            "energy_enrichment",
            "mean"
        ),

        median_energy_enrichment=(
            "energy_enrichment",
            "median"
        ),

        pointing_game_accuracy=(
            "pointing_game",
            "mean"
        ),

        mean_normalized_entropy=(
            "normalized_entropy",
            "mean"
        ),

        mean_opacity_probability=(
            "prob_lung_opacity",
            "mean"
        )
    )
    .reset_index()
)

group_summary.to_csv(
    TEST_LAYERCAM_DIR
    / "test_layercam_position_correctness_summary.csv",
    index=False
)

print("\nPosition x correctness")
print("--------------------------------------------")

print(
    group_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nXAI protocol locked at:")
print(XAI_PROTOCOL_PATH)

print("\nLayerCAM results saved to:")
print(FINAL_METRICS_PATH)


In [ ]:
import numpy as np
import pandas as pd

LAYERCAM_STATS_DIR = (
    TEST_LAYERCAM_DIR
    / "statistical_analysis"
)

LAYERCAM_STATS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BOOTSTRAP_N = 5000
BOOTSTRAP_SEED = 42

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

analysis_layercam = (
    test_layercam_df
    .copy()
)

analysis_layercam["correct_opacity"] = (
    analysis_layercam["correct_opacity"]
    .astype(bool)
)

analysis_layercam["position"] = (
    analysis_layercam["position"]
    .astype(str)
    .str.upper()
    .str.strip()
)


def bootstrap_mean_difference(
    group_a,
    group_b,
    column,
    n_bootstrap=5000
):

    a = (
        group_a[column]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    b = (
        group_b[column]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    observed = (
        np.mean(a)
        - np.mean(b)
    )

    boot = np.empty(
        n_bootstrap
    )

    for i in range(
        n_bootstrap
    ):

        a_sample = rng.choice(
            a,
            size=len(a),
            replace=True
        )

        b_sample = rng.choice(
            b,
            size=len(b),
            replace=True
        )

        boot[i] = (
            np.mean(a_sample)
            - np.mean(b_sample)
        )

    ci_low = np.percentile(
        boot,
        2.5
    )

    ci_high = np.percentile(
        boot,
        97.5
    )

    return {
        "n_a": len(a),
        "n_b": len(b),
        "mean_a": np.mean(a),
        "mean_b": np.mean(b),
        "difference_a_minus_b": observed,
        "ci95_low": ci_low,
        "ci95_high": ci_high
    }


groups = {
    "correct_all": (
        analysis_layercam[
            analysis_layercam[
                "correct_opacity"
            ]
        ]
    ),

    "missed_all": (
        analysis_layercam[
            ~analysis_layercam[
                "correct_opacity"
            ]
        ]
    ),

    "AP_all": (
        analysis_layercam[
            analysis_layercam[
                "position"
            ] == "AP"
        ]
    ),

    "PA_all": (
        analysis_layercam[
            analysis_layercam[
                "position"
            ] == "PA"
        ]
    ),

    "AP_correct": (
        analysis_layercam[
            (
                analysis_layercam[
                    "position"
                ] == "AP"
            )
            &
            (
                analysis_layercam[
                    "correct_opacity"
                ]
            )
        ]
    ),

    "AP_missed": (
        analysis_layercam[
            (
                analysis_layercam[
                    "position"
                ] == "AP"
            )
            &
            (
                ~analysis_layercam[
                    "correct_opacity"
                ]
            )
        ]
    ),

    "PA_correct": (
        analysis_layercam[
            (
                analysis_layercam[
                    "position"
                ] == "PA"
            )
            &
            (
                analysis_layercam[
                    "correct_opacity"
                ]
            )
        ]
    ),

    "PA_missed": (
        analysis_layercam[
            (
                analysis_layercam[
                    "position"
                ] == "PA"
            )
            &
            (
                ~analysis_layercam[
                    "correct_opacity"
                ]
            )
        ]
    )
}


comparisons = [
    (
        "Correct vs Missed",
        "correct_all",
        "missed_all"
    ),

    (
        "AP vs PA",
        "AP_all",
        "PA_all"
    ),

    (
        "AP Correct vs AP Missed",
        "AP_correct",
        "AP_missed"
    ),

    (
        "PA Correct vs PA Missed",
        "PA_correct",
        "PA_missed"
    ),

    (
        "AP Correct vs PA Correct",
        "AP_correct",
        "PA_correct"
    ),

    (
        "AP Missed vs PA Missed",
        "AP_missed",
        "PA_missed"
    ),

    (
        "AP Correct vs PA Missed",
        "AP_correct",
        "PA_missed"
    )
]


metrics = [
    "bbox_energy_fraction",
    "pointing_game",
    "normalized_entropy",
    "energy_enrichment",
    "bbox_area_fraction"
]

bootstrap_rows = []

for comparison_name, a_name, b_name in comparisons:

    group_a = groups[a_name]
    group_b = groups[b_name]

    for metric in metrics:

        result = bootstrap_mean_difference(
            group_a,
            group_b,
            metric,
            n_bootstrap=BOOTSTRAP_N
        )

        bootstrap_rows.append(
            {
                "comparison":
                    comparison_name,

                "group_a":
                    a_name,

                "group_b":
                    b_name,

                "metric":
                    metric,

                **result
            }
        )


bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

bootstrap_df.to_csv(
    LAYERCAM_STATS_DIR
    / "layercam_bootstrap_group_differences.csv",
    index=False
)


# Focused table with the two most interpretable
# localization outcomes
focused = (
    bootstrap_df[
        bootstrap_df[
            "metric"
        ].isin(
            [
                "bbox_energy_fraction",
                "pointing_game"
            ]
        )
    ]
    .copy()
)

print(
    "LayerCAM bootstrap comparisons"
)
print(
    "95% percentile confidence intervals"
)
print(
    "---------------------------------------------"
)

for comparison_name in [
    x[0]
    for x in comparisons
]:

    print(
        f"\n{comparison_name}"
    )

    subset = focused[
        focused[
            "comparison"
        ] == comparison_name
    ]

    for _, row in subset.iterrows():

        print(
            f"{row['metric']:24s} "
            f"{row['mean_a']:.4f} vs "
            f"{row['mean_b']:.4f} | "
            f"diff={row['difference_a_minus_b']:+.4f} | "
            f"95% CI "
            f"[{row['ci95_low']:+.4f}, "
            f"{row['ci95_high']:+.4f}]"
        )


print(
    "\nBounding-box area differences"
)
print(
    "---------------------------------------------"
)

area_results = bootstrap_df[
    bootstrap_df[
        "metric"
    ] == "bbox_area_fraction"
]

for _, row in area_results.iterrows():

    print(
        f"{row['comparison']:28s} "
        f"diff="
        f"{row['difference_a_minus_b']:+.4f} | "
        f"95% CI "
        f"[{row['ci95_low']:+.4f}, "
        f"{row['ci95_high']:+.4f}]"
    )


print(
    "\nFull results saved to:"
)
print(
    LAYERCAM_STATS_DIR
    / "layercam_bootstrap_group_differences.csv"
)


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import json

ADJUSTED_XAI_DIR = (
    TEST_LAYERCAM_DIR
    / "statistical_analysis"
    / "adjusted_localization_model"
)

ADJUSTED_XAI_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# 1. Merge LayerCAM metrics with lesion burden metadata
# ---------------------------------------------------------

burden_df = pd.read_csv(
    LESION_BURDEN_DIR
    / "test_opacity_cases_with_lesion_burden.csv"
)

needed_columns = [
    "patientId",
    "bbox_count",
    "union_bbox_area_fraction",
    "age_clean",
    "sex",
    "position"
]

burden_subset = (
    burden_df[
        needed_columns
    ]
    .copy()
)

xai_adjusted_df = (
    test_layercam_df
    .merge(
        burden_subset,
        on="patientId",
        how="left",
        validate="one_to_one",
        suffixes=("", "_metadata")
    )
)

xai_adjusted_df["correct"] = (
    xai_adjusted_df["correct_opacity"]
    .astype(int)
)

xai_adjusted_df["PA"] = (
    xai_adjusted_df["position"]
    .astype(str)
    .str.upper()
    .str.strip()
    .eq("PA")
    .astype(int)
)

xai_adjusted_df["sex"] = (
    xai_adjusted_df["sex"]
    .astype(str)
    .str.upper()
    .str.strip()
)

xai_adjusted_df["age_clean"] = pd.to_numeric(
    xai_adjusted_df["age_clean"],
    errors="coerce"
)

# One unit = 10 percentage-point increase
# in attribution energy inside radiologist boxes
xai_adjusted_df[
    "bbox_energy_per_10pct"
] = (
    xai_adjusted_df[
        "bbox_energy_fraction"
    ]
    * 10.0
)


# ---------------------------------------------------------
# 2. Complete-case audit
# ---------------------------------------------------------

model_df = (
    xai_adjusted_df
    .dropna(
        subset=[
            "correct",
            "bbox_energy_per_10pct",
            "union_bbox_area_fraction",
            "bbox_count",
            "PA",
            "age_clean",
            "sex"
        ]
    )
    .copy()
)

print("Adjusted LayerCAM analysis")
print("--------------------------------------------")
print("Total opacity cases:", len(xai_adjusted_df))
print("Complete cases:", len(model_df))

print("\nOutcome:")
print(
    model_df["correct"]
    .value_counts()
    .sort_index()
)

print("\nPosition x outcome:")
print(
    pd.crosstab(
        model_df["PA"],
        model_df["correct"]
    )
)


# ---------------------------------------------------------
# 3. Logistic regression
#
# Flexible nonlinear adjustment for lesion area
# using a restricted spline basis.
# ---------------------------------------------------------

formula = (
    "correct ~ "
    "bbox_energy_per_10pct + "
    "PA + "
    "bs(union_bbox_area_fraction, df=4, degree=3) + "
    "bbox_count + "
    "age_clean + "
    "C(sex)"
)

localization_model = smf.logit(
    formula=formula,
    data=model_df
).fit(
    disp=False,
    cov_type="HC3"
)

print("\nAdjusted logistic regression")
print("--------------------------------------------")
print(
    localization_model.summary()
)


# ---------------------------------------------------------
# 4. Extract LayerCAM association
# ---------------------------------------------------------

beta = float(
    localization_model.params[
        "bbox_energy_per_10pct"
    ]
)

se = float(
    localization_model.bse[
        "bbox_energy_per_10pct"
    ]
)

p_value = float(
    localization_model.pvalues[
        "bbox_energy_per_10pct"
    ]
)

ci = (
    localization_model
    .conf_int()
    .loc[
        "bbox_energy_per_10pct"
    ]
)

odds_ratio = float(
    np.exp(beta)
)

or_low = float(
    np.exp(
        ci.iloc[0]
    )
)

or_high = float(
    np.exp(
        ci.iloc[1]
    )
)

print("\nPrimary adjusted XAI result")
print("--------------------------------------------")

print(
    "Effect scale: each +10 percentage-point "
    "increase in LayerCAM bbox energy"
)

print(
    f"Log-odds coefficient: "
    f"{beta:+.4f}"
)

print(
    f"Odds ratio:           "
    f"{odds_ratio:.4f}"
)

print(
    f"95% CI for OR:        "
    f"[{or_low:.4f}, {or_high:.4f}]"
)

print(
    f"p-value:              "
    f"{p_value:.6g}"
)


# ---------------------------------------------------------
# 5. Position effect after localization adjustment
# ---------------------------------------------------------

pa_beta = float(
    localization_model.params["PA"]
)

pa_ci = (
    localization_model
    .conf_int()
    .loc["PA"]
)

pa_or = float(
    np.exp(pa_beta)
)

pa_or_low = float(
    np.exp(
        pa_ci.iloc[0]
    )
)

pa_or_high = float(
    np.exp(
        pa_ci.iloc[1]
    )
)

pa_p = float(
    localization_model.pvalues["PA"]
)

print("\nAdjusted position effect")
print("--------------------------------------------")

print(
    f"PA odds ratio:        "
    f"{pa_or:.4f}"
)

print(
    f"95% CI:               "
    f"[{pa_or_low:.4f}, "
    f"{pa_or_high:.4f}]"
)

print(
    f"p-value:              "
    f"{pa_p:.6g}"
)


# ---------------------------------------------------------
# 6. Predicted probabilities at representative localization
# ---------------------------------------------------------

energy_values = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70
]

typical_burden = float(
    model_df[
        "union_bbox_area_fraction"
    ].median()
)

typical_bbox_count = int(
    model_df[
        "bbox_count"
    ].median()
)

typical_age = float(
    model_df[
        "age_clean"
    ].median()
)

reference_sex = (
    model_df["sex"]
    .mode()
    .iloc[0]
)

prediction_rows = []

for position_name, pa_value in [
    ("AP", 0),
    ("PA", 1)
]:

    for energy in energy_values:

        prediction_rows.append(
            {
                "bbox_energy_per_10pct":
                    energy * 10.0,

                "PA":
                    pa_value,

                "union_bbox_area_fraction":
                    typical_burden,

                "bbox_count":
                    typical_bbox_count,

                "age_clean":
                    typical_age,

                "sex":
                    reference_sex,

                "position_label":
                    position_name,

                "bbox_energy_fraction":
                    energy
            }
        )

prediction_df = pd.DataFrame(
    prediction_rows
)

prediction_df[
    "predicted_probability_correct"
] = localization_model.predict(
    prediction_df
)

print(
    "\nAdjusted predicted probability of correct "
    "Lung Opacity classification"
)
print("--------------------------------------------")

print(
    prediction_df[
        [
            "position_label",
            "bbox_energy_fraction",
            "predicted_probability_correct"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ---------------------------------------------------------
# 7. Save
# ---------------------------------------------------------

summary = {
    "n": int(
        len(model_df)
    ),

    "outcome":
        "correct Lung Opacity classification",

    "primary_predictor":
        "LayerCAM bbox energy fraction",

    "effect_unit":
        "10 percentage-point increase",

    "adjustments": [
        "AP/PA position",
        "nonlinear union bbox area fraction",
        "bbox count",
        "age",
        "sex"
    ],

    "localization_beta":
        beta,

    "localization_odds_ratio":
        odds_ratio,

    "localization_or_ci95_lower":
        or_low,

    "localization_or_ci95_upper":
        or_high,

    "localization_p_value":
        p_value,

    "pa_odds_ratio":
        pa_or,

    "pa_or_ci95_lower":
        pa_or_low,

    "pa_or_ci95_upper":
        pa_or_high,

    "pa_p_value":
        pa_p
}

with open(
    ADJUSTED_XAI_DIR
    / "adjusted_layercam_localization_model.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

with open(
    ADJUSTED_XAI_DIR
    / "adjusted_layercam_localization_model.txt",
    "w"
) as f:

    f.write(
        localization_model.summary().as_text()
    )

prediction_df.to_csv(
    ADJUSTED_XAI_DIR
    / "adjusted_correctness_probability_examples.csv",
    index=False
)

xai_adjusted_df.to_csv(
    ADJUSTED_XAI_DIR
    / "layercam_test_cases_adjusted_dataset.csv",
    index=False
)

print("\nResults saved to:")
print(ADJUSTED_XAI_DIR)


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

TEST_IG_DIR = (
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "xai"
    / "integrated_gradients_256"
)

TEST_IG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

IG_TEST_CHECKPOINT = (
    TEST_IG_DIR
    / "test_ig256_case_metrics_checkpoint.csv"
)

IG_TEST_FINAL = (
    TEST_IG_DIR
    / "test_ig256_case_metrics.csv"
)

IG_STEPS_LOCKED = 256
TARGET_CLASS = 2


# ---------------------------------------------------------
# Resume support
# ---------------------------------------------------------

if IG_TEST_CHECKPOINT.exists():

    completed_ig_df = pd.read_csv(
        IG_TEST_CHECKPOINT
    )

    completed_ids = set(
        completed_ig_df["patientId"]
    )

    ig_test_results = (
        completed_ig_df
        .to_dict("records")
    )

    print(
        "Resuming IG test analysis"
    )

    print(
        "Already completed:",
        len(completed_ids)
    )

else:

    completed_ids = set()
    ig_test_results = []

    print(
        "Starting IG test analysis from beginning."
    )


remaining_ig_df = (
    test_opacity_xai[
        ~test_opacity_xai[
            "patientId"
        ].isin(
            completed_ids
        )
    ]
    .reset_index(drop=True)
)

print(
    "Remaining cases:",
    len(remaining_ig_df)
)


# ---------------------------------------------------------
# Run locked IG protocol
# ---------------------------------------------------------

for row_index, row in (
    remaining_ig_df
    .iterrows()
):

    image_tensor, _ = load_test_image(
        tf.constant(
            row["image_relpath"]
        ),
        tf.constant(
            TARGET_CLASS
        )
    )

    (
        ig_map,
        true_delta,
        ig_sum,
        abs_error,
        rel_error
    ) = integrated_gradients_audit(
        image_tensor,
        steps=IG_STEPS_LOCKED
    )

    bbox_mask = make_bbox_union_mask(
        row["patientId"],
        size=224
    )

    metrics = heatmap_metrics(
        ig_map,
        bbox_mask
    )

    ig_test_results.append(
        {
            "patientId":
                row["patientId"],

            "position":
                row["position"],

            "predicted_class_id":
                int(
                    row[
                        "predicted_class_id"
                    ]
                ),

            "correct_opacity":
                bool(
                    row[
                        "correct_opacity"
                    ]
                ),

            "prob_lung_opacity":
                float(
                    row[
                        "prob_lung_opacity"
                    ]
                ),

            "method":
                "Integrated Gradients",

            "steps":
                IG_STEPS_LOCKED,

            "true_logit_delta":
                true_delta,

            "integrated_attribution_sum":
                ig_sum,

            "absolute_completeness_error":
                abs_error,

            "relative_completeness_error":
                rel_error,

            **metrics
        }
    )

    total_completed = len(
        ig_test_results
    )

    if (
        total_completed % 25
        == 0
    ):

        pd.DataFrame(
            ig_test_results
        ).to_csv(
            IG_TEST_CHECKPOINT,
            index=False
        )

        print(
            "Completed",
            total_completed,
            "of",
            len(test_opacity_xai)
        )


# ---------------------------------------------------------
# Final save
# ---------------------------------------------------------

test_ig256_df = pd.DataFrame(
    ig_test_results
)

test_ig256_df = (
    test_ig256_df
    .drop_duplicates(
        subset="patientId",
        keep="last"
    )
)

test_ig256_df.to_csv(
    IG_TEST_FINAL,
    index=False
)

test_ig256_df.to_csv(
    IG_TEST_CHECKPOINT,
    index=False
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("\nIG 256 locked-test analysis complete")
print("--------------------------------------------")

print(
    "Cases:",
    len(test_ig256_df)
)

print(
    "Mean bbox energy fraction:",
    f"{test_ig256_df['bbox_energy_fraction'].mean():.4f}"
)

print(
    "Mean energy enrichment:",
    f"{test_ig256_df['energy_enrichment'].mean():.4f}"
)

print(
    "Median energy enrichment:",
    f"{test_ig256_df['energy_enrichment'].median():.4f}"
)

print(
    "Pointing game accuracy:",
    f"{test_ig256_df['pointing_game'].mean():.4f}"
)

print(
    "Mean normalized entropy:",
    f"{test_ig256_df['normalized_entropy'].mean():.4f}"
)

print("\nCompleteness")
print("--------------------------------------------")

print(
    "Mean absolute error:",
    f"{test_ig256_df['absolute_completeness_error'].mean():.4f}"
)

print(
    "Median absolute error:",
    f"{test_ig256_df['absolute_completeness_error'].median():.4f}"
)

print(
    "Median relative error:",
    f"{test_ig256_df['relative_completeness_error'].median():.4f}"
)


ig_group_summary = (
    test_ig256_df
    .groupby(
        [
            "position",
            "correct_opacity"
        ]
    )
    .agg(
        n=(
            "patientId",
            "count"
        ),

        mean_bbox_energy_fraction=(
            "bbox_energy_fraction",
            "mean"
        ),

        mean_energy_enrichment=(
            "energy_enrichment",
            "mean"
        ),

        median_energy_enrichment=(
            "energy_enrichment",
            "median"
        ),

        pointing_game_accuracy=(
            "pointing_game",
            "mean"
        ),

        mean_normalized_entropy=(
            "normalized_entropy",
            "mean"
        ),

        mean_relative_completeness_error=(
            "relative_completeness_error",
            "mean"
        )
    )
    .reset_index()
)

ig_group_summary.to_csv(
    TEST_IG_DIR
    / "test_ig256_position_correctness_summary.csv",
    index=False
)

print("\nPosition x correctness")
print("--------------------------------------------")

print(
    ig_group_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nResults saved to:")
print(IG_TEST_FINAL)


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import json

IG_ADJUSTED_DIR = (
    TEST_IG_DIR
    / "statistical_analysis"
    / "adjusted_localization_model"
)

IG_ADJUSTED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------------
# 1. Merge IG metrics with lesion burden / metadata
# ---------------------------------------------------------

burden_df = pd.read_csv(
    LESION_BURDEN_DIR
    / "test_opacity_cases_with_lesion_burden.csv"
)

needed_columns = [
    "patientId",
    "bbox_count",
    "union_bbox_area_fraction",
    "age_clean",
    "sex",
    "position"
]

burden_subset = (
    burden_df[
        needed_columns
    ]
    .copy()
)

ig_adjusted_df = (
    test_ig256_df
    .merge(
        burden_subset,
        on="patientId",
        how="left",
        validate="one_to_one",
        suffixes=("", "_metadata")
    )
)

ig_adjusted_df["correct"] = (
    ig_adjusted_df[
        "correct_opacity"
    ]
    .astype(int)
)

ig_adjusted_df["PA"] = (
    ig_adjusted_df["position"]
    .astype(str)
    .str.upper()
    .str.strip()
    .eq("PA")
    .astype(int)
)

ig_adjusted_df["sex"] = (
    ig_adjusted_df["sex"]
    .astype(str)
    .str.upper()
    .str.strip()
)

ig_adjusted_df["age_clean"] = pd.to_numeric(
    ig_adjusted_df["age_clean"],
    errors="coerce"
)

# One unit = +10 percentage points
# attribution energy inside radiologist boxes
ig_adjusted_df[
    "bbox_energy_per_10pct"
] = (
    ig_adjusted_df[
        "bbox_energy_fraction"
    ]
    * 10.0
)

model_df_ig = (
    ig_adjusted_df
    .dropna(
        subset=[
            "correct",
            "bbox_energy_per_10pct",
            "union_bbox_area_fraction",
            "bbox_count",
            "PA",
            "age_clean",
            "sex"
        ]
    )
    .copy()
)

print("Adjusted Integrated Gradients analysis")
print("--------------------------------------------")
print("Total opacity cases:", len(ig_adjusted_df))
print("Complete cases:", len(model_df_ig))


# ---------------------------------------------------------
# 2. Bootstrap correct vs missed differences
# ---------------------------------------------------------

rng = np.random.default_rng(42)
N_BOOTSTRAP = 5000

def bootstrap_difference(a, b):

    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    observed = (
        np.mean(a)
        - np.mean(b)
    )

    boot = np.empty(
        N_BOOTSTRAP
    )

    for i in range(
        N_BOOTSTRAP
    ):

        a_sample = rng.choice(
            a,
            size=len(a),
            replace=True
        )

        b_sample = rng.choice(
            b,
            size=len(b),
            replace=True
        )

        boot[i] = (
            np.mean(a_sample)
            - np.mean(b_sample)
        )

    return (
        observed,
        np.percentile(
            boot,
            2.5
        ),
        np.percentile(
            boot,
            97.5
        )
    )


correct_ig = model_df_ig[
    model_df_ig["correct"] == 1
]

missed_ig = model_df_ig[
    model_df_ig["correct"] == 0
]

energy_diff = bootstrap_difference(
    correct_ig[
        "bbox_energy_fraction"
    ],
    missed_ig[
        "bbox_energy_fraction"
    ]
)

point_diff = bootstrap_difference(
    correct_ig[
        "pointing_game"
    ],
    missed_ig[
        "pointing_game"
    ]
)

print("\nBootstrap correct vs missed")
print("--------------------------------------------")

print(
    "bbox energy:"
)

print(
    f"{correct_ig['bbox_energy_fraction'].mean():.4f} "
    f"vs "
    f"{missed_ig['bbox_energy_fraction'].mean():.4f}"
)

print(
    f"Difference: {energy_diff[0]:+.4f}"
)

print(
    f"95% CI: "
    f"[{energy_diff[1]:+.4f}, "
    f"{energy_diff[2]:+.4f}]"
)

print("\npointing game:")

print(
    f"{correct_ig['pointing_game'].mean():.4f} "
    f"vs "
    f"{missed_ig['pointing_game'].mean():.4f}"
)

print(
    f"Difference: {point_diff[0]:+.4f}"
)

print(
    f"95% CI: "
    f"[{point_diff[1]:+.4f}, "
    f"{point_diff[2]:+.4f}]"
)


# ---------------------------------------------------------
# 3. Adjusted logistic regression
# ---------------------------------------------------------

formula = (
    "correct ~ "
    "bbox_energy_per_10pct + "
    "PA + "
    "bs(union_bbox_area_fraction, df=4, degree=3) + "
    "bbox_count + "
    "age_clean + "
    "C(sex)"
)

ig_localization_model = smf.logit(
    formula=formula,
    data=model_df_ig
).fit(
    disp=False,
    cov_type="HC3"
)

print("\nAdjusted logistic regression")
print("--------------------------------------------")

print(
    ig_localization_model.summary()
)


# ---------------------------------------------------------
# 4. Extract primary IG localization effect
# ---------------------------------------------------------

beta = float(
    ig_localization_model.params[
        "bbox_energy_per_10pct"
    ]
)

ci = (
    ig_localization_model
    .conf_int()
    .loc[
        "bbox_energy_per_10pct"
    ]
)

p_value = float(
    ig_localization_model.pvalues[
        "bbox_energy_per_10pct"
    ]
)

odds_ratio = float(
    np.exp(beta)
)

or_low = float(
    np.exp(
        ci.iloc[0]
    )
)

or_high = float(
    np.exp(
        ci.iloc[1]
    )
)

print("\nPrimary adjusted IG result")
print("--------------------------------------------")

print(
    "Effect scale: each +10 percentage-point "
    "increase in IG bbox energy"
)

print(
    f"Odds ratio:     {odds_ratio:.4f}"
)

print(
    f"95% CI:         "
    f"[{or_low:.4f}, {or_high:.4f}]"
)

print(
    f"p-value:        {p_value:.6g}"
)


# ---------------------------------------------------------
# 5. Adjusted PA effect
# ---------------------------------------------------------

pa_beta = float(
    ig_localization_model.params[
        "PA"
    ]
)

pa_ci = (
    ig_localization_model
    .conf_int()
    .loc["PA"]
)

pa_p = float(
    ig_localization_model.pvalues[
        "PA"
    ]
)

pa_or = float(
    np.exp(
        pa_beta
    )
)

pa_or_low = float(
    np.exp(
        pa_ci.iloc[0]
    )
)

pa_or_high = float(
    np.exp(
        pa_ci.iloc[1]
    )
)

print("\nAdjusted position effect")
print("--------------------------------------------")

print(
    f"PA odds ratio:  {pa_or:.4f}"
)

print(
    f"95% CI:         "
    f"[{pa_or_low:.4f}, "
    f"{pa_or_high:.4f}]"
)

print(
    f"p-value:        {pa_p:.6g}"
)


# ---------------------------------------------------------
# 6. Save
# ---------------------------------------------------------

summary = {
    "n": int(
        len(model_df_ig)
    ),

    "method":
        "Integrated Gradients 256",

    "effect_unit":
        "10 percentage-point increase in bbox energy",

    "bootstrap_correct_minus_missed_bbox_energy":
        float(
            energy_diff[0]
        ),

    "bootstrap_bbox_energy_ci95_lower":
        float(
            energy_diff[1]
        ),

    "bootstrap_bbox_energy_ci95_upper":
        float(
            energy_diff[2]
        ),

    "bootstrap_correct_minus_missed_pointing_game":
        float(
            point_diff[0]
        ),

    "ig_localization_odds_ratio":
        odds_ratio,

    "ig_localization_or_ci95_lower":
        or_low,

    "ig_localization_or_ci95_upper":
        or_high,

    "ig_localization_p_value":
        p_value,

    "pa_odds_ratio":
        pa_or,

    "pa_or_ci95_lower":
        pa_or_low,

    "pa_or_ci95_upper":
        pa_or_high,

    "pa_p_value":
        pa_p
}

with open(
    IG_ADJUSTED_DIR
    / "adjusted_ig_localization_model.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

with open(
    IG_ADJUSTED_DIR
    / "adjusted_ig_localization_model.txt",
    "w"
) as f:

    f.write(
        ig_localization_model
        .summary()
        .as_text()
    )

ig_adjusted_df.to_csv(
    IG_ADJUSTED_DIR
    / "ig_test_cases_adjusted_dataset.csv",
    index=False
)

print("\nResults saved to:")
print(IG_ADJUSTED_DIR)


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

ROBUST_XAI_DIR = (
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "xai"
    / "cross_method_area_adjusted_robustness"
)

ROBUST_XAI_DIR.mkdir(
    parents=True,
    exist_ok=True
)

burden_source = pd.read_csv(
    LESION_BURDEN_DIR
    / "test_opacity_cases_with_lesion_burden.csv"
)

metadata_cols = [
    "patientId",
    "bbox_count",
    "union_bbox_area_fraction",
    "age_clean",
    "sex"
]

metadata_xai = burden_source[
    metadata_cols
].copy()


def prepare_method_df(
    source_df,
    method_name
):

    df = source_df.copy()

    df = df.merge(
        metadata_xai,
        on="patientId",
        how="left",
        validate="one_to_one"
    )

    if df["correct_opacity"].dtype == object:

        df["correct"] = (
            df["correct_opacity"]
            .astype(str)
            .str.lower()
            .map(
                {
                    "true": 1,
                    "false": 0
                }
            )
        )

    else:

        df["correct"] = (
            df["correct_opacity"]
            .astype(int)
        )

    df["position"] = (
        df["position"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df["PA"] = (
        df["position"] == "PA"
    ).astype(int)

    df["sex"] = (
        df["sex"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df["age_clean"] = pd.to_numeric(
        df["age_clean"],
        errors="coerce"
    )

    # Raw energy: +10 percentage points
    df[
        "raw_energy_per_10pct"
    ] = (
        df["bbox_energy_fraction"]
        * 10.0
    )

    # Area-normalized excess energy
    # 0 = exactly proportional to box area
    df[
        "excess_energy"
    ] = (
        df["bbox_energy_fraction"]
        - df["bbox_area_fraction"]
    )

    df[
        "excess_energy_per_10pct"
    ] = (
        df["excess_energy"]
        * 10.0
    )

    # 0 means enrichment = 1
    # +1 means twice expected concentration
    # -1 means half expected concentration
    enrichment_safe = np.clip(
        df["energy_enrichment"]
        .astype(float),
        1e-8,
        None
    )

    df[
        "log2_energy_enrichment"
    ] = np.log2(
        enrichment_safe
    )

    df["method_name"] = (
        method_name
    )

    return df


layercam_robust = prepare_method_df(
    test_layercam_df,
    "LayerCAM"
)

ig_robust = prepare_method_df(
    test_ig256_df,
    "Integrated Gradients 256"
)


# ---------------------------------------------------------
# Descriptive diagnostics
# ---------------------------------------------------------

print("Area dependence diagnostics")
print("--------------------------------------------")

for name, df in [
    ("LayerCAM", layercam_robust),
    ("Integrated Gradients 256", ig_robust)
]:

    corr = df[
        [
            "bbox_energy_fraction",
            "bbox_area_fraction"
        ]
    ].corr().iloc[0, 1]

    print(
        f"{name}: corr("
        "bbox energy, bbox area) = "
        f"{corr:.4f}"
    )


print("\nCorrect vs missed descriptive comparison")
print("--------------------------------------------")

for name, df in [
    ("LayerCAM", layercam_robust),
    ("Integrated Gradients 256", ig_robust)
]:

    summary = (
        df
        .groupby("correct")
        .agg(
            n=(
                "patientId",
                "count"
            ),
            bbox_area=(
                "bbox_area_fraction",
                "mean"
            ),
            raw_energy=(
                "bbox_energy_fraction",
                "mean"
            ),
            excess_energy=(
                "excess_energy",
                "mean"
            ),
            enrichment=(
                "energy_enrichment",
                "mean"
            ),
            pointing_game=(
                "pointing_game",
                "mean"
            )
        )
    )

    print(f"\n{name}")
    print(summary)


# ---------------------------------------------------------
# Adjusted robustness models
# ---------------------------------------------------------

base_adjustment = (
    "PA + "
    "bs(union_bbox_area_fraction, df=4, degree=3) + "
    "bbox_count + "
    "age_clean + "
    "C(sex)"
)

predictors = {
    "Raw bbox energy (+10 pct points)":
        "raw_energy_per_10pct",

    "Excess bbox energy (+10 pct points)":
        "excess_energy_per_10pct",

    "Energy enrichment (per doubling)":
        "log2_energy_enrichment",

    "Pointing game (peak inside box)":
        "pointing_game"
}

result_rows = []

for method_name, df in [
    ("LayerCAM", layercam_robust),
    ("Integrated Gradients 256", ig_robust)
]:

    model_df = df.dropna(
        subset=[
            "correct",
            "PA",
            "union_bbox_area_fraction",
            "bbox_count",
            "age_clean",
            "sex",
            "bbox_energy_fraction",
            "bbox_area_fraction",
            "energy_enrichment",
            "pointing_game"
        ]
    ).copy()

    for metric_label, predictor in (
        predictors.items()
    ):

        formula = (
            f"correct ~ {predictor} + "
            f"{base_adjustment}"
        )

        model = smf.logit(
            formula=formula,
            data=model_df
        ).fit(
            disp=False,
            cov_type="HC3"
        )

        beta = float(
            model.params[predictor]
        )

        ci = (
            model.conf_int()
            .loc[predictor]
        )

        p_value = float(
            model.pvalues[predictor]
        )

        odds_ratio = float(
            np.exp(beta)
        )

        ci_low = float(
            np.exp(
                ci.iloc[0]
            )
        )

        ci_high = float(
            np.exp(
                ci.iloc[1]
            )
        )

        result_rows.append(
            {
                "method":
                    method_name,

                "metric":
                    metric_label,

                "predictor":
                    predictor,

                "n":
                    len(model_df),

                "odds_ratio":
                    odds_ratio,

                "ci95_low":
                    ci_low,

                "ci95_high":
                    ci_high,

                "p_value":
                    p_value,

                "pa_odds_ratio":
                    float(
                        np.exp(
                            model.params["PA"]
                        )
                    ),

                "pseudo_r2":
                    float(
                        model.prsquared
                    )
            }
        )


robust_results = pd.DataFrame(
    result_rows
)

robust_results.to_csv(
    ROBUST_XAI_DIR
    / "area_adjusted_cross_method_models.csv",
    index=False
)

print("\nArea-adjusted localization robustness")
print("--------------------------------------------")

for method_name in (
    robust_results["method"].unique()
):

    print(f"\n{method_name}")

    subset = robust_results[
        robust_results["method"]
        == method_name
    ]

    for _, row in subset.iterrows():

        print(
            f"{row['metric']:38s} "
            f"OR={row['odds_ratio']:.3f} "
            f"95% CI "
            f"[{row['ci95_low']:.3f}, "
            f"{row['ci95_high']:.3f}] "
            f"p={row['p_value']:.3g}"
        )


print("\nPA effect in each model")
print("--------------------------------------------")

print(
    robust_results[
        [
            "method",
            "metric",
            "pa_odds_ratio"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nResults saved to:")
print(ROBUST_XAI_DIR)


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from matplotlib.patches import Rectangle


QUALITATIVE_FIG_DIR = (
    QUALITATIVE_DIR
    / "figures"
)

QUALITATIVE_MAP_DIR = (
    QUALITATIVE_DIR
    / "maps"
)

QUALITATIVE_FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

QUALITATIVE_MAP_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# Add image paths
# ---------------------------------------------------------

selected_plot_df = (
    selected_cases
    .merge(
        test_opacity_xai[
            [
                "patientId",
                "image_relpath"
            ]
        ],
        on="patientId",
        how="left",
        validate="one_to_one"
    )
)

selected_plot_df[
    "selection_group"
] = pd.Categorical(
    selected_plot_df[
        "selection_group"
    ],
    categories=[
        "AP_correct",
        "AP_missed",
        "PA_correct",
        "PA_missed"
    ],
    ordered=True
)

selected_plot_df = (
    selected_plot_df
    .sort_values(
        [
            "selection_group",
            "selection_quantile"
        ]
    )
)


# ---------------------------------------------------------
# Locked LayerCAM map
# ---------------------------------------------------------

def generate_locked_layercam(
    image_tensor
):

    seed_input = np.expand_dims(
        image_tensor.numpy(),
        axis=0
    )

    cam = test_layercam(
        test_layercam_score,
        seed_input,
        penultimate_layer="multiscale_fusion",
        seek_penultimate_conv_layer=False,
        expand_cam=True,
        normalize_cam=True
    )

    return np.asarray(
        cam[0],
        dtype=np.float32
    )


# ---------------------------------------------------------
# Locked IG-256 map
# ---------------------------------------------------------

def generate_locked_ig(
    image_tensor
):

    (
        ig_map,
        true_delta,
        ig_sum,
        abs_error,
        rel_error
    ) = integrated_gradients_audit(
        image_tensor,
        steps=256
    )

    return (
        ig_map,
        abs_error,
        rel_error
    )


# ---------------------------------------------------------
# Helper to draw radiologist boxes
# ---------------------------------------------------------

def add_boxes(
    axis,
    patient_id,
    scale_x=1.0,
    scale_y=1.0
):

    boxes = bbox_df[
        bbox_df["patientId"]
        == patient_id
    ]

    for _, box in boxes.iterrows():

        rectangle = Rectangle(
            (
                box["x"]
                * scale_x,

                box["y"]
                * scale_y
            ),

            box["width"]
            * scale_x,

            box["height"]
            * scale_y,

            fill=False,
            linewidth=2
        )

        axis.add_patch(
            rectangle
        )


# ---------------------------------------------------------
# Generate maps for all 12 selected cases
# ---------------------------------------------------------

map_rows = []

print("Generating locked XAI maps for selected cases")
print("--------------------------------------------")

for index, row in (
    selected_plot_df
    .reset_index(drop=True)
    .iterrows()
):

    patient_id = row["patientId"]

    image_tensor, _ = load_test_image(
        tf.constant(
            row["image_relpath"]
        ),
        tf.constant(2)
    )

    layercam_map = generate_locked_layercam(
        image_tensor
    )

    (
        ig_map,
        ig_abs_error,
        ig_rel_error
    ) = generate_locked_ig(
        image_tensor
    )

    np.save(
        QUALITATIVE_MAP_DIR
        / f"{patient_id}_layercam.npy",
        layercam_map
    )

    np.save(
        QUALITATIVE_MAP_DIR
        / f"{patient_id}_ig256.npy",
        ig_map
    )

    map_rows.append(
        {
            "patientId":
                patient_id,

            "selection_group":
                row["selection_group"],

            "selection_quantile":
                row["selection_quantile"],

            "ig_absolute_completeness_error":
                ig_abs_error,

            "ig_relative_completeness_error":
                ig_rel_error
        }
    )

    print(
        "Generated",
        index + 1,
        "of",
        len(selected_plot_df)
    )


pd.DataFrame(
    map_rows
).to_csv(
    QUALITATIVE_DIR
    / "selected_case_xai_map_diagnostics.csv",
    index=False
)


# ---------------------------------------------------------
# Create one figure per AP/PA x correct/missed group
# ---------------------------------------------------------

group_labels = {
    "AP_correct":
        "AP - Correct Lung Opacity",

    "AP_missed":
        "AP - Missed Lung Opacity",

    "PA_correct":
        "PA - Correct Lung Opacity",

    "PA_missed":
        "PA - Missed Lung Opacity"
}


for group_name in [
    "AP_correct",
    "AP_missed",
    "PA_correct",
    "PA_missed"
]:

    group_df = (
        selected_plot_df[
            selected_plot_df[
                "selection_group"
            ] == group_name
        ]
        .sort_values(
            "selection_quantile"
        )
        .reset_index(drop=True)
    )

    fig, axes = plt.subplots(
        3,
        3,
        figsize=(13, 15)
    )

    for row_index, row in (
        group_df.iterrows()
    ):

        patient_id = row[
            "patientId"
        ]

        original_path = (
            Path(ACTIVE_DATASET)
            / row["image_relpath"]
        )

        original_image = np.array(
            Image.open(
                original_path
            ).convert("L")
        )

        layercam_map = np.load(
            QUALITATIVE_MAP_DIR
            / f"{patient_id}_layercam.npy"
        )

        ig_map = np.load(
            QUALITATIVE_MAP_DIR
            / f"{patient_id}_ig256.npy"
        )

        layercam_original = tf.image.resize(
            layercam_map[
                ...,
                np.newaxis
            ],
            original_image.shape,
            method="bilinear"
        ).numpy()[..., 0]

        ig_original = tf.image.resize(
            ig_map[
                ...,
                np.newaxis
            ],
            original_image.shape,
            method="bilinear"
        ).numpy()[..., 0]

        quantile_label = {
            0.25: "Low burden",
            0.50: "Median burden",
            0.75: "High burden"
        }.get(
            float(
                row[
                    "selection_quantile"
                ]
            ),
            "Selected case"
        )

        # -----------------------------------------
        # Original + boxes
        # -----------------------------------------

        ax = axes[
            row_index,
            0
        ]

        ax.imshow(
            original_image,
            cmap="gray"
        )

        add_boxes(
            ax,
            patient_id
        )

        ax.set_title(
            (
                f"{quantile_label}\n"
                f"P(opacity)="
                f"{row['prob_lung_opacity']:.3f} | "
                f"burden="
                f"{row['union_bbox_area_fraction']:.3f}"
            ),
            fontsize=10
        )

        ax.axis("off")


        # -----------------------------------------
        # LayerCAM
        # -----------------------------------------

        ax = axes[
            row_index,
            1
        ]

        ax.imshow(
            original_image,
            cmap="gray"
        )

        ax.imshow(
            layercam_original,
            alpha=0.42
        )

        add_boxes(
            ax,
            patient_id
        )

        ax.set_title(
            (
                "LayerCAM\n"
                f"box energy="
                f"{row['layercam_bbox_energy']:.3f} | "
                f"point="
                f"{int(row['layercam_pointing'])}"
            ),
            fontsize=10
        )

        ax.axis("off")


        # -----------------------------------------
        # Integrated Gradients
        # -----------------------------------------

        ax = axes[
            row_index,
            2
        ]

        ax.imshow(
            original_image,
            cmap="gray"
        )

        ax.imshow(
            ig_original,
            alpha=0.42
        )

        add_boxes(
            ax,
            patient_id
        )

        ax.set_title(
            (
                "Integrated Gradients 256\n"
                f"box energy="
                f"{row['ig_bbox_energy']:.3f} | "
                f"point="
                f"{int(row['ig_pointing'])}"
            ),
            fontsize=10
        )

        ax.axis("off")


    fig.suptitle(
        group_labels[
            group_name
        ],
        fontsize=16
    )

    plt.tight_layout(
        rect=[
            0,
            0,
            1,
            0.97
        ]
    )

    figure_path = (
        QUALITATIVE_FIG_DIR
        / f"{group_name}_xai_comparison.png"
    )

    plt.savefig(
        figure_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

    print(
        "Saved:",
        figure_path
    )


print("\nAll qualitative figures saved to:")
print(QUALITATIVE_FIG_DIR)


In [ ]:
import numpy as np
import pandas as pd

STRATIFIED_XAI_DIR = (
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "xai"
    / "lesion_burden_stratified_analysis"
)

STRATIFIED_XAI_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def prepare_stratified_df(
    source_df,
    method_name
):

    df = source_df.copy()

    df = df.merge(
        metadata_xai,
        on="patientId",
        how="left",
        validate="one_to_one"
    )

    if df["correct_opacity"].dtype == object:

        df["correct"] = (
            df["correct_opacity"]
            .astype(str)
            .str.lower()
            .map(
                {
                    "true": 1,
                    "false": 0
                }
            )
        )

    else:

        df["correct"] = (
            df["correct_opacity"]
            .astype(int)
        )

    df["position"] = (
        df["position"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df["method"] = method_name

    df[
        "log2_enrichment"
    ] = np.log2(
        np.clip(
            df["energy_enrichment"],
            1e-8,
            None
        )
    )

    return df


layer_strat = prepare_stratified_df(
    test_layercam_df,
    "LayerCAM"
)

ig_strat = prepare_stratified_df(
    test_ig256_df,
    "Integrated Gradients 256"
)


# Use the SAME lesion-burden cut points for both methods
combined_burden = layer_strat[
    "union_bbox_area_fraction"
]

quantile_edges = np.unique(
    np.quantile(
        combined_burden,
        [
            0.0,
            0.2,
            0.4,
            0.6,
            0.8,
            1.0
        ]
    )
)

print("Lesion-burden quintile edges")
print("--------------------------------------------")
print(
    np.round(
        quantile_edges,
        4
    )
)


def add_burden_strata(df):

    df = df.copy()

    df["burden_stratum"] = pd.cut(
        df[
            "union_bbox_area_fraction"
        ],
        bins=quantile_edges,
        include_lowest=True,
        duplicates="drop"
    )

    return df


layer_strat = add_burden_strata(
    layer_strat
)

ig_strat = add_burden_strata(
    ig_strat
)


def stratified_summary(
    df,
    method_name
):

    rows = []

    for stratum, group in df.groupby(
        "burden_stratum",
        observed=True
    ):

        correct = group[
            group["correct"] == 1
        ]

        missed = group[
            group["correct"] == 0
        ]

        if (
            len(correct) == 0
            or len(missed) == 0
        ):
            continue

        rows.append(
            {
                "method":
                    method_name,

                "burden_stratum":
                    str(stratum),

                "n":
                    len(group),

                "n_correct":
                    len(correct),

                "n_missed":
                    len(missed),

                "mean_burden_correct":
                    correct[
                        "union_bbox_area_fraction"
                    ].mean(),

                "mean_burden_missed":
                    missed[
                        "union_bbox_area_fraction"
                    ].mean(),

                "bbox_energy_correct":
                    correct[
                        "bbox_energy_fraction"
                    ].mean(),

                "bbox_energy_missed":
                    missed[
                        "bbox_energy_fraction"
                    ].mean(),

                "bbox_energy_difference":
                    correct[
                        "bbox_energy_fraction"
                    ].mean()
                    -
                    missed[
                        "bbox_energy_fraction"
                    ].mean(),

                "log2_enrichment_correct":
                    correct[
                        "log2_enrichment"
                    ].mean(),

                "log2_enrichment_missed":
                    missed[
                        "log2_enrichment"
                    ].mean(),

                "log2_enrichment_difference":
                    correct[
                        "log2_enrichment"
                    ].mean()
                    -
                    missed[
                        "log2_enrichment"
                    ].mean(),

                "pointing_correct":
                    correct[
                        "pointing_game"
                    ].mean(),

                "pointing_missed":
                    missed[
                        "pointing_game"
                    ].mean(),

                "pointing_difference":
                    correct[
                        "pointing_game"
                    ].mean()
                    -
                    missed[
                        "pointing_game"
                    ].mean()
            }
        )

    return pd.DataFrame(
        rows
    )


layer_summary = stratified_summary(
    layer_strat,
    "LayerCAM"
)

ig_summary = stratified_summary(
    ig_strat,
    "Integrated Gradients 256"
)

stratified_results = pd.concat(
    [
        layer_summary,
        ig_summary
    ],
    ignore_index=True
)

stratified_results.to_csv(
    STRATIFIED_XAI_DIR
    / "correct_vs_missed_by_lesion_burden_quintile.csv",
    index=False
)


print("\nCorrect minus missed within lesion-burden strata")
print("--------------------------------------------")

for method in [
    "LayerCAM",
    "Integrated Gradients 256"
]:

    print(f"\n{method}")

    subset = stratified_results[
        stratified_results[
            "method"
        ] == method
    ]

    print(
        subset[
            [
                "burden_stratum",
                "n_correct",
                "n_missed",
                "bbox_energy_difference",
                "log2_enrichment_difference",
                "pointing_difference"
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:+.4f}"
        )
    )


# Weighted within-stratum effect:
# weights proportional to number of observations
weighted_rows = []

for method in [
    "LayerCAM",
    "Integrated Gradients 256"
]:

    subset = stratified_results[
        stratified_results[
            "method"
        ] == method
    ].copy()

    weights = (
        subset["n"]
        / subset["n"].sum()
    )

    weighted_rows.append(
        {
            "method":
                method,

            "weighted_bbox_energy_difference":
                np.sum(
                    weights
                    * subset[
                        "bbox_energy_difference"
                    ]
                ),

            "weighted_log2_enrichment_difference":
                np.sum(
                    weights
                    * subset[
                        "log2_enrichment_difference"
                    ]
                ),

            "weighted_pointing_difference":
                np.sum(
                    weights
                    * subset[
                        "pointing_difference"
                    ]
                ),

            "strata_positive_enrichment":
                int(
                    (
                        subset[
                            "log2_enrichment_difference"
                        ] > 0
                    ).sum()
                ),

            "strata_positive_pointing":
                int(
                    (
                        subset[
                            "pointing_difference"
                        ] > 0
                    ).sum()
                )
        }
    )

weighted_summary = pd.DataFrame(
    weighted_rows
)

weighted_summary.to_csv(
    STRATIFIED_XAI_DIR
    / "weighted_within_stratum_summary.csv",
    index=False
)

print("\nWeighted within-stratum summary")
print("--------------------------------------------")

print(
    weighted_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:+.4f}"
    )
)

print("\nResults saved to:")
print(STRATIFIED_XAI_DIR)


In [ ]:
import numpy as np
import pandas as pd

QUALITATIVE_DIR = (
    FINAL_TEST_DIR
    / "advanced_stage1"
    / "xai"
    / "qualitative_case_selection"
)

QUALITATIVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Merge LayerCAM and IG case-level results
case_compare = (
    test_layercam_df[
        [
            "patientId",
            "position",
            "correct_opacity",
            "prob_lung_opacity",
            "bbox_energy_fraction",
            "energy_enrichment",
            "pointing_game"
        ]
    ]
    .rename(
        columns={
            "bbox_energy_fraction":
                "layercam_bbox_energy",
            "energy_enrichment":
                "layercam_enrichment",
            "pointing_game":
                "layercam_pointing"
        }
    )
    .merge(
        test_ig256_df[
            [
                "patientId",
                "bbox_energy_fraction",
                "energy_enrichment",
                "pointing_game"
            ]
        ].rename(
            columns={
                "bbox_energy_fraction":
                    "ig_bbox_energy",
                "energy_enrichment":
                    "ig_enrichment",
                "pointing_game":
                    "ig_pointing"
            }
        ),
        on="patientId",
        how="inner",
        validate="one_to_one"
    )
)

# Add lesion burden
case_compare = case_compare.merge(
    burden_source[
        [
            "patientId",
            "union_bbox_area_fraction",
            "bbox_count"
        ]
    ],
    on="patientId",
    how="left",
    validate="one_to_one"
)

groups = {
    "AP_correct": (
        (case_compare["position"] == "AP")
        & (case_compare["correct_opacity"] == True)
    ),
    "AP_missed": (
        (case_compare["position"] == "AP")
        & (case_compare["correct_opacity"] == False)
    ),
    "PA_correct": (
        (case_compare["position"] == "PA")
        & (case_compare["correct_opacity"] == True)
    ),
    "PA_missed": (
        (case_compare["position"] == "PA")
        & (case_compare["correct_opacity"] == False)
    )
}

selected_rows = []

for group_name, mask in groups.items():

    group = (
        case_compare[mask]
        .copy()
        .sort_values(
            "union_bbox_area_fraction"
        )
        .reset_index(drop=True)
    )

    # Select approximately low, median and high lesion burden
    quantiles = [
        0.25,
        0.50,
        0.75
    ]

    for q in quantiles:

        target_index = int(
            round(
                q
                * (
                    len(group) - 1
                )
            )
        )

        row = group.iloc[
            target_index
        ].copy()

        row["selection_group"] = (
            group_name
        )

        row["selection_quantile"] = q

        selected_rows.append(
            row
        )

selected_cases = pd.DataFrame(
    selected_rows
)

selected_cases.to_csv(
    QUALITATIVE_DIR
    / "selected_representative_cases.csv",
    index=False
)

print("Representative XAI cases")
print("--------------------------------------------")

print(
    selected_cases[
        [
            "selection_group",
            "selection_quantile",
            "patientId",
            "prob_lung_opacity",
            "union_bbox_area_fraction",
            "layercam_bbox_energy",
            "ig_bbox_energy",
            "layercam_pointing",
            "ig_pointing"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nCases selected:", len(selected_cases))
print("Saved to:")
print(
    QUALITATIVE_DIR
    / "selected_representative_cases.csv"
)


## 16. Post-hoc training-derived spatial-prior sensitivity analysis

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


TRAIN_PATH = PROJECT_ROOT / "data/splits/train.csv"
TEST_PATH = PROJECT_ROOT / "data/splits/test.csv"
BBOX_PATH = PROJECT_ROOT / "data/processed/bbox_annotations.csv"

LAYERCAM_PATH = (
    PROJECT_ROOT
    / "results/final_test/advanced_stage1/xai/layercam"
    / "test_layercam_case_metrics.csv"
)

IG_PATH = (
    PROJECT_ROOT
    / "results/final_test/advanced_stage1/xai/integrated_gradients_256"
    / "test_ig256_case_metrics.csv"
)

OUT_DIR = (
    PROJECT_ROOT
    / "results/final_test/advanced_stage1/xai"
    / "spatial_prior_baseline"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
boxes_df = pd.read_csv(BBOX_PATH)

layercam_df = pd.read_csv(LAYERCAM_PATH)
ig_df = pd.read_csv(IG_PATH)


OPACITY_CLASS = "Lung Opacity"
MAP_SIZE = 224
ORIGINAL_SIZE = 1024
SCALE = MAP_SIZE / ORIGINAL_SIZE


# ---------------------------------------------------------
# Basic integrity checks
# ---------------------------------------------------------

train_opacity = (
    train_df.loc[
        train_df["class"] == OPACITY_CLASS
    ]
    .copy()
    .reset_index(drop=True)
)

test_opacity = (
    test_df.loc[
        test_df["class"] == OPACITY_CLASS
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(train_opacity) == 4208
assert len(test_opacity) == 902
assert len(layercam_df) == 902
assert len(ig_df) == 902

assert set(test_opacity["patientId"]) == set(layercam_df["patientId"])
assert set(test_opacity["patientId"]) == set(ig_df["patientId"])

print("Integrity checks passed.")
print("Training opacity cases:", len(train_opacity))
print("Test opacity cases:", len(test_opacity))

print("\nTraining opacity position counts:")
print(train_opacity["position"].value_counts())


# ---------------------------------------------------------
# Prepare bbox lookup
# ---------------------------------------------------------

bbox_lookup = {}

for patient_id, group in boxes_df.groupby("patientId"):

    bbox_lookup[patient_id] = group[
        ["x", "y", "width", "height"]
    ].to_numpy(dtype=float)


def make_bbox_mask(patient_id):

    mask = np.zeros(
        (MAP_SIZE, MAP_SIZE),
        dtype=np.float32
    )

    patient_boxes = bbox_lookup.get(
        patient_id,
        None
    )

    if patient_boxes is None:
        return mask

    for x, y, width, height in patient_boxes:

        x1 = int(np.floor(x * SCALE))
        y1 = int(np.floor(y * SCALE))

        x2 = int(
            np.ceil(
                (x + width) * SCALE
            )
        )

        y2 = int(
            np.ceil(
                (y + height) * SCALE
            )
        )

        x1 = max(0, min(MAP_SIZE, x1))
        y1 = max(0, min(MAP_SIZE, y1))
        x2 = max(0, min(MAP_SIZE, x2))
        y2 = max(0, min(MAP_SIZE, y2))

        if x2 > x1 and y2 > y1:
            mask[
                y1:y2,
                x1:x2
            ] = 1.0

    return mask


# ---------------------------------------------------------
# Build priors from TRAINING cases only
#
# Each image contributes one union mask, regardless of the
# number of boxes.
# ---------------------------------------------------------

global_accumulator = np.zeros(
    (MAP_SIZE, MAP_SIZE),
    dtype=np.float64
)

ap_accumulator = np.zeros_like(
    global_accumulator
)

pa_accumulator = np.zeros_like(
    global_accumulator
)

global_n = 0
ap_n = 0
pa_n = 0


for row in train_opacity.itertuples(index=False):

    mask = make_bbox_mask(
        row.patientId
    )

    if mask.sum() == 0:
        continue

    global_accumulator += mask
    global_n += 1

    if row.position == "AP":

        ap_accumulator += mask
        ap_n += 1

    elif row.position == "PA":

        pa_accumulator += mask
        pa_n += 1


global_prior = (
    global_accumulator / global_n
)

ap_prior = (
    ap_accumulator / ap_n
)

pa_prior = (
    pa_accumulator / pa_n
)


print("\nTraining cases used for priors:")
print("Global:", global_n)
print("AP:", ap_n)
print("PA:", pa_n)

assert global_n == len(train_opacity)
assert ap_n + pa_n == global_n


# ---------------------------------------------------------
# Normalize each prior as an attribution-energy map
# ---------------------------------------------------------

def normalize_energy_map(arr):

    arr = np.asarray(
        arr,
        dtype=np.float64
    )

    arr = np.clip(
        arr,
        0,
        None
    )

    total = arr.sum()

    if total <= 0:
        raise ValueError(
            "Spatial prior has zero total energy."
        )

    return arr / total


global_energy = normalize_energy_map(
    global_prior
)

ap_energy = normalize_energy_map(
    ap_prior
)

pa_energy = normalize_energy_map(
    pa_prior
)


np.save(
    OUT_DIR / "global_training_spatial_prior.npy",
    global_prior
)

np.save(
    OUT_DIR / "ap_training_spatial_prior.npy",
    ap_prior
)

np.save(
    OUT_DIR / "pa_training_spatial_prior.npy",
    pa_prior
)


# ---------------------------------------------------------
# Pointing-game handling
#
# If a fixed spatial prior has multiple exactly maximal
# pixels, count success when ANY maximal pixel overlaps
# the patient's box. This deliberately favors the prior.
# ---------------------------------------------------------

def maximal_pixel_mask(energy_map):

    maximum = np.max(
        energy_map
    )

    return np.isclose(
        energy_map,
        maximum,
        rtol=1e-12,
        atol=1e-15
    )


global_peak_mask = maximal_pixel_mask(
    global_energy
)

ap_peak_mask = maximal_pixel_mask(
    ap_energy
)

pa_peak_mask = maximal_pixel_mask(
    pa_energy
)


# ---------------------------------------------------------
# Localization metrics
# ---------------------------------------------------------

def compute_prior_metrics(
    energy_map,
    peak_mask,
    bbox_mask
):

    bbox_bool = (
        bbox_mask > 0
    )

    area_fraction = (
        bbox_bool.mean()
    )

    energy_fraction = (
        energy_map[
            bbox_bool
        ].sum()
    )

    enrichment = (
        energy_fraction
        / area_fraction
    )

    pointing = int(
        np.any(
            peak_mask
            & bbox_bool
        )
    )

    p = energy_map[
        energy_map > 0
    ]

    entropy = (
        -np.sum(
            p * np.log(p)
        )
        / np.log(
            energy_map.size
        )
    )

    return (
        area_fraction,
        energy_fraction,
        enrichment,
        pointing,
        entropy
    )


# ---------------------------------------------------------
# Evaluate priors on locked test opacity cases
# ---------------------------------------------------------

records = []


for row in test_opacity.itertuples(index=False):

    patient_id = row.patientId
    position = row.position

    bbox_mask = make_bbox_mask(
        patient_id
    )

    if bbox_mask.sum() == 0:
        raise ValueError(
            f"No bbox found for {patient_id}"
        )

    global_metrics = compute_prior_metrics(
        global_energy,
        global_peak_mask,
        bbox_mask
    )

    if position == "AP":

        position_metrics = compute_prior_metrics(
            ap_energy,
            ap_peak_mask,
            bbox_mask
        )

    elif position == "PA":

        position_metrics = compute_prior_metrics(
            pa_energy,
            pa_peak_mask,
            bbox_mask
        )

    else:
        raise ValueError(
            f"Unexpected position: {position}"
        )

    records.append(
        {
            "patientId": patient_id,
            "position": position,

            "prior_bbox_area_fraction":
                global_metrics[0],

            "global_prior_bbox_energy_fraction":
                global_metrics[1],

            "global_prior_energy_enrichment":
                global_metrics[2],

            "global_prior_pointing_game":
                global_metrics[3],

            "global_prior_normalized_entropy":
                global_metrics[4],

            "position_prior_bbox_energy_fraction":
                position_metrics[1],

            "position_prior_energy_enrichment":
                position_metrics[2],

            "position_prior_pointing_game":
                position_metrics[3],

            "position_prior_normalized_entropy":
                position_metrics[4],
        }
    )


prior_df = pd.DataFrame(
    records
)


# ---------------------------------------------------------
# Merge existing LayerCAM and IG results
# ---------------------------------------------------------

layercam_cols = (
    layercam_df[
        [
            "patientId",
            "position",
            "bbox_area_fraction",
            "bbox_energy_fraction",
            "energy_enrichment",
            "pointing_game",
            "normalized_entropy"
        ]
    ]
    .rename(
        columns={
            "bbox_area_fraction":
                "layercam_bbox_area_fraction",

            "bbox_energy_fraction":
                "layercam_bbox_energy_fraction",

            "energy_enrichment":
                "layercam_energy_enrichment",

            "pointing_game":
                "layercam_pointing_game",

            "normalized_entropy":
                "layercam_normalized_entropy",
        }
    )
)


ig_cols = (
    ig_df[
        [
            "patientId",
            "bbox_energy_fraction",
            "energy_enrichment",
            "pointing_game",
            "normalized_entropy"
        ]
    ]
    .rename(
        columns={
            "bbox_energy_fraction":
                "ig_bbox_energy_fraction",

            "energy_enrichment":
                "ig_energy_enrichment",

            "pointing_game":
                "ig_pointing_game",

            "normalized_entropy":
                "ig_normalized_entropy",
        }
    )
)


comparison = (
    prior_df
    .merge(
        layercam_cols,
        on=[
            "patientId",
            "position"
        ],
        how="inner",
        validate="one_to_one"
    )
    .merge(
        ig_cols,
        on="patientId",
        how="inner",
        validate="one_to_one"
    )
)


assert len(comparison) == 902


# ---------------------------------------------------------
# Check whether our resized bbox masks closely reproduce
# the area fractions used in the locked XAI analysis.
# ---------------------------------------------------------

area_difference = np.abs(
    comparison[
        "prior_bbox_area_fraction"
    ]
    -
    comparison[
        "layercam_bbox_area_fraction"
    ]
)


print("\nBBox-area reconstruction check:")
print(
    "Mean absolute difference:",
    f"{area_difference.mean():.6f}"
)

print(
    "Median absolute difference:",
    f"{area_difference.median():.6f}"
)

print(
    "Maximum absolute difference:",
    f"{area_difference.max():.6f}"
)


# ---------------------------------------------------------
# Overall method comparison
# ---------------------------------------------------------

method_prefixes = {
    "Global spatial prior":
        "global_prior",

    "Position-conditioned spatial prior":
        "position_prior",

    "LayerCAM":
        "layercam",

    "Integrated Gradients 256":
        "ig"
}


summary_records = []


for method, prefix in method_prefixes.items():

    summary_records.append(
        {
            "method": method,

            "bbox_energy_fraction":
                comparison[
                    f"{prefix}_bbox_energy_fraction"
                ].mean(),

            "energy_enrichment":
                comparison[
                    f"{prefix}_energy_enrichment"
                ].mean(),

            "median_energy_enrichment":
                comparison[
                    f"{prefix}_energy_enrichment"
                ].median(),

            "pointing_game":
                comparison[
                    f"{prefix}_pointing_game"
                ].mean(),

            "normalized_entropy":
                comparison[
                    f"{prefix}_normalized_entropy"
                ].mean(),
        }
    )


summary_df = pd.DataFrame(
    summary_records
)


print("\nOverall locked-test localization comparison:")
print(
    summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ---------------------------------------------------------
# AP / PA subgroup comparison
# ---------------------------------------------------------

subgroup_records = []


for position in ["AP", "PA"]:

    subset = comparison.loc[
        comparison["position"] == position
    ]

    for method, prefix in method_prefixes.items():

        subgroup_records.append(
            {
                "position": position,
                "method": method,
                "n": len(subset),

                "bbox_energy_fraction":
                    subset[
                        f"{prefix}_bbox_energy_fraction"
                    ].mean(),

                "energy_enrichment":
                    subset[
                        f"{prefix}_energy_enrichment"
                    ].mean(),

                "pointing_game":
                    subset[
                        f"{prefix}_pointing_game"
                    ].mean(),
            }
        )


subgroup_df = pd.DataFrame(
    subgroup_records
)


print("\nAP / PA localization comparison:")
print(
    subgroup_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ---------------------------------------------------------
# Position-stratified paired bootstrap
#
# Preserve AP and PA sample sizes in each replicate.
# Compare LayerCAM and IG with the stronger
# position-conditioned spatial prior.
# ---------------------------------------------------------

rng = np.random.default_rng(42)
N_BOOT = 5000

ap_indices = np.where(
    comparison["position"].to_numpy() == "AP"
)[0]

pa_indices = np.where(
    comparison["position"].to_numpy() == "PA"
)[0]


def stratified_bootstrap_difference(
    method_values,
    prior_values
):

    paired_difference = (
        method_values
        - prior_values
    )

    observed = (
        paired_difference.mean()
    )

    boot = np.empty(
        N_BOOT,
        dtype=np.float64
    )

    for b in range(N_BOOT):

        sample_ap = rng.choice(
            ap_indices,
            size=len(ap_indices),
            replace=True
        )

        sample_pa = rng.choice(
            pa_indices,
            size=len(pa_indices),
            replace=True
        )

        indices = np.concatenate(
            [
                sample_ap,
                sample_pa
            ]
        )

        boot[b] = (
            paired_difference[
                indices
            ].mean()
        )

    low, high = np.percentile(
        boot,
        [2.5, 97.5]
    )

    return (
        observed,
        low,
        high
    )


bootstrap_records = []


for method_name, prefix in [
    ("LayerCAM", "layercam"),
    ("Integrated Gradients 256", "ig")
]:

    # Bbox energy
    result = stratified_bootstrap_difference(
        comparison[
            f"{prefix}_bbox_energy_fraction"
        ].to_numpy(dtype=float),

        comparison[
            "position_prior_bbox_energy_fraction"
        ].to_numpy(dtype=float)
    )

    bootstrap_records.append(
        {
            "method": method_name,
            "metric": "bbox_energy_fraction",
            "method_minus_position_prior":
                result[0],
            "ci_low": result[1],
            "ci_high": result[2],
        }
    )

    # Log2 enrichment
    method_enrichment = np.log2(
        np.clip(
            comparison[
                f"{prefix}_energy_enrichment"
            ].to_numpy(dtype=float),
            1e-12,
            None
        )
    )

    prior_enrichment = np.log2(
        np.clip(
            comparison[
                "position_prior_energy_enrichment"
            ].to_numpy(dtype=float),
            1e-12,
            None
        )
    )

    result = stratified_bootstrap_difference(
        method_enrichment,
        prior_enrichment
    )

    bootstrap_records.append(
        {
            "method": method_name,
            "metric": "log2_energy_enrichment",
            "method_minus_position_prior":
                result[0],
            "ci_low": result[1],
            "ci_high": result[2],
        }
    )

    # Pointing game
    result = stratified_bootstrap_difference(
        comparison[
            f"{prefix}_pointing_game"
        ].to_numpy(dtype=float),

        comparison[
            "position_prior_pointing_game"
        ].to_numpy(dtype=float)
    )

    bootstrap_records.append(
        {
            "method": method_name,
            "metric": "pointing_game",
            "method_minus_position_prior":
                result[0],
            "ci_low": result[1],
            "ci_high": result[2],
        }
    )


bootstrap_df = pd.DataFrame(
    bootstrap_records
)


print(
    "\nMethod minus position-conditioned spatial prior:"
)

print(
    bootstrap_df.to_string(
        index=False,
        float_format=lambda x: f"{x:+.4f}"
    )
)


# ---------------------------------------------------------
# Save all results
# ---------------------------------------------------------

prior_df.to_csv(
    OUT_DIR
    / "test_spatial_prior_case_metrics.csv",
    index=False
)

comparison.to_csv(
    OUT_DIR
    / "test_spatial_prior_full_comparison.csv",
    index=False
)

summary_df.to_csv(
    OUT_DIR
    / "overall_spatial_prior_comparison.csv",
    index=False
)

subgroup_df.to_csv(
    OUT_DIR
    / "ap_pa_spatial_prior_comparison.csv",
    index=False
)

bootstrap_df.to_csv(
    OUT_DIR
    / "method_vs_position_prior_bootstrap.csv",
    index=False
)


manifest = {
    "analysis_type":
        "post_hoc_spatial_prior_sensitivity_analysis",

    "training_only_prior":
        True,

    "map_size":
        MAP_SIZE,

    "global_training_opacity_n":
        int(global_n),

    "ap_training_opacity_n":
        int(ap_n),

    "pa_training_opacity_n":
        int(pa_n),

    "locked_test_opacity_n":
        int(len(comparison)),

    "bootstrap_replicates":
        N_BOOT,

    "bootstrap_seed":
        42,

    "bootstrap_design":
        "paired and stratified by AP_PA position",

    "pointing_tie_rule":
        "success if any globally maximal prior pixel overlaps the bbox",

    "xai_methods":
        [
            "LayerCAM",
            "Integrated Gradients 256"
        ],

    "strongest_prior":
        "position-conditioned training-box spatial prior"
}


with open(
    OUT_DIR / "spatial_prior_analysis_manifest.json",
    "w"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


print("\nSaved to:")
print(OUT_DIR)
